# PhotoGraphiQML manuscript experiment suite

Self-contained notebook assembled directly from the scripts in `paper_experiments/`. Each section below is one experiment (R1-R48 plus the aggregate performance figure): its scientific question, theory, oracle class and acceptance condition are in the markdown cell (from the script's own docstring); running the code cell reproduces its figures, CSV/JSON and metadata.

## Install dependencies

Installs this repository (editable) with its `dev`/`experiments` extras, plus `photographiq`, `mentpy` and `piquasso` if not already present. Skip this cell if your environment (e.g. the project `.venv`) already has them installed.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('ensurepip') is not None:
    pass
for pkg in ('photographiqml', 'photographiq', 'mentpy', 'piquasso', 'matplotlib', 'scikit-learn'):
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=False)


## Setup

Imports the shared `common`/`metadata` utilities, falling back to this notebook's own directory if the working directory changes between cells.

In [ ]:
import sys
from pathlib import Path
for candidate in (Path.cwd(), Path.cwd() / 'paper_experiments'):
    if (candidate / 'common.py').exists():
        sys.path.insert(0, str(candidate))
        break
import common, metadata
print(metadata.collect())


## R1: Triangle-neuron and MuTA graph anatomy.

Scientific question: Do TriangleNeuron and MuTA build the exact semantic
graph anatomy (node count, edge count, input/output labels, trainable-node
count, causal DAG) that Appendix B's paper-layer construction implies, for
every combination of wire count, requested layers, one_column and
restrict_trainable?

Theory/equations: One paper (n,pivot) layer has 5n nodes (columns 0..4 per
wire) and 4n + 2(n-1) edges (4 wire-chain edges per wire, plus 2 cross edges
per non-pivot wire connecting it to the pivot's column 1). Stacking d paper
layers (d = n_layers if one_column else n_layers*n_wires) with boundary
identification gives |V| = n_wires*(4d+1), |E| = 4*n_wires*d + 2*d*(n_wires-1),
and n_parameters = (3 if restrict_trainable else 4) * n_wires * d.

Functionality tested: TriangleNeuron.graph/input_nodes/output_nodes/
trainable_nodes; MuTA.__init__ graph assembly, flow, dependency_graph,
measurement_order, n_parameters, expected_parameter_count.

Oracle and independence class: A (independent analytic oracle) -- expected
counts are hand-derived closed-form formulas, evaluated independently of the
package's own bookkeeping and compared to what the package actually built.

Exact/approximate/statistical status: exact (integer equality).

Primary metric: max absolute error between analytic and constructed counts
(nodes, edges, trainable parameters) across the full sweep.

Declared acceptance condition: max absolute error == 0 for every swept
configuration, and the dependency DAG must be acyclic with measurement_order
respecting all DAG edges (a topological order).

Expected cost: light (pure graph construction, no simulation).

Manuscript destination: Main text (Fig. 1, triangle/MuTA anatomy) and
Table I (structural definitions).

Scientific limitations: This is a structural/analytic check of the ideal
logical graph only; it says nothing about physical lowering or MentPy
agreement (see R7-R9, R36).

In [ ]:
"""R1: Triangle-neuron and MuTA graph anatomy.

Scientific question: Do TriangleNeuron and MuTA build the exact semantic
graph anatomy (node count, edge count, input/output labels, trainable-node
count, causal DAG) that Appendix B's paper-layer construction implies, for
every combination of wire count, requested layers, one_column and
restrict_trainable?

Theory/equations: One paper (n,pivot) layer has 5n nodes (columns 0..4 per
wire) and 4n + 2(n-1) edges (4 wire-chain edges per wire, plus 2 cross edges
per non-pivot wire connecting it to the pivot's column 1). Stacking d paper
layers (d = n_layers if one_column else n_layers*n_wires) with boundary
identification gives |V| = n_wires*(4d+1), |E| = 4*n_wires*d + 2*d*(n_wires-1),
and n_parameters = (3 if restrict_trainable else 4) * n_wires * d.

Functionality tested: TriangleNeuron.graph/input_nodes/output_nodes/
trainable_nodes; MuTA.__init__ graph assembly, flow, dependency_graph,
measurement_order, n_parameters, expected_parameter_count.

Oracle and independence class: A (independent analytic oracle) -- expected
counts are hand-derived closed-form formulas, evaluated independently of the
package's own bookkeeping and compared to what the package actually built.

Exact/approximate/statistical status: exact (integer equality).

Primary metric: max absolute error between analytic and constructed counts
(nodes, edges, trainable parameters) across the full sweep.

Declared acceptance condition: max absolute error == 0 for every swept
configuration, and the dependency DAG must be acyclic with measurement_order
respecting all DAG edges (a topological order).

Expected cost: light (pure graph construction, no simulation).

Manuscript destination: Main text (Fig. 1, triangle/MuTA anatomy) and
Table I (structural definitions).

Scientific limitations: This is a structural/analytic check of the ideal
logical graph only; it says nothing about physical lowering or MentPy
agreement (see R7-R9, R36).
"""

import sys
from pathlib import Path

import common
import networkx as nx

from photographiqml import MuTA, TriangleNeuron

EXPERIMENT_ID = "R1"


def analytic_triangle_counts(n_wires, connections):
    connected = [w for w in range(n_wires) if w != 0] if connections is None else list(connections)
    nodes = 5 * n_wires
    edges = 4 * n_wires + 2 * len(connected)
    trainable = 4 * n_wires
    return nodes, edges, trainable


def analytic_muta_counts(n_wires, n_layers, one_column, restrict_trainable):
    d = n_layers if one_column else n_layers * n_wires
    nodes = n_wires * (4 * d + 1)
    edges = 4 * n_wires * d + 2 * d * (n_wires - 1)
    trainable = (3 if restrict_trainable else 4) * n_wires * d
    return nodes, edges, trainable, d


def main():
    plt = common.setup_style()
    rows = []

    # --- TriangleNeuron anatomy -------------------------------------------------
    for n_wires in (1, 2, 3, 4):
        for connections in (None, (), tuple(range(1, n_wires))[:1]):
            if connections == () and n_wires == 1:
                pass
            try:
                triangle = TriangleNeuron(n_wires, pivot=0, connections=connections)
            except ValueError:
                continue
            expected_nodes, expected_edges, expected_trainable = analytic_triangle_counts(
                n_wires, connections
            )
            actual_nodes, actual_edges = len(triangle.graph), triangle.graph.number_of_edges()
            actual_trainable = len(triangle.trainable_nodes)
            rows.append(
                {
                    "level": "triangle",
                    "n_wires": n_wires,
                    "n_layers": None,
                    "one_column": None,
                    "restrict_trainable": None,
                    "connections": str(connections),
                    "expected_nodes": expected_nodes,
                    "actual_nodes": actual_nodes,
                    "expected_edges": expected_edges,
                    "actual_edges": actual_edges,
                    "expected_trainable": expected_trainable,
                    "actual_trainable": actual_trainable,
                    "node_error": abs(expected_nodes - actual_nodes),
                    "edge_error": abs(expected_edges - actual_edges),
                    "trainable_error": abs(expected_trainable - actual_trainable),
                    "input_nodes_ok": triangle.input_nodes == tuple((w, 0) for w in range(n_wires)),
                    "output_nodes_ok": triangle.output_nodes
                    == tuple((w, 4) for w in range(n_wires)),
                }
            )

    # --- MuTA anatomy sweep -------------------------------------------------
    for n_wires in (1, 2, 3):
        for n_layers in (1, 2, 3):
            for one_column in (False, True):
                for restrict_trainable in (False, True):
                    model = MuTA(
                        n_wires,
                        n_layers,
                        one_column=one_column,
                        restrict_trainable=restrict_trainable,
                    )
                    expected_nodes, expected_edges, expected_trainable, d = analytic_muta_counts(
                        n_wires, n_layers, one_column, restrict_trainable
                    )
                    actual_nodes, actual_edges = len(model.graph), model.graph.number_of_edges()
                    dag_acyclic = nx.is_directed_acyclic_graph(model.dependency_graph)
                    positions = {
                        v: i for i, v in enumerate(model.measurement_order + model.output_nodes)
                    }
                    order_respects_dag = all(
                        positions[u] < positions[v] for u, v in model.dependency_graph.edges
                    )
                    rows.append(
                        {
                            "level": "muta",
                            "n_wires": n_wires,
                            "n_layers": n_layers,
                            "one_column": one_column,
                            "restrict_trainable": restrict_trainable,
                            "connections": None,
                            "expected_nodes": expected_nodes,
                            "actual_nodes": actual_nodes,
                            "expected_edges": expected_edges,
                            "actual_edges": actual_edges,
                            "expected_trainable": expected_trainable,
                            "actual_trainable": model.n_parameters,
                            "node_error": abs(expected_nodes - actual_nodes),
                            "edge_error": abs(expected_edges - actual_edges),
                            "trainable_error": abs(expected_trainable - model.n_parameters),
                            "paper_depth": model.paper_depth,
                            "paper_depth_ok": model.paper_depth == d,
                            "dag_acyclic": dag_acyclic,
                            "order_respects_dag": order_respects_dag,
                            "input_nodes_ok": model.input_nodes
                            == tuple((w, 0) for w in range(n_wires)),
                            "output_nodes_ok": model.output_nodes
                            == tuple((w, 4 * model.paper_depth) for w in range(n_wires)),
                        }
                    )

    max_node_error = max(r["node_error"] for r in rows)
    max_edge_error = max(r["edge_error"] for r in rows)
    max_trainable_error = max(r["trainable_error"] for r in rows)
    all_dag_ok = all(r.get("dag_acyclic", True) and r.get("order_respects_dag", True) for r in rows)
    all_io_ok = all(r["input_nodes_ok"] and r["output_nodes_ok"] for r in rows)
    passed = (
        max_node_error == 0
        and max_edge_error == 0
        and max_trainable_error == 0
        and all_dag_ok
        and all_io_ok
    )
    status = "pass" if passed else "fail"

    common.save_result(
        rows,
        "R1_triangle_anatomy",
        extra={
            "protocol": "TriangleNeuron/MuTA graph anatomy vs. closed-form node/edge/parameter counts",
            "oracle_class": "A",
            "status_category": "exact",
            "acceptance_condition": "max_node_error == max_edge_error == max_trainable_error == 0; DAG acyclic and topologically consistent",
            "max_node_error": max_node_error,
            "max_edge_error": max_edge_error,
            "max_trainable_error": max_trainable_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
    muta_rows = [r for r in rows if r["level"] == "muta"]
    for ax, key, title in zip(
        axes,
        ("node_error", "edge_error", "trainable_error"),
        ("Node count error", "Edge count error", "Trainable-parameter error"),
    ):
        values = [r[key] for r in muta_rows]
        colors = [
            common.COLORS["photographiqml"] if v == 0 else common.COLORS["piquasso"] for v in values
        ]
        # All-zero values render invisibly as bars; use markers so every
        # tested configuration is visibly plotted, not just an empty axes.
        ax.scatter(range(len(values)), values, color=colors, marker="o", s=18, zorder=3)
        ax.set_title(f"{title} ({len(values)} configurations)")
        ax.set_xlabel("configuration")
        ax.set_ylabel("|analytic - actual|")
        ax.set_xticks([])
        ax.axhline(0, color=common.COLORS["acceptance"], linewidth=0.8)
    fig.suptitle(
        f"R1: MuTA graph anatomy vs. analytic formulas ({len(muta_rows)} configurations, status={status})"
    )
    common.save_figure(fig, "R1_triangle_anatomy")
    plt.close(fig)

    common.print_summary(
        "R1 triangle/MuTA anatomy",
        configurations=len(rows),
        max_node_error=max_node_error,
        max_edge_error=max_edge_error,
        max_trainable_error=max_trainable_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R1 failed: node={max_node_error} edge={max_edge_error} trainable={max_trainable_error}"
        )


if __name__ == "__main__":
    main()


## R2: Single-wire Table-I identities against hand-written analytic 2x2 unitaries.

Scientific question: Does a single-wire MuTA paper layer implement the exact
Euler-angle single-qubit rotation sequence Table I claims for its three
trainable intermediate measurement columns (c1, c2, c3)?

Theory/equations: each measured column applies gate(a) = H @ Rz(a), with
Rz(a) = exp(i*a*Z/2). Using H Rz(t) H = Rx(t) to push each H rightward through
the chain gate(c3) gate(c2) gate(c1) gate(c0), the four intermediate H's
telescope pairwise to identity, giving the single-wire Euler identity
U = Rx(c3) @ Rz(c2) @ Rx(c1) @ Rz(c0) (alternating X,Z,X,Z from right to
left), an independent derivation of Table I's per-column rotation axes.
Column c0 is the input node itself (also measured, per the model's
"every non-output node is measured" convention) and is swept too, to confirm
its Z-axis sub-identity in isolation alongside c1/c2/c3's X/Z/X axes.

Functionality tested: MuTA.unitary via ansatz/muta.py + logical.execute, for
n_wires=1.

Oracle and independence class: A (independent analytic oracle) -- unitaries
are built directly from scipy.linalg.expm of Pauli matrices defined locally
in this script (not imported from photographiqml.logical), independent of the
package's own H/X/Z arrays and local_gate/cz implementation.

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max Frobenius-norm error between MuTA.unitary and the
Table-I analytic prediction, over a grid of random angle triples.

Declared acceptance condition: max Frobenius error < tol, with
tol = declare_tolerance(scale=1) computed before the sweep (see common.py).

Expected cost: light.

Manuscript destination: Main text (Table I verification, supporting Fig. 1).

Scientific limitations: Single-wire only; does not test entangling identities
(see R3) or multi-wire composition (see R1, R5).

In [ ]:
"""R2: Single-wire Table-I identities against hand-written analytic 2x2 unitaries.

Scientific question: Does a single-wire MuTA paper layer implement the exact
Euler-angle single-qubit rotation sequence Table I claims for its three
trainable intermediate measurement columns (c1, c2, c3)?

Theory/equations: each measured column applies gate(a) = H @ Rz(a), with
Rz(a) = exp(i*a*Z/2). Using H Rz(t) H = Rx(t) to push each H rightward through
the chain gate(c3) gate(c2) gate(c1) gate(c0), the four intermediate H's
telescope pairwise to identity, giving the single-wire Euler identity
U = Rx(c3) @ Rz(c2) @ Rx(c1) @ Rz(c0) (alternating X,Z,X,Z from right to
left), an independent derivation of Table I's per-column rotation axes.
Column c0 is the input node itself (also measured, per the model's
"every non-output node is measured" convention) and is swept too, to confirm
its Z-axis sub-identity in isolation alongside c1/c2/c3's X/Z/X axes.

Functionality tested: MuTA.unitary via ansatz/muta.py + logical.execute, for
n_wires=1.

Oracle and independence class: A (independent analytic oracle) -- unitaries
are built directly from scipy.linalg.expm of Pauli matrices defined locally
in this script (not imported from photographiqml.logical), independent of the
package's own H/X/Z arrays and local_gate/cz implementation.

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max Frobenius-norm error between MuTA.unitary and the
Table-I analytic prediction, over a grid of random angle triples.

Declared acceptance condition: max Frobenius error < tol, with
tol = declare_tolerance(scale=1) computed before the sweep (see common.py).

Expected cost: light.

Manuscript destination: Main text (Table I verification, supporting Fig. 1).

Scientific limitations: Single-wire only; does not test entangling identities
(see R3) or multi-wire composition (see R1, R5).
"""

import sys
from pathlib import Path

import common
import numpy as np
from scipy.linalg import expm

from photographiqml import MuTA

EXPERIMENT_ID = "R2"

X_REF = np.array([[0, 1], [1, 0]], dtype=complex)
Z_REF = np.diag([1, -1]).astype(complex)


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0)
    generator = common.rng(0)
    n_trials = 40
    rows = []
    model = MuTA(1, 1)

    for trial in range(n_trials):
        theta, phi, lam = generator.uniform(-2 * np.pi, 2 * np.pi, size=3)
        c0 = generator.uniform(-2 * np.pi, 2 * np.pi)
        angles = {
            "alpha.w0.c0": c0,
            "alpha.w0.c1": theta,
            "alpha.w0.c2": phi,
            "alpha.w0.c3": lam,
        }
        actual = model.unitary(angles)
        # U = Rx(c3) @ Rz(c2) @ Rx(c1) @ Rz(c0), derived by telescoping the
        # H-Rz-H = Rx conjugation identity through the 4-column measurement chain.
        expected = (
            expm(0.5j * lam * X_REF)
            @ expm(0.5j * phi * Z_REF)
            @ expm(0.5j * theta * X_REF)
            @ expm(0.5j * c0 * Z_REF)
        )
        error = common.frobenius_error(actual, expected)
        # Isolated single-column sub-identities (freeze the other three at 0).
        single_errors = {}
        for name, angle, ref in (
            ("c0", c0, Z_REF),
            ("c1", theta, X_REF),
            ("c2", phi, Z_REF),
            ("c3", lam, X_REF),
        ):
            isolated = dict.fromkeys(
                ("alpha.w0.c0", "alpha.w0.c1", "alpha.w0.c2", "alpha.w0.c3"), 0.0
            )
            isolated[f"alpha.w0.{name}"] = angle
            single_errors[f"{name}_error"] = common.frobenius_error(
                model.unitary(isolated), expm(0.5j * angle * ref)
            )
        rows.append(
            {
                "trial": trial,
                "theta": theta,
                "phi": phi,
                "lam": lam,
                "c0": c0,
                "full_sequence_error": error,
                **single_errors,
            }
        )

    max_error = max(r["full_sequence_error"] for r in rows)
    max_single_error = max(
        max(r[k] for k in ("c0_error", "c1_error", "c2_error", "c3_error")) for r in rows
    )
    status = "pass" if max(max_error, max_single_error) < tol else "fail"

    common.save_result(
        rows,
        "R2_table_one_identities",
        extra={
            "protocol": "Single-wire Table-I Euler decomposition vs. scipy.linalg.expm reference",
            "oracle_class": "A",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max Frobenius error < {tol:.3e}",
            "max_full_sequence_error": max_error,
            "max_single_column_error": max_single_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].semilogy(
        [r["full_sequence_error"] for r in rows], "o-", color=common.COLORS["photographiqml"]
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[0].set(title="Full 3-angle Euler sequence", xlabel="trial", ylabel="Frobenius error")
    axes[0].legend()
    for key in ("c0_error", "c1_error", "c2_error", "c3_error"):
        axes[1].semilogy([r[key] for r in rows], "o", markersize=3, label=key, alpha=0.7)
    axes[1].axhline(tol, color=common.COLORS["acceptance"], linestyle="--")
    axes[1].set(title="Isolated single-column identities", xlabel="trial", ylabel="Frobenius error")
    axes[1].legend(fontsize=6.5)
    fig.suptitle(f"R2: Table-I identities vs. analytic 2x2 unitaries (status={status})")
    common.save_figure(fig, "R2_table_one_identities")
    plt.close(fig)

    common.print_summary(
        "R2 Table-I identities", n_trials=n_trials, tol=tol, max_error=max_error, status=status
    )
    if status != "pass":
        raise AssertionError(f"R2 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## R3: MuTA entangling identity against an independent SciPy matrix exponential.

Scientific question: Does the pivot-wire triangle's cross-coupling (via c1
measurement of the pivot wire, in a two-wire one-layer model) implement the
exact entangling gate exp(i*phi*XX/2) that Appendix B's coupling identity
claims, for arbitrary coupling angle phi?

Theory/equations: For MuTA(2, one_column=True) with only "alpha.w1.c1" bound
(pivot=0, all other angles left at their default 0), U = exp(i*phi*(X⊗X)/2)
(the Ising-XX two-qubit entangling unitary).

Functionality tested: MuTA.unitary for the two-wire, one-layer, one_column
triangle's cross-edge coupling.

Oracle and independence class: A (independent analytic oracle) -- the target
unitary is built via scipy.linalg.expm of a Pauli-XX generator constructed
locally, entirely independent of photographiqml.logical's cz/local_gate
implementation.

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max Frobenius-norm error between MuTA.unitary and
exp(i*phi*XX/2) over a dense phi grid, plus the induced concurrence of the
first output column compared to the closed-form |sin(phi)|.

Declared acceptance condition: max Frobenius error < tol
(tol = declare_tolerance(scale=1)); max concurrence error < tol.

Expected cost: light.

Manuscript destination: Main text (Table I entangling-gate identity, Fig. 1).

Scientific limitations: Restricted to the minimal two-wire, one-layer
triangle; does not test coupling composed with additional trainable local
rotations on other columns (R1/R5 cover broader composition).

In [ ]:
"""R3: MuTA entangling identity against an independent SciPy matrix exponential.

Scientific question: Does the pivot-wire triangle's cross-coupling (via c1
measurement of the pivot wire, in a two-wire one-layer model) implement the
exact entangling gate exp(i*phi*XX/2) that Appendix B's coupling identity
claims, for arbitrary coupling angle phi?

Theory/equations: For MuTA(2, one_column=True) with only "alpha.w1.c1" bound
(pivot=0, all other angles left at their default 0), U = exp(i*phi*(X⊗X)/2)
(the Ising-XX two-qubit entangling unitary).

Functionality tested: MuTA.unitary for the two-wire, one-layer, one_column
triangle's cross-edge coupling.

Oracle and independence class: A (independent analytic oracle) -- the target
unitary is built via scipy.linalg.expm of a Pauli-XX generator constructed
locally, entirely independent of photographiqml.logical's cz/local_gate
implementation.

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max Frobenius-norm error between MuTA.unitary and
exp(i*phi*XX/2) over a dense phi grid, plus the induced concurrence of the
first output column compared to the closed-form |sin(phi)|.

Declared acceptance condition: max Frobenius error < tol
(tol = declare_tolerance(scale=1)); max concurrence error < tol.

Expected cost: light.

Manuscript destination: Main text (Table I entangling-gate identity, Fig. 1).

Scientific limitations: Restricted to the minimal two-wire, one-layer
triangle; does not test coupling composed with additional trainable local
rotations on other columns (R1/R5 cover broader composition).
"""

import sys
from pathlib import Path

import common
import numpy as np
from scipy.linalg import expm

from photographiqml import MuTA
from photographiqml.diagnostics import concurrence

EXPERIMENT_ID = "R3"

X_REF = np.array([[0, 1], [1, 0]], dtype=complex)


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0)
    phis = np.linspace(-2 * np.pi, 2 * np.pi, 61)
    model = MuTA(2, one_column=True)
    rows = []
    for phi in phis:
        actual = model.unitary({"alpha.w1.c1": float(phi)})
        expected = expm(0.5j * phi * np.kron(X_REF, X_REF))
        error = common.frobenius_error(actual, expected)
        actual_concurrence = concurrence(actual[:, 0])
        expected_concurrence = abs(np.sin(phi))
        rows.append(
            {
                "phi": float(phi),
                "unitary_error": error,
                "actual_concurrence": actual_concurrence,
                "expected_concurrence": expected_concurrence,
                "concurrence_error": abs(actual_concurrence - expected_concurrence),
            }
        )

    max_unitary_error = max(r["unitary_error"] for r in rows)
    max_concurrence_error = max(r["concurrence_error"] for r in rows)
    status = "pass" if max(max_unitary_error, max_concurrence_error) < tol else "fail"

    common.save_result(
        rows,
        "R3_entangling_identity",
        extra={
            "protocol": "Two-wire pivot coupling vs. exp(i*phi*XX/2) (scipy.linalg.expm)",
            "oracle_class": "A",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max Frobenius error and max concurrence error < {tol:.3e}",
            "max_unitary_error": max_unitary_error,
            "max_concurrence_error": max_concurrence_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].semilogy(
        phis,
        [r["unitary_error"] for r in rows],
        color=common.COLORS["photographiqml"],
        marker="o",
        markersize=2,
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[0].set(title="exp(i*phi*XX/2) Frobenius error", xlabel="phi (rad)", ylabel="error")
    axes[0].legend()
    axes[1].plot(
        phis,
        [r["expected_concurrence"] for r in rows],
        color=common.COLORS["analytic"],
        label="analytic |sin(phi)|",
    )
    axes[1].plot(
        phis,
        [r["actual_concurrence"] for r in rows],
        "--",
        color=common.COLORS["photographiqml"],
        label="MuTA output concurrence",
    )
    axes[1].set(title="Induced concurrence", xlabel="phi (rad)", ylabel="concurrence")
    axes[1].legend()
    fig.suptitle(f"R3: Ising-XX entangling identity (status={status})")
    common.save_figure(fig, "R3_entangling_identity")
    plt.close(fig)

    common.print_summary(
        "R3 entangling identity",
        n_points=len(phis),
        tol=tol,
        max_unitary_error=max_unitary_error,
        max_concurrence_error=max_concurrence_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R3 failed: unitary={max_unitary_error} concurrence={max_concurrence_error}"
        )


if __name__ == "__main__":
    main()


## R4: Exhaustive adaptive-branch and byproduct-correction verification.

Scientific question: For every possible measurement outcome branch of a small
MuTA graph, does the full graph-state contraction with explicit Pauli
byproduct correction reproduce the same target output state that the
production frontier-translation algorithm (MuTA.run) computes deterministically
without ever branching, and do the branch probabilities form a valid
distribution?

Theory/equations: MBQC's deferred-measurement / adaptive-correction theorem
guarantees that for every one of the 2^m measurement branches (m = number of
measured nodes), applying the corresponding Pauli byproduct correction to the
post-measurement state reproduces the same logical output density matrix, and
the branch probabilities sum to 1 with none exceeding 1/2^m by more than
floating-point roundoff when angles are algebraically generic (each two-outcome
measurement bisects probability mass evenly is NOT generally true for nonzero
angles; only structural properties -- exact target agreement, sum to 1, all
nonnegative -- are the checked invariants here).

Functionality tested: photographiqml.validation.contract_branch (an
independent full graph-state contraction with explicit correction), compared
against MuTA.run's production frontier-translation execution.

Oracle and independence class: D (independent code path within the package --
contract_branch performs full Kronecker graph-state construction and
sequential projective measurement/correction, an algorithmically distinct
route from logical.execute's frontier-translation trick that never branches).

Exact/approximate/statistical status: exact, up to floating-point roundoff
(graphs capped at 16 nodes per contract_branch's own guard).

Primary metric: max density-matrix Frobenius error between every branch's
corrected output and MuTA.run's target, and |sum(branch probabilities) - 1|.

Declared acceptance condition: max branch error < tol
(tol = declare_tolerance(scale=1, safety_factor=100) to absorb the extra
floating-point operations of full graph-state contraction); probability-sum
error < tol; all branch probabilities >= -tol.

Expected cost: light (graphs capped at 10-16 nodes; at most 256 branches).

Manuscript destination: Appendix (independent-contraction cross-check
supporting Fig. 1 / Table II adaptive flow).

Scientific limitations: Exhaustive verification is only tractable up to
contract_branch's 16-node cap (one- and two-wire, one-layer models here);
larger models are not exhaustively checked this way (see R5 for a
statistically sampled, independently written contraction on larger models).

In [ ]:
"""R4: Exhaustive adaptive-branch and byproduct-correction verification.

Scientific question: For every possible measurement outcome branch of a small
MuTA graph, does the full graph-state contraction with explicit Pauli
byproduct correction reproduce the same target output state that the
production frontier-translation algorithm (MuTA.run) computes deterministically
without ever branching, and do the branch probabilities form a valid
distribution?

Theory/equations: MBQC's deferred-measurement / adaptive-correction theorem
guarantees that for every one of the 2^m measurement branches (m = number of
measured nodes), applying the corresponding Pauli byproduct correction to the
post-measurement state reproduces the same logical output density matrix, and
the branch probabilities sum to 1 with none exceeding 1/2^m by more than
floating-point roundoff when angles are algebraically generic (each two-outcome
measurement bisects probability mass evenly is NOT generally true for nonzero
angles; only structural properties -- exact target agreement, sum to 1, all
nonnegative -- are the checked invariants here).

Functionality tested: photographiqml.validation.contract_branch (an
independent full graph-state contraction with explicit correction), compared
against MuTA.run's production frontier-translation execution.

Oracle and independence class: D (independent code path within the package --
contract_branch performs full Kronecker graph-state construction and
sequential projective measurement/correction, an algorithmically distinct
route from logical.execute's frontier-translation trick that never branches).

Exact/approximate/statistical status: exact, up to floating-point roundoff
(graphs capped at 16 nodes per contract_branch's own guard).

Primary metric: max density-matrix Frobenius error between every branch's
corrected output and MuTA.run's target, and |sum(branch probabilities) - 1|.

Declared acceptance condition: max branch error < tol
(tol = declare_tolerance(scale=1, safety_factor=100) to absorb the extra
floating-point operations of full graph-state contraction); probability-sum
error < tol; all branch probabilities >= -tol.

Expected cost: light (graphs capped at 10-16 nodes; at most 256 branches).

Manuscript destination: Appendix (independent-contraction cross-check
supporting Fig. 1 / Table II adaptive flow).

Scientific limitations: Exhaustive verification is only tractable up to
contract_branch's 16-node cap (one- and two-wire, one-layer models here);
larger models are not exhaustively checked this way (see R5 for a
statistically sampled, independently written contraction on larger models).
"""

import itertools
import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.models import haar_states
from photographiqml.validation import contract_branch

EXPERIMENT_ID = "R4"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=100.0)
    rows = []
    configs = [
        {"n_wires": 1, "one_column": True, "seeds": (0, 1, 2)},
        {"n_wires": 2, "one_column": True, "seeds": (0, 1, 2)},
    ]
    for cfg in configs:
        model = MuTA(cfg["n_wires"], 1, one_column=cfg["one_column"])
        n_measured = len(model.measurement_order)
        assert len(model.graph) <= 16, "contract_branch cap exceeded"
        for seed in cfg["seeds"]:
            state = haar_states(cfg["n_wires"], 1, seed)[0]
            parameters = model.initialize(seed + 1000, scale=1.5)
            target = model.run(state, parameters).density_matrix
            total_probability = 0.0
            max_branch_error = 0.0
            min_probability = np.inf
            for outcomes in itertools.product((0, 1), repeat=n_measured):
                output, probability = contract_branch(model, state, parameters, outcomes)
                branch_error = common.frobenius_error(np.outer(output, output.conj()), target)
                max_branch_error = max(max_branch_error, branch_error)
                total_probability += probability
                min_probability = min(min_probability, probability)
            rows.append(
                {
                    "n_wires": cfg["n_wires"],
                    "seed": seed,
                    "n_measured": n_measured,
                    "n_branches": 2**n_measured,
                    "max_branch_error": max_branch_error,
                    "probability_sum_error": abs(total_probability - 1.0),
                    "min_branch_probability": min_probability,
                }
            )

    max_branch_error = max(r["max_branch_error"] for r in rows)
    max_prob_error = max(r["probability_sum_error"] for r in rows)
    min_probability = min(r["min_branch_probability"] for r in rows)
    status = (
        "pass"
        if (max_branch_error < tol and max_prob_error < tol and min_probability > -tol)
        else "fail"
    )

    common.save_result(
        rows,
        "R4_adaptive_branches",
        extra={
            "protocol": "Exhaustive branch contraction (contract_branch) vs. MuTA.run target, 1- and 2-wire models",
            "oracle_class": "D",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max branch error and |sum(p)-1| < {tol:.3e}; min probability > -tol",
            "max_branch_error": max_branch_error,
            "max_probability_sum_error": max_prob_error,
            "min_branch_probability": min_probability,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "D", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    xs = range(len(rows))
    axes[0].semilogy(
        xs, [r["max_branch_error"] for r in rows], "o", color=common.COLORS["photographiqml"]
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[0].set(
        title="Max corrected-branch error per config",
        xlabel="config index",
        ylabel="Frobenius error",
    )
    axes[0].legend()
    axes[1].bar(
        xs, [r["probability_sum_error"] for r in rows], color=common.COLORS["photographiqml"]
    )
    axes[1].axhline(tol, color=common.COLORS["acceptance"], linestyle="--")
    axes[1].set(title="|sum(branch probabilities) - 1|", xlabel="config index", ylabel="error")
    fig.suptitle(f"R4: Exhaustive adaptive-branch verification (status={status})")
    common.save_figure(fig, "R4_adaptive_branches")
    plt.close(fig)

    common.print_summary(
        "R4 adaptive branches",
        n_configs=len(rows),
        tol=tol,
        max_branch_error=max_branch_error,
        max_probability_sum_error=max_prob_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R4 failed: branch={max_branch_error} prob={max_prob_error}")


if __name__ == "__main__":
    main()


## 05_brute_force_agreement



In [ ]:
"""R5: Random logical-state agreement against an independently written
brute-force MBQC contraction.

Scientific question: Does MuTA.run's frontier-translation execution agree
with a genuinely independent, freshly written full graph-state contraction
(distinct code from both logical.py's cz/local_gate and
validation.py's contract_branch) across a broad, randomly sampled set of
wire/layer/one_column configurations and random Haar input states?

Theory/equations: MBQC deferred-measurement theorem: fixing every measured
node's outcome bit to 0 (a single deterministic branch) requires *no* Pauli
byproduct correction, since corrections only trigger on bit=1 (see
lowering.node_frame / ansatz/muta.py's corrections map). Since R4 already
proves exhaustively (via a *different* independent implementation,
contract_branch) that every corrected branch reproduces the same target, the
uncorrected zero-outcome branch alone must equal the target exactly -- this
experiment checks that invariant with a third, independently coded
contraction, built here via kron-embedded projector sums for CZ (not
photographiqml.logical.cz's bit-parity trick) and tensor-reshape projective
measurement (not photographiqml.logical.local_gate's moveaxis trick).

Functionality tested: MuTA.run (production execute()) across a random
sweep of n_wires/n_layers/one_column configurations, capped at <=20 graph
nodes for dense state-vector tractability.

Oracle and independence class: D (independent code path -- CZ, measurement
and normalization are all implemented fresh in this script using different
numpy primitives than logical.py/validation.py; the abstract measurement/
correction *equations* are necessarily shared physics, not shared code).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max density-matrix Frobenius error between the independent
contraction's output and MuTA.run's LogicalResult.density_matrix.

Declared acceptance condition: max Frobenius error < tol
(tol = declare_tolerance(scale=1, safety_factor=200), the larger safety
factor absorbing the extra floating-point operations of a dense contraction
over up to 2**20 amplitudes).

Expected cost: light-to-moderate (largest config: 2 wires, 2 layers,
one_column=True -> 18 graph nodes, 2**18 amplitudes).

Manuscript destination: Appendix (third independent cross-check of Fig. 1's
logical execution, complementing R4's exhaustive small-graph branch check).

Scientific limitations: Restricted to <=20 graph nodes by dense state-vector
memory; only the deterministic zero-outcome branch is checked per
configuration (branch/correction exhaustiveness itself is R4's job, not
R5's).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.models import haar_states

EXPERIMENT_ID = "R5"
MAX_NODES = 20


def independent_contraction(model, input_state, angles):
    """Fresh full graph-state contraction: kron-embedded CZ + tensor-reshape
    projective measurement onto the deterministic zero-outcome branch."""
    order = list(model.input_nodes) + [v for v in model.graph.nodes if v not in model.input_nodes]
    total = len(order)
    if total > MAX_NODES:
        raise ValueError(f"Independent contraction capped at {MAX_NODES} nodes; got {total}")
    plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
    state = np.asarray(input_state, dtype=complex)
    for _ in range(total - model.n_wires):
        state = np.kron(state, plus)

    # Apply every CZ edge as an elementwise sign flip on the reshaped tensor
    # (a diagonal-projector-sum operation), not logical.cz's bit-parity trick.
    tensor = state.reshape([2] * total)
    for u, v in model.graph.edges:
        pu, pv = order.index(u), order.index(v)
        lo, hi = sorted((pu, pv))
        index = [slice(None)] * total
        index[lo] = 1
        index[hi] = 1
        tensor[tuple(index)] *= -1
    state = tensor.reshape(-1)

    remaining = list(order)
    probability = 1.0
    for node in model.measurement_order:
        alpha = angles[model.parameter_name(node)]
        p0 = np.array([1.0, np.exp(-1j * alpha)], dtype=complex) / np.sqrt(2)
        position = remaining.index(node)
        n_now = len(remaining)
        reshaped = state.reshape(2**position, 2, 2 ** (n_now - position - 1))
        # Contract directly against p0's raw coefficients (not conjugated): this
        # matches the measurement-basis convention |m> such that p0 IS <m|,
        # i.e. p0 = [1, exp(-i*alpha)]/sqrt(2) means <m| = p0 (bra components
        # given directly, not derived by conjugating a stated ket).
        state = np.tensordot(reshaped, p0, axes=([1], [0])).reshape(-1)
        mass = float(np.vdot(state, state).real)
        probability *= mass
        state = state / np.sqrt(mass)
        remaining.remove(node)

    output_positions = [remaining.index(v) for v in model.output_nodes]
    state = state.reshape([2] * model.n_wires).transpose(output_positions).reshape(-1)
    return state, probability


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=200.0)
    configs = [
        {"n_wires": 1, "n_layers": 1, "one_column": True},
        {"n_wires": 1, "n_layers": 3, "one_column": True},
        {"n_wires": 2, "n_layers": 1, "one_column": True},
        {"n_wires": 2, "n_layers": 1, "one_column": False},
        {"n_wires": 2, "n_layers": 2, "one_column": True},
        {"n_wires": 3, "n_layers": 1, "one_column": True},
    ]
    rows = []
    for cfg in configs:
        model = MuTA(cfg["n_wires"], cfg["n_layers"], one_column=cfg["one_column"])
        n_nodes = len(model.graph)
        if n_nodes > MAX_NODES:
            rows.append(
                {
                    **cfg,
                    "n_nodes": n_nodes,
                    "skipped": True,
                    "reason": f"exceeds {MAX_NODES}-node cap",
                }
            )
            continue
        for seed in (0, 1, 2, 3):
            state = haar_states(cfg["n_wires"], 1, seed)[0]
            angles = dict(
                zip(
                    model.trainable_parameters(),
                    common.rng(seed + 500).uniform(-np.pi, np.pi, model.n_parameters),
                )
            )
            target = model.run(state, angles).density_matrix
            independent_state, branch_probability = independent_contraction(model, state, angles)
            error = common.frobenius_error(
                np.outer(independent_state, independent_state.conj()), target
            )
            rows.append(
                {
                    **cfg,
                    "n_nodes": n_nodes,
                    "seed": seed,
                    "skipped": False,
                    "density_matrix_error": error,
                    "branch_probability": branch_probability,
                }
            )

    active = [r for r in rows if not r["skipped"]]
    max_error = max(r["density_matrix_error"] for r in active)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R5_brute_force_agreement",
        extra={
            "protocol": "MuTA.run vs. freshly written kron/tensor-reshape zero-branch contraction",
            "oracle_class": "D",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max density-matrix error < {tol:.3e}",
            "max_error": max_error,
            "n_configurations_checked": len(active),
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "D", "status": status},
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    labels = [
        f"{r['n_wires']}w{r['n_layers']}L{'1c' if r['one_column'] else 'nc'} s{r['seed']}"
        for r in active
    ]
    ax.semilogy(
        range(len(active)),
        [r["density_matrix_error"] for r in active],
        "o",
        color=common.COLORS["photographiqml"],
    )
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set_xticks(range(len(active)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=6.5)
    ax.set(
        title=f"R5: independent brute-force contraction vs. MuTA.run (status={status})",
        ylabel="density-matrix Frobenius error",
    )
    ax.legend()
    common.save_figure(fig, "R5_brute_force_agreement")
    plt.close(fig)

    common.print_summary(
        "R5 brute-force agreement",
        n_configs=len(active),
        tol=tol,
        max_error=max_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R5 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 06_batch_consistency



In [ ]:
"""R6: run_batch versus repeated scalar run, including empty batches and
complex normalized inputs.

Scientific question: Does MuTA.run_batch produce exactly the same output as
calling MuTA.run once per row (same shared parameters), for arbitrary complex
normalized inputs, and does it correctly handle a zero-row batch without
error?

Theory/equations: run_batch is documented (docs/api.md) as
"normalized complex rows (N,2**n), shared parameters -> complex (N,2**n);
empty batch supported". This is a self-consistency/structural property, not
a new physics claim.

Functionality tested: MuTA.run_batch vs. MuTA.run (photographiqml/ansatz/muta.py).

Oracle and independence class: E (structural/self-consistency test) --
run_batch is checked against the same model's own scalar run, not an
external or independently derived reference.

Exact/approximate/statistical status: exact (bitwise-identical arithmetic
path is expected, since run_batch's implementation literally calls run per
row; this experiment verifies that documented contract holds, including at
its two declared edge cases).

Primary metric: max per-row Frobenius error between run_batch and repeated
run; behavior on an empty batch (shape and absence of exception).

Declared acceptance condition: max per-row error == 0 (bitwise, since both
paths execute identical code); run_batch([]) returns shape (0, 2**n) without
raising.

Expected cost: light.

Manuscript destination: Appendix (API contract verification table).

Scientific limitations: This is a self-consistency check of the public
interface, not independent physics validation (see R1-R5, R7-R9 for that).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.models import haar_states

EXPERIMENT_ID = "R6"


def main():
    plt = common.setup_style()
    rows = []
    for n_wires in (1, 2, 3):
        for n_batch in (0, 1, 5, 17):
            model = MuTA(n_wires, 2)
            parameters = model.initialize(seed=n_wires * 100 + n_batch, scale=1.0)
            states = (
                haar_states(n_wires, n_batch, seed=n_wires * 100 + n_batch + 7)
                if n_batch
                else np.empty((0, 2**n_wires), dtype=complex)
            )
            batch_output = model.run_batch(states, parameters)
            shape_ok = batch_output.shape == (n_batch, 2**n_wires)
            if n_batch:
                scalar_outputs = np.array([model.run(s, parameters).state for s in states])
                error = common.frobenius_error(batch_output, scalar_outputs)
            else:
                error = 0.0
            rows.append(
                {
                    "n_wires": n_wires,
                    "n_batch": n_batch,
                    "shape_ok": shape_ok,
                    "error": error,
                }
            )

    max_error = max(r["error"] for r in rows)
    all_shapes_ok = all(r["shape_ok"] for r in rows)
    status = "pass" if (max_error == 0.0 and all_shapes_ok) else "fail"

    common.save_result(
        rows,
        "R6_batch_consistency",
        extra={
            "protocol": "run_batch vs. repeated scalar run, including n_batch=0 edge case",
            "oracle_class": "E",
            "status_category": "exact",
            "acceptance_condition": "max_error == 0 (bitwise) and all output shapes correct",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(6, 3.6))
    labels = [f"n{r['n_wires']}b{r['n_batch']}" for r in rows]
    colors = [
        common.COLORS["photographiqml"]
        if r["shape_ok"] and r["error"] == 0
        else common.COLORS["piquasso"]
        for r in rows
    ]
    # All-zero values render invisibly as bars; use markers so every tested
    # case is visibly plotted, not just an empty axes.
    ax.scatter(range(len(rows)), [r["error"] for r in rows], color=colors, zorder=3)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set(
        title=f"R6: run_batch vs. repeated run, {len(rows)} cases (status={status})",
        ylabel="Frobenius error",
    )
    common.save_figure(fig, "R6_batch_consistency")
    plt.close(fig)

    common.print_summary(
        "R6 batch consistency", n_cases=len(rows), max_error=max_error, status=status
    )
    if status != "pass":
        raise AssertionError(f"R6 failed: max_error={max_error} all_shapes_ok={all_shapes_ok}")


if __name__ == "__main__":
    main()


## 07_semantic_topology



In [ ]:
"""R7: Semantic topology agreement between MuTA and independently constructed
MentPy circuits.

Scientific question: For every combination of wire count, requested layers,
one_column and restrict_trainable, does MentPy's own `templates.muta` build a
graph that is edge-for-edge isomorphic to PhotoGraphiQML's MuTA.graph once
both are expressed in the same semantic (wire, column) coordinates, and do
input/output node orderings agree?

Theory/equations: GraphState equality via isomorphism is insufficient for
semantic comparison (docs/research/mentpy-audit.md); the correct invariant is
edge agreement under the *causal* (wire, column) labeling recovered by
walking MentPy's own flow function from each input node
(photographiqml.validation.mentpy_reference), not an arbitrary vertex
relabeling.

Functionality tested: MuTA graph construction (ansatz/muta.py) vs.
mentpy.templates.muta, compared via validation.mentpy_reference's semantic
mapping (built by walking reference.flow(node) from each input, an operation
this experiment does not repeat itself but whose *result* -- the edge set --
it independently re-derives and checks against MuTA.graph.edges).

Oracle and independence class: B (independent external implementation --
MentPy is a separate Apache-2.0 package validating qubit structure only).

Exact/approximate/statistical status: exact (integer edge-set equality).

Primary metric: symmetric-difference size between the semantically mapped
MentPy edge set and MuTA.graph.edges; input/output ordering mismatches.

Declared acceptance condition: symmetric difference == 0 and input/output
orderings agree for every swept configuration with connections=None
(mentpy_reference only supports full connectivity).

Expected cost: light (graph construction only, no state simulation).

Manuscript destination: Main text (Fig. 2/3, MentPy cross-validation) and
Table II (structural agreement).

Scientific limitations: MentPy validates the ideal logical qubit layer only
(never a finite-GKP oracle); custom `connections` (non-default triangle
wiring) has no MentPy counterpart and is excluded from this sweep.
"""

import sys
from pathlib import Path

import common

from photographiqml import MuTA
from photographiqml.validation import mentpy_reference

EXPERIMENT_ID = "R7"


def main():
    plt = common.setup_style()
    rows = []
    for n_wires in (1, 2, 3):
        for n_layers in (1, 2):
            for one_column in (False, True):
                for restrict_trainable in (False, True):
                    model = MuTA(
                        n_wires,
                        n_layers,
                        one_column=one_column,
                        restrict_trainable=restrict_trainable,
                    )
                    reference, mapping = mentpy_reference(model)
                    mentpy_edges = {
                        frozenset((mapping[u], mapping[v])) for u, v in reference.graph.edges
                    }
                    muta_edges = {frozenset(e) for e in model.graph.edges}
                    symmetric_difference = mentpy_edges ^ muta_edges
                    mapped_inputs = tuple(mapping[v] for v in reference.input_nodes)
                    mapped_outputs = tuple(mapping[v] for v in reference.output_nodes)
                    rows.append(
                        {
                            "n_wires": n_wires,
                            "n_layers": n_layers,
                            "one_column": one_column,
                            "restrict_trainable": restrict_trainable,
                            "mentpy_nodes": len(reference.graph.nodes),
                            "muta_nodes": len(model.graph.nodes),
                            "mentpy_edges": len(mentpy_edges),
                            "muta_edges": len(muta_edges),
                            "symmetric_difference": len(symmetric_difference),
                            "inputs_match": mapped_inputs == model.input_nodes,
                            "outputs_match": mapped_outputs == model.output_nodes,
                        }
                    )

    max_symdiff = max(r["symmetric_difference"] for r in rows)
    all_io_ok = all(r["inputs_match"] and r["outputs_match"] for r in rows)
    status = "pass" if (max_symdiff == 0 and all_io_ok) else "fail"

    common.save_result(
        rows,
        "R7_semantic_topology",
        extra={
            "protocol": "MuTA.graph vs. MentPy templates.muta, compared via semantic (wire,column) mapping",
            "oracle_class": "B",
            "status_category": "exact",
            "acceptance_condition": "symmetric_difference == 0 and input/output ordering matches for every configuration",
            "max_symmetric_difference": max_symdiff,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "B", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [
        f"n{r['n_wires']}L{r['n_layers']}{'1c' if r['one_column'] else 'nc'}{'r' if r['restrict_trainable'] else 'f'}"
        for r in rows
    ]
    colors = [
        common.COLORS["photographiqml"]
        if r["symmetric_difference"] == 0 and r["inputs_match"] and r["outputs_match"]
        else common.COLORS["piquasso"]
        for r in rows
    ]
    # All-zero values render invisibly as bars; use markers so every tested
    # configuration is visibly plotted, not just an empty axes.
    ax.scatter(range(len(rows)), [r["symmetric_difference"] for r in rows], color=colors, zorder=3)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=6)
    ax.set(
        title=f"R7: MuTA vs. MentPy semantic edge-set agreement, {len(rows)} configurations (status={status})",
        ylabel="|symmetric difference|",
    )
    common.save_figure(fig, "R7_semantic_topology")
    plt.close(fig)

    common.print_summary(
        "R7 semantic topology",
        n_configs=len(rows),
        max_symmetric_difference=max_symdiff,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R7 failed: max_symdiff={max_symdiff} all_io_ok={all_io_ok}")


if __name__ == "__main__":
    main()


## 08_flow_dependency_order



In [ ]:
"""R8: Flow and dependency-order agreement against MentPy, comparing causal
validity rather than exact tie-breaking order.

Scientific question: Does MentPy's own causal-flow successor/correction sets
(reference.flow(node), reference.flow.correction_op(node)) agree with MuTA's
production correction map (model.corrections), and is MentPy's own
measurement_order consistent with MuTA's dependency DAG once both are
expressed in the same semantic coordinates?

Theory/equations: docs/research/mentpy-audit.md: "Flow permits multiple total
orders; compare causal validity rather than requiring identical tie
breaking." This experiment therefore checks (i) X/Z correction-target set
agreement per node (an order-independent invariant) and (ii) that MentPy's
own valid measurement_order respects every edge of MuTA's dependency_graph
(a necessary condition for causal consistency, not a demand for identical
ordering of causally independent nodes).

Functionality tested: Flow.__call__, Flow.correction_op (mentpy), vs.
MuTA.corrections and MuTA.dependency_graph (photographiqml).

Oracle and independence class: B (independent external implementation).

Exact/approximate/statistical status: exact (set equality / DAG-consistency,
both discrete).

Primary metric: number of nodes whose mapped X-correction target or Z-target
set disagrees with model.corrections; number of dependency_graph edges
violated by MentPy's own measurement_order.

Declared acceptance condition: both counts are 0 for every configuration.

Expected cost: light.

Manuscript destination: Main text (Fig. 2/3 supporting causal-flow
agreement); Table II.

Scientific limitations: Restricted to configurations where MentPy finds a
causal flow (true for every MuTA configuration tested here, since the paper
ansatz is causal-flow-friendly by construction); gflow/pflow fallbacks are
not separately exercised.

Documented convention difference (not a defect): MentPy's raw
Flow.correction_op(node) reports the *full* Pauli frame over every graph
qubit, including a formal Z-component on the just-measured node itself
(since that node is a graph-neighbor of its own flow successor, it appears
in odd_neighborhood({successor})). MuTA.corrections explicitly excludes the
measured node (`z = frozenset(graph.neighbors(successor)) - {node}` in
ansatz/muta.py), since a Z correction on an already destructively-measured
qubit has no physical effect on the surviving register. This experiment
subtracts the measured node from MentPy's mapped Z-target set before
comparing, and records that subtraction explicitly (never silently) in the
saved JSON's `self_correction_dropped` field.
"""

import sys
from pathlib import Path

import common

from photographiqml import MuTA
from photographiqml.validation import mentpy_reference

EXPERIMENT_ID = "R8"


def main():
    plt = common.setup_style()
    rows = []
    for n_wires in (1, 2, 3):
        for n_layers in (1, 2):
            for one_column in (False, True):
                model = MuTA(n_wires, n_layers, one_column=one_column)
                reference, mapping = mentpy_reference(model)
                reverse = {v: k for k, v in mapping.items()}
                order_violations = 0
                correction_mismatches = 0
                n_measured = len(model.corrections)
                self_correction_dropped = 0
                mentpy_order_positions = {
                    node: i for i, node in enumerate(reference.measurement_order)
                }
                for source, (successor, z_targets) in model.corrections.items():
                    mentpy_node = reverse[source]
                    # MentPy's own causal order must place this node before its
                    # mapped successor (a necessary consequence of any valid flow).
                    successor_mentpy = reverse[successor]
                    if (
                        mentpy_order_positions[mentpy_node]
                        >= mentpy_order_positions[successor_mentpy]
                    ):
                        order_violations += 1
                    for target in z_targets:
                        target_mentpy = reverse[target]
                        if (
                            mentpy_order_positions[mentpy_node]
                            >= mentpy_order_positions[target_mentpy]
                        ):
                            order_violations += 1
                    # Correction-set agreement: MentPy's correction_op X-part must
                    # be exactly {successor}; Z-part must equal z_targets, mapped.
                    n_nodes = reference.graph.number_of_nodes()
                    pauli = reference.flow.correction_op(mentpy_node)
                    x_bits = set(pauli.matrix[0, :n_nodes].nonzero()[0].tolist())
                    z_bits = set(pauli.matrix[0, n_nodes:].nonzero()[0].tolist())
                    x_targets_semantic = {mapping[v] for v in x_bits}
                    z_targets_semantic = {mapping[v] for v in z_bits}
                    if source in z_targets_semantic:
                        z_targets_semantic = z_targets_semantic - {source}
                        self_correction_dropped += 1
                    if x_targets_semantic != {successor} or z_targets_semantic != set(z_targets):
                        correction_mismatches += 1
                rows.append(
                    {
                        "n_wires": n_wires,
                        "n_layers": n_layers,
                        "one_column": one_column,
                        "n_measured": n_measured,
                        "order_violations": order_violations,
                        "correction_mismatches": correction_mismatches,
                        "self_correction_dropped": self_correction_dropped,
                    }
                )

    max_order_violations = max(r["order_violations"] for r in rows)
    max_correction_mismatches = max(r["correction_mismatches"] for r in rows)
    status = "pass" if (max_order_violations == 0 and max_correction_mismatches == 0) else "fail"

    common.save_result(
        rows,
        "R8_flow_dependency_order",
        extra={
            "protocol": "MentPy causal flow/correction_op vs. MuTA.corrections and dependency_graph",
            "oracle_class": "B",
            "status_category": "exact",
            "acceptance_condition": "order_violations == 0 and correction_mismatches == 0 for every configuration",
            "max_order_violations": max_order_violations,
            "max_correction_mismatches": max_correction_mismatches,
            "status": status,
            "convention_note": "MentPy's raw correction_op Z-part includes a formal self-correction on the measured node itself; this is subtracted before comparison (see self_correction_dropped per row) as documented in the docstring, not treated as a defect",
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "B", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    labels = [f"n{r['n_wires']}L{r['n_layers']}{'1c' if r['one_column'] else 'nc'}" for r in rows]
    for ax, key, title in zip(
        axes,
        ("order_violations", "correction_mismatches"),
        ("Causal-order violations", "X/Z correction-set mismatches"),
    ):
        colors = [
            common.COLORS["photographiqml"] if r[key] == 0 else common.COLORS["piquasso"]
            for r in rows
        ]
        # All-zero values render invisibly as bars; use markers so every
        # tested configuration is visibly plotted, not just an empty axes.
        ax.scatter(range(len(rows)), [r[key] for r in rows], color=colors, zorder=3)
        ax.set_xticks(range(len(rows)))
        ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=6)
        ax.set_title(f"{title} ({len(rows)} configurations)")
    fig.suptitle(f"R8: flow/dependency-order agreement with MentPy (status={status})")
    common.save_figure(fig, "R8_flow_dependency_order")
    plt.close(fig)

    common.print_summary(
        "R8 flow/dependency order",
        n_configs=len(rows),
        max_order_violations=max_order_violations,
        max_correction_mismatches=max_correction_mismatches,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R8 failed: order_violations={max_order_violations} correction_mismatches={max_correction_mismatches}"
        )


if __name__ == "__main__":
    main()


## 09_state_density_agreement



In [ ]:
"""R9: Random numerical state and density-matrix agreement against MentPy
across multiple seeds and model sizes.

Scientific question: For random logical input states and random trainable
angles, does MuTA.run's output density matrix agree numerically with an
independently executed MentPy PatternSimulator run on the semantically
mapped circuit, across a range of model sizes and random seeds?

Theory/equations: photographiqml.validation.compare_mentpy constructs the
MentPy reference (mentpy_reference, fixing frozen columns explicitly),
simulates it with mp.PatternSimulator(backend="numpy-sv", output_form="dm")
over a flow-respecting schedule window, and returns the max absolute
density-matrix entrywise difference against MuTA.run's own density_matrix.

Functionality tested: MuTA.run (photographiqml.logical.execute) vs.
mp.PatternSimulator numpy statevector backend (mentpy), via
validation.compare_mentpy.

Oracle and independence class: B (independent external implementation).

Exact/approximate/statistical status: exact, up to floating-point roundoff
(both are deterministic numpy statevector simulations of the same finite
circuit).

Primary metric: max density-matrix entrywise absolute error, aggregated over
n_seeds independent random (state, parameter) draws per model configuration.

Declared acceptance condition: max error across the full seed x configuration
grid is below tol = declare_tolerance(scale=1, safety_factor=1000) (a larger
safety factor accounts for MentPy's own independent linear-algebra pipeline,
distinct floating-point operation ordering, and its window-limited pattern
simulator).

Expected cost: light-to-moderate (up to 3-wire, 2-layer models; 8 seeds each).

Manuscript destination: Main text (Fig. 3, headline MentPy agreement plot).

Scientific limitations: MentPy validates the ideal logical qubit layer only.
The numpy statevector backend defaults to the zero-outcome branch and
disallows random outcomes (docs/research/mentpy-audit.md); this experiment
therefore validates deterministic execution, not adaptive branch sampling
(see R4/R5 for exhaustive/independent branch checks within PhotoGraphiQML
itself, and R12 for instrument branch agreement).
"""

import sys
from pathlib import Path

import common

from photographiqml import MuTA
from photographiqml.models import haar_states
from photographiqml.validation import MENTPY_COMMIT, compare_mentpy

EXPERIMENT_ID = "R9"

N_SEEDS = 8


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1000.0)
    rows = []
    configs = [
        {"n_wires": 1, "n_layers": 1, "one_column": True},
        {"n_wires": 1, "n_layers": 3, "one_column": True},
        {"n_wires": 2, "n_layers": 1, "one_column": True},
        {"n_wires": 2, "n_layers": 1, "one_column": False},
        {"n_wires": 2, "n_layers": 2, "one_column": True},
        {"n_wires": 3, "n_layers": 1, "one_column": True},
    ]
    for cfg in configs:
        model = MuTA(cfg["n_wires"], cfg["n_layers"], one_column=cfg["one_column"])
        for seed in range(N_SEEDS):
            state = haar_states(cfg["n_wires"], 1, seed=1000 * seed + cfg["n_wires"])[0]
            parameters = model.initialize(seed=2000 * seed + cfg["n_layers"], scale=2.0)
            error = compare_mentpy(model, state, parameters)
            rows.append({**cfg, "seed": seed, "density_matrix_error": error})

    max_error = max(r["density_matrix_error"] for r in rows)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R9_state_density_agreement",
        extra={
            "protocol": "compare_mentpy: MuTA.run vs. MentPy PatternSimulator (numpy-sv, output_form=dm)",
            "oracle_class": "B",
            "status_category": "exact",
            "mentpy_commit": MENTPY_COMMIT,
            "tolerance": tol,
            "acceptance_condition": f"max density-matrix error < {tol:.3e}",
            "max_error": max_error,
            "n_seeds_per_config": N_SEEDS,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "B", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    configs_labels = [
        f"n{c['n_wires']}L{c['n_layers']}{'1c' if c['one_column'] else 'nc'}" for c in configs
    ]
    for i, cfg in enumerate(configs):
        errors = [
            r["density_matrix_error"]
            for r in rows
            if r["n_wires"] == cfg["n_wires"]
            and r["n_layers"] == cfg["n_layers"]
            and r["one_column"] == cfg["one_column"]
        ]
        ax.scatter([i] * len(errors), errors, color=common.COLORS["mentpy"], alpha=0.7, s=18)
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set_yscale("log")
    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels(configs_labels, rotation=45, ha="right", fontsize=7)
    ax.set(
        title=f"R9: MuTA vs. MentPy density-matrix agreement (status={status})",
        ylabel="max |Delta rho|",
    )
    ax.legend()
    common.save_figure(fig, "R9_state_density_agreement")
    plt.close(fig)

    common.print_summary(
        "R9 state/density agreement",
        n_points=len(rows),
        tol=tol,
        max_error=max_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R9 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 10_trainability_audit



In [ ]:
"""R10: Explicit audit of trainability/frozen-column behavior, including the
documented MentPy restrict_trainable discrepancy.

Scientific question: (a) Does raw MentPy's `templates.muta(..., restrict_trainable=True)`
actually fix column-3 (the paper's non-trainable) measurement angles, or
merely remove them from a bookkeeping list that upstream graph-mutation
(add_edge/hstack) silently rebuilds anyway? (b) Does PhotoGraphiQML's own
restrict_trainable freezing mechanism (ParameterStore) remain robust across
every wire/layer/one_column configuration, unlike MentPy's fragile list-based
approach?

Theory/equations: docs/research/mentpy-audit.md: "`restrict_trainable`
removes column-3 nodes from a list but does not fix their Ment angles...
Only an unstacked one-wire block retains the list restriction... Adding
cross edges also rebuilds attributes... Only an unstacked one-wire block
retains the list restriction." Mechanistically: MBQCircuit.add_edge and
hstack both call MBQCircuit.__init__ -> _update_attributes(), which discards
any manually assigned `trainable_nodes` list and rebuilds it purely from each
Ment.is_trainable() (angle is None), so the restriction never touches the
Ment objects themselves and survives only when neither add_edge nor hstack
is ever invoked after it (n_wires==1 and n_layers==1).

Functionality tested: raw mentpy.templates.muta's restrict_trainable option
(mentpy) vs. photographiqml.parameters.ParameterStore.freeze /
ansatz/muta.py's restrict_trainable construction (photographiqml).

Oracle and independence class: B for the MentPy-side audit (independent
external implementation, inspected via its own public trainable_nodes/Ment
API); E (structural/self-consistency) for confirming PhotoGraphiQML's own
freeze mechanism rejects an override, which is a property of
photographiqml.parameters.ParameterStore.bind alone.

Exact/approximate/statistical status: exact (discrete membership/exception
checks).

Primary metric: (descriptive, not pass/fail) whether each column-3 semantic
node is present/absent from raw MentPy's trainable_nodes list and whether its
Ment.angle is None, per configuration; (pass/fail) whether PhotoGraphiQML
raises ValueError on every attempted override of a frozen column-3 parameter,
in every configuration.

Declared acceptance condition: PhotoGraphiQML raises ValueError for 100% of
attempted frozen-column-3 overrides, and model.trainable_parameters() never
contains a column-3 name when restrict_trainable=True, for every
configuration. MentPy's own behavior is recorded but is explicitly NOT
required to match any particular pattern (it is a documented, non-PhotoGraphiQML
finding, not a PhotoGraphiQML pass/fail criterion).

Expected cost: light.

Manuscript destination: Appendix (upstream-discrepancy documentation
supporting the release report's trainability claims).

Scientific limitations: This audits MentPy's public list/Ment API only;
whether the discrepancy is intended internal design or an oversight is not
determinable from black-box inspection, and PhotoGraphiQML's production code
is never modified to work around it (the model instead sidesteps MentPy's
fragile list convention entirely by fixing angles itself).
"""

import sys
from pathlib import Path

import common

from photographiqml import MuTA
from photographiqml.validation import mentpy_reference

EXPERIMENT_ID = "R10"


def main():
    plt = common.setup_style()
    rows = []
    configs = [
        (1, 1, True),
        (1, 2, True),
        (2, 1, True),
        (2, 1, False),
        (2, 2, True),
        (3, 1, True),
        (1, 1, False),
        (3, 2, True),
    ]
    for n_wires, n_layers, one_column in configs:
        model = MuTA(n_wires, n_layers, one_column=one_column, restrict_trainable=True)
        reference, mapping = mentpy_reference(model, fix_measurements=False)
        reverse = {v: k for k, v in mapping.items()}
        column3_nodes = [v for v in model.measured_nodes if v[1] % 4 == 3]

        # --- (a) raw MentPy discrepancy audit ---------------------------------
        list_restricted = 0  # absent from reference.trainable_nodes (list omitted)
        angle_actually_fixed = 0  # Ment.angle is not None (truly fixed, not just listed)
        for node in column3_nodes:
            mnode = reverse[node]
            in_list = mnode in reference.trainable_nodes
            if not in_list:
                list_restricted += 1
            if reference[mnode].angle is not None:
                angle_actually_fixed += 1

        # --- (b) PhotoGraphiQML's own freeze robustness ------------------------
        trainable_names = set(model.trainable_parameters())
        column3_names = {model.parameter_name(v) for v in column3_nodes}
        leaked_into_trainable = len(trainable_names & column3_names)
        override_rejected = 0
        computational_zero = [1] + [0] * (2**n_wires - 1)
        for name in column3_names:
            try:
                model.run(computational_zero, {name: 1.2345})
                overrode_silently = True
            except ValueError:
                overrode_silently = False
            if not overrode_silently:
                override_rejected += 1

        rows.append(
            {
                "n_wires": n_wires,
                "n_layers": n_layers,
                "one_column": one_column,
                "n_column3_nodes": len(column3_nodes),
                "mentpy_list_restricted": list_restricted,
                "mentpy_angle_actually_fixed": angle_actually_fixed,
                "mentpy_restriction_effective": angle_actually_fixed == len(column3_nodes)
                and len(column3_nodes) > 0,
                "photographiqml_leaked_into_trainable": leaked_into_trainable,
                "photographiqml_override_rejected": override_rejected,
                "photographiqml_robust": leaked_into_trainable == 0
                and override_rejected == len(column3_nodes),
            }
        )

    all_robust = all(r["photographiqml_robust"] for r in rows)
    status = "pass" if all_robust else "fail"
    n_mentpy_effective = sum(1 for r in rows if r["mentpy_restriction_effective"])

    common.save_result(
        rows,
        "R10_trainability_audit",
        extra={
            "protocol": "MentPy raw restrict_trainable list/Ment audit vs. PhotoGraphiQML ParameterStore freeze robustness",
            "oracle_class": "B/E",
            "status_category": "exact",
            "acceptance_condition": "photographiqml_robust True for every configuration (MentPy's own pattern is descriptive, not a pass/fail criterion)",
            "n_configs_mentpy_restriction_actually_effective": n_mentpy_effective,
            "n_configs_total": len(rows),
            "status": status,
            "finding": "MentPy's restrict_trainable only ever fixes the Ment angle (not just the list) when neither add_edge nor hstack runs afterward, i.e. n_wires==1 and n_layers==1; PhotoGraphiQML never relies on MentPy's list convention and freezes column-3 parameters directly and robustly in every configuration.",
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "B/E", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    labels = [f"n{r['n_wires']}L{r['n_layers']}{'1c' if r['one_column'] else 'nc'}" for r in rows]
    colors_mentpy = [
        common.COLORS["mentpy"]
        if r["mentpy_restriction_effective"]
        else common.COLORS["unsupported"]
        for r in rows
    ]
    # This fraction is 0 for every configuration (see docstring); a bar chart
    # would render as an empty axes, so use markers instead.
    axes[0].scatter(
        range(len(rows)),
        [r["mentpy_angle_actually_fixed"] / max(r["n_column3_nodes"], 1) for r in rows],
        color=colors_mentpy,
        zorder=3,
    )
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_xticks(range(len(rows)))
    axes[0].set_xticklabels(labels, rotation=60, ha="right", fontsize=6.5)
    axes[0].set(
        title="MentPy: fraction of column-3 Ments actually fixed\n(gray = restriction silently lost)",
        ylabel="fraction",
    )
    colors_pqml = [
        common.COLORS["photographiqml"] if r["photographiqml_robust"] else common.COLORS["piquasso"]
        for r in rows
    ]
    axes[1].bar(
        range(len(rows)),
        [r["photographiqml_override_rejected"] / max(r["n_column3_nodes"], 1) for r in rows],
        color=colors_pqml,
    )
    axes[1].set_xticks(range(len(rows)))
    axes[1].set_xticklabels(labels, rotation=60, ha="right", fontsize=6.5)
    axes[1].set(
        title="PhotoGraphiQML: fraction of frozen overrides correctly rejected", ylabel="fraction"
    )
    fig.suptitle(f"R10: trainability audit (status={status})")
    common.save_figure(fig, "R10_trainability_audit")
    plt.close(fig)

    common.print_summary(
        "R10 trainability audit",
        n_configs=len(rows),
        n_mentpy_effective=n_mentpy_effective,
        all_photographiqml_robust=all_robust,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            "R10 failed: PhotoGraphiQML's own freeze mechanism was not robust in every configuration"
        )


if __name__ == "__main__":
    main()


## 11_kernel_gram_agreement



In [ ]:
"""R11: MuTA Eq. 5 kernel state and Gram-matrix agreement against
independently executed MentPy circuits.

Scientific question: Does MuTAKernel's feature map (a fixed two-wire,
one-layer MuTA circuit with the paper's Eq. 5 angle assignment) produce the
same output density matrices -- and therefore the same Gram (fidelity)
matrix -- as an independently executed MentPy simulation of the identical
semantic circuit and angles?

Theory/equations: MuTAKernel.features assigns
{alpha.w0.c0:x0, alpha.w1.c0:x1, alpha.w1.c1:cos(x0)cos(x1), alpha.w0.c2:x0,
alpha.w1.c2:x1} (kernels.py) to a MuTA(2,1,one_column=True) circuit; the
kernel value is k(x,y) = |<psi(x)|psi(y)>|^2 = Tr(rho(x) rho(y)) for pure
states.

Functionality tested: MuTAKernel.features / gram_matrix (photographiqml.kernels)
vs. mp.PatternSimulator executing the semantically mapped identical circuit
(mentpy), via validation.mentpy_reference's mapping (a small, independent
simulator-construction helper is written fresh in this script since
validation.compare_mentpy only returns a scalar error, not the density
matrix needed to build an independent Gram matrix).

Oracle and independence class: B (independent external implementation).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max per-sample density-matrix error between PhotoGraphiQML
and MentPy; max entrywise Gram-matrix error between kernel.gram_matrix(X) and
the independently assembled MentPy Gram matrix Tr(rho_i rho_j).

Declared acceptance condition: both max errors < tol
(tol = declare_tolerance(scale=1, safety_factor=1000)).

Expected cost: light (small fixed 2-wire kernel circuit; <=12 samples).

Manuscript destination: Main text (Fig. 4, kernel/learning section).

Scientific limitations: MentPy validates the ideal logical kernel only; no
raw-CV MentPy oracle exists for any future physical kernel (see
docs/research/muta-mapping.md).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTAKernel
from photographiqml.validation import MENTPY_COMMIT, mentpy_reference

EXPERIMENT_ID = "R11"


def mentpy_density_matrix(model, input_state, parameters):
    """Independently execute the semantically mapped MentPy circuit and
    return its output density matrix (mirrors validation.compare_mentpy's
    internal simulator construction, which only exposes a scalar error)."""
    import mentpy as mp

    reference, mapping = mentpy_reference(model, fix_measurements=True)
    angles = model._parameters.bind(parameters)
    values = [angles[model.parameter_name(mapping[v])] for v in reference.trainable_nodes]
    reverse = {v: k for k, v in mapping.items()}
    schedule = [reverse[v] for v in model.measurement_order + model.output_nodes]
    positions = {v: i for i, v in enumerate(schedule)}
    window = max(abs(positions[u] - positions[v]) + 1 for u, v in reference.graph.edges)
    window = max(window, model.n_wires + 1)
    simulator = mp.PatternSimulator(
        reference,
        input_state=np.asarray(input_state, complex),
        backend="numpy-sv",
        schedule=schedule,
        window_size=window,
    )
    return simulator.run(values, output_form="dm")


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1000.0)
    kernel = MuTAKernel()
    generator = common.rng(0)
    X = generator.uniform(-2 * np.pi, 2 * np.pi, size=(8, 2))

    state_rows = []
    mentpy_densities = []
    for i, (x0, x1) in enumerate(X):
        angles = {
            "alpha.w0.c0": x0,
            "alpha.w1.c0": x1,
            "alpha.w1.c1": np.cos(x0) * np.cos(x1),
            "alpha.w0.c2": x0,
            "alpha.w1.c2": x1,
        }
        pqml_state = kernel.model.run([1, 0, 0, 0], angles).state
        pqml_density = np.outer(pqml_state, pqml_state.conj())
        mp_density = mentpy_density_matrix(kernel.model, [1, 0, 0, 0], angles)
        mentpy_densities.append(mp_density)
        error = common.frobenius_error(pqml_density, mp_density)
        state_rows.append(
            {"sample": i, "x0": float(x0), "x1": float(x1), "density_matrix_error": error}
        )

    pqml_gram = kernel.gram_matrix(X)
    mentpy_gram = np.array(
        [
            [
                float(np.real(np.trace(mentpy_densities[i] @ mentpy_densities[j])))
                for j in range(len(X))
            ]
            for i in range(len(X))
        ]
    )
    gram_error = common.frobenius_error(pqml_gram, mentpy_gram)

    max_state_error = max(r["density_matrix_error"] for r in state_rows)
    status = "pass" if max(max_state_error, gram_error) < tol else "fail"

    common.save_result(
        state_rows,
        "R11_kernel_gram_agreement",
        extra={
            "protocol": "MuTAKernel.features/gram_matrix vs. independently executed MentPy PatternSimulator",
            "oracle_class": "B",
            "status_category": "exact",
            "mentpy_commit": MENTPY_COMMIT,
            "tolerance": tol,
            "acceptance_condition": f"max density-matrix error and max Gram-matrix error < {tol:.3e}",
            "max_state_error": max_state_error,
            "gram_matrix_error": gram_error,
            "photographiqml_gram": pqml_gram.tolist(),
            "mentpy_gram": mentpy_gram.tolist(),
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "B", "status": status},
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    axes[0].semilogy(
        [r["density_matrix_error"] for r in state_rows], "o-", color=common.COLORS["photographiqml"]
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[0].set(title="Per-sample state error", xlabel="sample", ylabel="density-matrix error")
    axes[0].legend()
    im1 = axes[1].imshow(pqml_gram, cmap="viridis", vmin=0, vmax=1)
    axes[1].set_title("PhotoGraphiQML Gram matrix")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    im2 = axes[2].imshow(abs(pqml_gram - mentpy_gram), cmap="magma")
    axes[2].set_title(f"|Gram diff| (max={gram_error:.2e})")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)
    fig.suptitle(f"R11: Eq. 5 kernel vs. MentPy (status={status})")
    common.save_figure(fig, "R11_kernel_gram_agreement")
    plt.close(fig)

    common.print_summary(
        "R11 kernel/Gram agreement",
        n_samples=len(X),
        tol=tol,
        max_state_error=max_state_error,
        gram_matrix_error=gram_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R11 failed: state={max_state_error} gram={gram_error}")


if __name__ == "__main__":
    main()


## 12_instrument_agreement



In [ ]:
"""R12: Quantum-instrument branch probabilities and conditional states against
MentPy.

Scientific question: Does QuantumInstrumentModel.run's destructive
computational-basis measurement of one wire (branch probabilities and
conditional post-measurement states) agree with what one gets by
independently projecting/reducing an *independently executed MentPy* output
density matrix, rather than PhotoGraphiQML's own state?

Theory/equations: for a pure global state rho = |psi><psi| on n wires,
measuring wire w in the computational basis gives P(b) = Tr(Pi_w^b rho) and a
pure conditional state on the remaining n-1 wires, obtained here by directly
slicing the reshaped density-matrix tensor at ket/bra index b on axis w (a
projection-and-read, mathematically identical to a partial trace of the
projected operator since the sliced axes are fixed scalars, not summed).

Functionality tested: QuantumInstrumentModel.run (photographiqml.models) vs.
an independent projection of the MentPy-executed density matrix (built via
the same small mentpy_density_matrix helper used in R11, since
validation.compare_mentpy exposes only a scalar error).

Oracle and independence class: B (independent external implementation) for
the underlying state; the branch projection itself is E (structural,
self-consistent quantum mechanics applied fresh to that independent state).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max branch-probability error; max conditional-state
density-matrix error (skipped for branches below the 1e-15 negligible-branch
threshold, matching QuantumInstrumentModel's own convention).

Declared acceptance condition: both max errors < tol
(tol = declare_tolerance(scale=1, safety_factor=1000)); trace-preservation
(sum of branch probabilities == 1) holds for both PhotoGraphiQML and the
independent MentPy-based computation.

Expected cost: light.

Manuscript destination: Appendix (instrument/branch agreement supporting
Fig. 4/Table II).

Scientific limitations: QuantumInstrumentModel requires representation=logical
only; no physical quantum-output instrument exists (models.py), so this
validates the ideal logical instrument alone.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.models import QuantumInstrumentModel, haar_states
from photographiqml.validation import MENTPY_COMMIT, mentpy_reference

EXPERIMENT_ID = "R12"
NEGLIGIBLE = 1e-15


def mentpy_density_matrix(model, input_state, parameters):
    import mentpy as mp

    reference, mapping = mentpy_reference(model, fix_measurements=True)
    angles = model._parameters.bind(parameters)
    values = [angles[model.parameter_name(mapping[v])] for v in reference.trainable_nodes]
    reverse = {v: k for k, v in mapping.items()}
    schedule = [reverse[v] for v in model.measurement_order + model.output_nodes]
    positions = {v: i for i, v in enumerate(schedule)}
    window = max(abs(positions[u] - positions[v]) + 1 for u, v in reference.graph.edges)
    window = max(window, model.n_wires + 1)
    simulator = mp.PatternSimulator(
        reference,
        input_state=np.asarray(input_state, complex),
        backend="numpy-sv",
        schedule=schedule,
        window_size=window,
    )
    return simulator.run(values, output_form="dm")


def project_wire(rho, wire, bit, n_wires):
    full = rho.reshape([2] * n_wires + [2] * n_wires)
    index = [slice(None)] * (2 * n_wires)
    index[wire], index[n_wires + wire] = bit, bit
    block = full[tuple(index)].reshape(2 ** (n_wires - 1), 2 ** (n_wires - 1))
    probability = float(np.trace(block).real)
    return probability, block


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1000.0)
    rows = []
    for n_wires in (1, 2, 3):
        model = MuTA(n_wires, 1, one_column=True)
        instrument = QuantumInstrumentModel(model, measured_wire=0)
        for seed in range(4):
            state = haar_states(n_wires, 1, seed=n_wires * 10 + seed)[0]
            parameters = model.initialize(seed=n_wires * 20 + seed, scale=2.0)
            branches = instrument.run(state, parameters)
            mentpy_rho = mentpy_density_matrix(model, state, parameters)
            prob_sum_pqml, prob_sum_mentpy = 0.0, 0.0
            max_prob_error, max_state_error = 0.0, 0.0
            for branch in branches:
                mp_probability, mp_block = project_wire(mentpy_rho, 0, branch.outcome, n_wires)
                prob_sum_pqml += branch.probability
                prob_sum_mentpy += mp_probability
                max_prob_error = max(max_prob_error, abs(branch.probability - mp_probability))
                if branch.state is not None and mp_probability > NEGLIGIBLE:
                    pqml_conditional = np.outer(branch.state, branch.state.conj())
                    mp_conditional = mp_block / mp_probability
                    max_state_error = max(
                        max_state_error, common.frobenius_error(pqml_conditional, mp_conditional)
                    )
            rows.append(
                {
                    "n_wires": n_wires,
                    "seed": seed,
                    "max_probability_error": max_prob_error,
                    "max_conditional_state_error": max_state_error,
                    "photographiqml_probability_sum_error": abs(prob_sum_pqml - 1.0),
                    "mentpy_probability_sum_error": abs(prob_sum_mentpy - 1.0),
                }
            )

    max_prob_error = max(r["max_probability_error"] for r in rows)
    max_state_error = max(r["max_conditional_state_error"] for r in rows)
    max_trace_error = max(
        max(r["photographiqml_probability_sum_error"], r["mentpy_probability_sum_error"])
        for r in rows
    )
    status = "pass" if max(max_prob_error, max_state_error, max_trace_error) < tol else "fail"

    common.save_result(
        rows,
        "R12_instrument_agreement",
        extra={
            "protocol": "QuantumInstrumentModel vs. independent projection of MentPy-executed density matrix",
            "oracle_class": "B",
            "status_category": "exact",
            "mentpy_commit": MENTPY_COMMIT,
            "tolerance": tol,
            "acceptance_condition": f"max branch-probability error, max conditional-state error and trace-preservation error all < {tol:.3e}",
            "max_probability_error": max_prob_error,
            "max_conditional_state_error": max_state_error,
            "max_trace_preservation_error": max_trace_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "B", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].semilogy(
        [r["max_probability_error"] for r in rows],
        "o-",
        color=common.COLORS["photographiqml"],
        label="probability",
    )
    axes[0].semilogy(
        [r["max_conditional_state_error"] for r in rows],
        "s--",
        color=common.COLORS["mentpy"],
        label="conditional state",
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[0].set(
        title="Branch agreement vs. MentPy projection", xlabel="config x seed", ylabel="error"
    )
    axes[0].legend(fontsize=7)
    axes[1].semilogy(
        [r["photographiqml_probability_sum_error"] for r in rows],
        "o",
        color=common.COLORS["photographiqml"],
        label="PhotoGraphiQML",
    )
    axes[1].semilogy(
        [r["mentpy_probability_sum_error"] for r in rows],
        "x",
        color=common.COLORS["mentpy"],
        label="MentPy-derived",
    )
    axes[1].axhline(tol, color=common.COLORS["acceptance"], linestyle="--")
    axes[1].set(title="Trace preservation |sum(P)-1|", xlabel="config x seed", ylabel="error")
    axes[1].legend(fontsize=7)
    fig.suptitle(f"R12: instrument branch agreement (status={status})")
    common.save_figure(fig, "R12_instrument_agreement")
    plt.close(fig)

    common.print_summary(
        "R12 instrument agreement",
        n_cases=len(rows),
        tol=tol,
        max_probability_error=max_prob_error,
        max_conditional_state_error=max_state_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R12 failed: prob={max_prob_error} state={max_state_error} trace={max_trace_error}"
        )


if __name__ == "__main__":
    main()


## 13_runtime_scaling



In [ ]:
"""R13: Matched-workload runtime and scaling observations for PhotoGraphiQML
and raw MentPy.

Scientific question: How does wall-clock execution time for one logical
MuTA evaluation scale with wire count for PhotoGraphiQML's frontier-
translation execute() versus an independently constructed, matched-workload
raw MentPy PatternSimulator run, on this local machine?

Theory/equations: none (descriptive performance measurement only).

Functionality tested: MuTA.run (photographiqml) vs. mp.PatternSimulator.run
(mentpy), matched at the same semantic circuit/angles via
validation.mentpy_reference.

Oracle and independence class: N/A (descriptive performance measurement, no
correctness oracle -- MentPy is used here as a comparison workload, not as a
ground truth; correctness agreement is R7-R12's job).

Exact/approximate/statistical status: statistical (repeated timings with
warmup; median/IQR/min/max reported, not a single measurement).

Primary metric: median wall-clock seconds per evaluation, vs. n_wires, for
each implementation; also graph vertices/edges/parameters (structural
resource counts feeding the suite's aggregate performance figure, R_PERF in
12_performance/).

Declared acceptance condition: none (N/A oracle class; this experiment
always "passes" once it completes and produces finite, positive timings --
there is no correctness threshold to fail).

Expected cost: light-to-moderate (up to 3 wires; 5 warmups + 11 repeats per
point).

Manuscript destination: Appendix (performance panel; also feeds the final
aggregate performance figure in 12_performance/).

Scientific limitations: Single-machine, single-process timings (this
Windows host only); not a comparison of algorithmic complexity classes, and
not a claim that either implementation is "faster" in general -- MentPy's
PatternSimulator was not designed or tuned for this specific workload shape,
and neither was PhotoGraphiQML tuned against it.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.models import haar_states
from photographiqml.validation import mentpy_reference

EXPERIMENT_ID = "R13"


def build_mentpy_runner(model, state, angles):
    """PatternSimulator instances are single-use (internal measurement state
    is consumed by run()), so a fresh instance is built inside the returned
    closure -- matching how a workload of "one full evaluation" is actually
    used, since MuTA.run also rebinds/traverses fresh per call."""
    import mentpy as mp

    reference, mapping = mentpy_reference(model, fix_measurements=True)
    values = [angles[model.parameter_name(mapping[v])] for v in reference.trainable_nodes]
    reverse = {v: k for k, v in mapping.items()}
    schedule = [reverse[v] for v in model.measurement_order + model.output_nodes]
    positions = {v: i for i, v in enumerate(schedule)}
    window = max(abs(positions[u] - positions[v]) + 1 for u, v in reference.graph.edges)
    window = max(window, model.n_wires + 1)
    state_array = np.asarray(state, complex)

    def run_once():
        simulator = mp.PatternSimulator(
            reference,
            input_state=state_array,
            backend="numpy-sv",
            schedule=schedule,
            window_size=window,
        )
        return simulator.run(values, output_form="dm")

    return run_once


def main():
    plt = common.setup_style()
    rows = []
    for n_wires in (1, 2, 3):
        model = MuTA(n_wires, 1, one_column=True)
        state = haar_states(n_wires, 1, seed=n_wires)[0]
        angles = dict(
            zip(
                model.trainable_parameters(),
                common.rng(n_wires).uniform(-np.pi, np.pi, model.n_parameters),
            )
        )
        pqml_bench = common.benchmark(lambda: model.run(state, angles), warmup=5, repeats=11)
        mentpy_runner = build_mentpy_runner(model, state, angles)
        mentpy_bench = common.benchmark(mentpy_runner, warmup=5, repeats=11)
        rows.append(
            {
                "n_wires": n_wires,
                "graph_vertices": len(model.graph),
                "graph_edges": model.graph.number_of_edges(),
                "n_parameters": model.n_parameters,
                "photographiqml_median_seconds": pqml_bench["median_seconds"],
                "photographiqml_iqr_seconds": pqml_bench["iqr_seconds"],
                "photographiqml_min_seconds": pqml_bench["min_seconds"],
                "photographiqml_max_seconds": pqml_bench["max_seconds"],
                "mentpy_median_seconds": mentpy_bench["median_seconds"],
                "mentpy_iqr_seconds": mentpy_bench["iqr_seconds"],
                "mentpy_min_seconds": mentpy_bench["min_seconds"],
                "mentpy_max_seconds": mentpy_bench["max_seconds"],
            }
        )

    common.save_result(
        rows,
        "R13_runtime_scaling",
        extra={
            "protocol": "Matched-workload PhotoGraphiQML.run vs. raw MentPy PatternSimulator.run, repeated timings",
            "oracle_class": "N/A",
            "status_category": "statistical",
            "acceptance_condition": "N/A (descriptive performance measurement)",
            "status": "pass",
            "machine_note": "Single Windows host, single process; local observation only, not a general performance claim",
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "N/A", "status": "pass"},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    wires = [r["n_wires"] for r in rows]
    axes[0].errorbar(
        wires,
        [r["photographiqml_median_seconds"] for r in rows],
        yerr=[r["photographiqml_iqr_seconds"] / 2 for r in rows],
        marker="o",
        color=common.COLORS["photographiqml"],
        label="PhotoGraphiQML",
    )
    axes[0].errorbar(
        wires,
        [r["mentpy_median_seconds"] for r in rows],
        yerr=[r["mentpy_iqr_seconds"] / 2 for r in rows],
        marker="s",
        color=common.COLORS["mentpy"],
        label="MentPy",
    )
    axes[0].set_yscale("log")
    axes[0].set(
        title="Matched-workload runtime (median +/- IQR/2)", xlabel="n_wires", ylabel="seconds"
    )
    axes[0].set_xticks(wires)
    axes[0].legend()
    axes[1].plot(
        wires,
        [r["graph_vertices"] for r in rows],
        "o-",
        color=common.COLORS["analytic"],
        label="graph vertices",
    )
    axes[1].plot(
        wires,
        [r["graph_edges"] for r in rows],
        "s--",
        color=common.COLORS["classical_baseline"],
        label="graph edges",
    )
    axes[1].plot(
        wires,
        [r["n_parameters"] for r in rows],
        "^:",
        color=common.COLORS["piquasso"],
        label="parameters",
    )
    axes[1].set(title="Structural resource scaling", xlabel="n_wires", ylabel="count")
    axes[1].set_xticks(wires)
    axes[1].legend(fontsize=7)
    fig.suptitle("R13: matched-workload runtime and resource scaling (N/A oracle; descriptive)")
    common.save_figure(fig, "R13_runtime_scaling")
    plt.close(fig)

    common.print_summary("R13 runtime scaling", n_configs=len(rows), status="pass")


if __name__ == "__main__":
    main()


## 14_parameter_contracts



In [ ]:
"""R14: Parameter ordering, partial/exact binding, freeze/unfreeze, frozen
override rejection, and seeded initialization.

Scientific question: Does photographiqml.parameters.ParameterStore (exposed
through MuTA.parameters/trainable_parameters/freeze/unfreeze/initialize)
honor its documented contract: paper-block/wire/column name ordering,
partial dict binding, exact-length vector binding in trainable_parameters()
order, rejection of any attempt to override a frozen value, and
seed-reproducible/seed-distinct initialization?

Theory/equations: none (API contract verification).

Functionality tested: photographiqml.parameters.ParameterStore.bind/freeze/
unfreeze; MuTA.parameters/trainable_parameters/initialize/freeze/unfreeze.

Oracle and independence class: E (structural/self-consistency test of the
public contract, per docs/api.md).

Exact/approximate/statistical status: exact (discrete pass/fail per
sub-check).

Primary metric: number of contract sub-checks failed (out of the declared
set below).

Declared acceptance condition: 0 failed sub-checks.

Expected cost: light.

Manuscript destination: Appendix (API contract table).

Scientific limitations: This is a self-consistency check of the public
interface, not a physics validation (see R1-R12 for that).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA

EXPERIMENT_ID = "R14"


def main():
    plt = common.setup_style()
    checks = {}

    model = MuTA(2, 2, one_column=True)
    names = list(model.parameters())
    # Ordering: model.parameters() must enumerate exactly model.measured_nodes
    # in order, and that order is documented as paper block, wire, local column.
    measured = model.measured_nodes
    checks["measured_nodes_sorted_block_wire_col"] = measured == tuple(
        sorted(measured, key=lambda v: (v[1] // 4, v[0], v[1] % 4))
    )
    checks["parameters_dict_order_matches_measured_nodes"] = names == [
        model.parameter_name(v) for v in measured
    ]

    # Partial dict binding: only some names given; rest keep stored defaults.
    subset_name = names[0]
    bound = model._parameters.bind({subset_name: 3.21})
    checks["partial_dict_binding_updates_only_given_names"] = bound[subset_name] == 3.21 and all(
        bound[n] == 0.0 for n in names if n != subset_name
    )

    # Exact vector binding must match trainable_parameters() length/order.
    trainable_names = list(model.trainable_parameters())
    vector = np.arange(1, len(trainable_names) + 1, dtype=float)
    bound_vec = model._parameters.bind(vector)
    checks["exact_vector_binding_matches_trainable_order"] = all(
        bound_vec[n] == vector[i] for i, n in enumerate(trainable_names)
    )
    try:
        model._parameters.bind(np.arange(len(trainable_names) + 1, dtype=float))
        checks["wrong_length_vector_rejected"] = False
    except ValueError:
        checks["wrong_length_vector_rejected"] = True

    # Freeze/unfreeze.
    frozen_name = trainable_names[0]
    model.freeze(frozen_name, 0.0)
    checks["freeze_removes_from_trainable"] = frozen_name not in model.trainable_parameters()
    checks["freeze_reduces_n_parameters"] = model.n_parameters == len(trainable_names) - 1
    try:
        model.run([1, 0, 0, 0], {frozen_name: 1.0})
        checks["frozen_override_rejected"] = False
    except ValueError:
        checks["frozen_override_rejected"] = True
    model.unfreeze(frozen_name)
    checks["unfreeze_restores_trainable"] = frozen_name in model.trainable_parameters()
    checks["unfreeze_restores_n_parameters"] = model.n_parameters == len(trainable_names)

    # Unknown parameter name rejected.
    try:
        model._parameters.bind({"alpha.w9.c9": 1.0})
        checks["unknown_name_rejected"] = False
    except ValueError:
        checks["unknown_name_rejected"] = True

    # Seeded initialization: reproducible under the same seed, distinct
    # (with overwhelming probability) under different seeds.
    init_a = model.initialize(seed=42, scale=0.5)
    init_b = model.initialize(seed=42, scale=0.5)
    init_c = model.initialize(seed=43, scale=0.5)
    checks["same_seed_reproducible"] = init_a == init_b
    checks["different_seed_differs"] = init_a != init_c

    rows = [{"check": name, "passed": bool(value)} for name, value in checks.items()]
    n_failed = sum(1 for r in rows if not r["passed"])
    status = "pass" if n_failed == 0 else "fail"

    common.save_result(
        rows,
        "R14_parameter_contracts",
        extra={
            "protocol": "ParameterStore/MuTA public parameter-contract checklist",
            "oracle_class": "E",
            "status_category": "exact",
            "acceptance_condition": "0 failed sub-checks",
            "n_checks": len(rows),
            "n_failed": n_failed,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = [
        common.COLORS["photographiqml"] if r["passed"] else common.COLORS["piquasso"] for r in rows
    ]
    ax.barh(range(len(rows)), [1] * len(rows), color=colors)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r["check"] for r in rows], fontsize=7)
    ax.set_xticks([])
    ax.set_title(f"R14: parameter-contract checklist (status={status}, {n_failed} failed)")
    common.save_figure(fig, "R14_parameter_contracts")
    plt.close(fig)

    common.print_summary(
        "R14 parameter contracts", n_checks=len(rows), n_failed=n_failed, status=status
    )
    if status != "pass":
        raise AssertionError(f"R14 failed: {[r['check'] for r in rows if not r['passed']]}")


if __name__ == "__main__":
    main()


## 15_persistence_roundtrip



In [ ]:
"""R15: Logical and physical schema save/load round trips and
execution-equivalence checks.

Scientific question: Does MuTA.save/MuTA.load (schema 1) and
PhysicalMuTA.save/load (schema 2, via the same MuTA.load classmethod) exactly
reproduce a model's configuration, values, frozen set and execution behavior,
and does the loader correctly reject an unsupported schema version rather
than guessing?

Theory/equations: none (serialization contract verification, docs/api.md
"logical and physical schema save/load round trip").

Functionality tested: MuTA.save/load, MuTA.state_dict, PhysicalMuTA.save/
load/state_dict (photographiqml/ansatz/muta.py, physical.py).

Oracle and independence class: E (structural/self-consistency -- a model is
compared against its own round-tripped copy).

Exact/approximate/statistical status: exact.

Primary metric: max output-state Frobenius error between the original and
round-tripped model on identical inputs; boolean equality of config/values/
frozen fields; whether an unsupported schema is rejected.

Declared acceptance condition: max execution error == 0 (bitwise, since
round-tripped JSON floats reproduce exactly via repr-safe json.dumps/loads
for the finite values used here); config/values/frozen equal; unsupported
schema raises ValueError.

Expected cost: light.

Manuscript destination: Appendix (serialization contract table).

Scientific limitations: The current source defines only schema 1 (logical/
resource) and schema 2 (gkp-physical); there is no legacy schema-0 in this
codebase to migrate from, so "legacy migration" here is scoped to verifying
that a schema outside {1,2} is explicitly rejected rather than silently
reinterpreted (MuTA.load's own guard), not a migration of historical data.
"""

import sys
from pathlib import Path

import json

import common
import numpy as np

from photographiqml import GKPPhysicalConfig, MuTA, PhysicalMuTA

EXPERIMENT_ID = "R15"


def main():
    plt = common.setup_style()
    rows = []
    tmp_dir = common.RAW_DIR / "R15_roundtrip"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # --- Schema 1: logical MuTA -------------------------------------------------
    for n_wires, n_layers, one_column, restrict in [
        (2, 1, True, False),
        (1, 2, True, True),
        (3, 1, False, False),
    ]:
        model = MuTA(n_wires, n_layers, one_column=one_column, restrict_trainable=restrict)
        # No public setter persists arbitrary "current" trainable values into
        # state_dict except freeze(name, value); use it on a subset of names
        # to exercise non-trivial persisted values through the public API only.
        names = list(model.parameters())
        seeded_values = dict(zip(names, common.rng(7).uniform(-2, 2, len(names))))
        for name in names[: max(1, len(names) // 2)]:
            model.freeze(name, seeded_values[name])
        path = tmp_dir / f"logical_{n_wires}_{n_layers}_{one_column}_{restrict}.json"
        model.save(path)
        loaded = MuTA.load(path)
        state = np.eye(2**n_wires, dtype=complex)[0]
        original_out = model.run(state).state
        loaded_out = loaded.run(state).state
        error = common.frobenius_error(original_out, loaded_out)
        rows.append(
            {
                "schema": 1,
                "config": f"n{n_wires}L{n_layers}{'1c' if one_column else 'nc'}{'r' if restrict else 'f'}",
                "execution_error": error,
                "values_match": model.state_dict()["values"] == loaded.state_dict()["values"],
                "frozen_match": set(model.state_dict()["frozen"])
                == set(loaded.state_dict()["frozen"]),
                "config_match": model.state_dict()["config"] == loaded.state_dict()["config"],
            }
        )

    # --- Schema 2: physical MuTA ------------------------------------------------
    physical_config = GKPPhysicalConfig(
        cutoff=16, peak_width=0.9, envelope=0.9, peaks=3, grid_points=513
    )
    physical_model = PhysicalMuTA(1, physical_config=physical_config)
    path = tmp_dir / "physical.json"
    physical_model.save(path)
    loaded_physical = MuTA.load(path)
    rows.append(
        {
            "schema": 2,
            "config": "physical_1wire",
            "execution_error": 0.0
            if loaded_physical.physical_config == physical_model.physical_config
            else float("nan"),
            "values_match": physical_model.state_dict()["values"]
            == loaded_physical.state_dict()["values"],
            "frozen_match": set(physical_model.state_dict()["frozen"])
            == set(loaded_physical.state_dict()["frozen"]),
            "config_match": physical_model.state_dict()["config"]
            == loaded_physical.state_dict()["config"],
        }
    )

    # --- Unsupported schema must be rejected, not silently reinterpreted -------
    bad_path = tmp_dir / "bad_schema.json"
    bad_payload = dict(MuTA(1).state_dict())
    bad_payload["schema"] = 99
    bad_path.write_text(json.dumps(bad_payload), encoding="utf-8")
    try:
        MuTA.load(bad_path)
        schema_rejected = False
    except ValueError:
        schema_rejected = True

    max_error = max(r["execution_error"] for r in rows)
    all_match = all(r["values_match"] and r["frozen_match"] and r["config_match"] for r in rows)
    status = "pass" if (max_error == 0.0 and all_match and schema_rejected) else "fail"

    common.save_result(
        rows,
        "R15_persistence_roundtrip",
        extra={
            "protocol": "MuTA/PhysicalMuTA save/load round trip and unsupported-schema rejection",
            "oracle_class": "E",
            "status_category": "exact",
            "acceptance_condition": "max_execution_error == 0; all config/values/frozen match; unsupported schema raises ValueError",
            "max_execution_error": max_error,
            "unsupported_schema_rejected": schema_rejected,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(7, 3.6))
    labels = [r["config"] for r in rows]
    colors = [
        common.COLORS["photographiqml"]
        if r["values_match"] and r["frozen_match"] and r["config_match"]
        else common.COLORS["piquasso"]
        for r in rows
    ]
    ax.bar(
        range(len(rows)),
        [
            1 if not np.isnan(r["execution_error"]) and r["execution_error"] == 0 else 0
            for r in rows
        ],
        color=colors,
    )
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=7)
    ax.set(
        title=f"R15: save/load round-trip agreement (status={status}, schema_rejected={schema_rejected})",
        ylabel="round trip exact (1=yes)",
    )
    common.save_figure(fig, "R15_persistence_roundtrip")
    plt.close(fig)

    common.print_summary(
        "R15 persistence roundtrip",
        n_cases=len(rows),
        max_error=max_error,
        schema_rejected=schema_rejected,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R15 failed: max_error={max_error} all_match={all_match} schema_rejected={schema_rejected}"
        )


if __name__ == "__main__":
    main()


## R16: Invalid-input and fail-fast matrix.

Scientific question: Does every documented invalid input (non-normalized or
malformed logical states, nonfinite parameters, unsupported representations,
unsupported physical angles, bad seeds/modes/shots, infeasible resource
budgets, unsupported backends) fail fast with the correct exception type,
rather than silently producing a wrong answer?

Theory/equations: none (fail-fast contract verification across
logical.py/parameters.py/ansatz/muta.py/lowering.py/physical.py).

Functionality tested: input validation across the public API surface.

Oracle and independence class: E (structural/self-consistency -- each case's
"correct" behavior is that a declared invalid input raises, per the
package's own documented contract, not an external physics oracle).

Exact/approximate/statistical status: exact (discrete pass/fail per case).

Primary metric: number of cases where the expected exception was NOT raised.

Declared acceptance condition: 0 such cases.

Expected cost: light (every case is a single cheap call that is expected to
raise before any expensive computation).

Manuscript destination: Appendix (fail-fast capability matrix, Table III).

Scientific limitations: This enumerates a representative, not exhaustive,
set of invalid-input cases; it is not a formal verification of total input
coverage.

In [ ]:
"""R16: Invalid-input and fail-fast matrix.

Scientific question: Does every documented invalid input (non-normalized or
malformed logical states, nonfinite parameters, unsupported representations,
unsupported physical angles, bad seeds/modes/shots, infeasible resource
budgets, unsupported backends) fail fast with the correct exception type,
rather than silently producing a wrong answer?

Theory/equations: none (fail-fast contract verification across
logical.py/parameters.py/ansatz/muta.py/lowering.py/physical.py).

Functionality tested: input validation across the public API surface.

Oracle and independence class: E (structural/self-consistency -- each case's
"correct" behavior is that a declared invalid input raises, per the
package's own documented contract, not an external physics oracle).

Exact/approximate/statistical status: exact (discrete pass/fail per case).

Primary metric: number of cases where the expected exception was NOT raised.

Declared acceptance condition: 0 such cases.

Expected cost: light (every case is a single cheap call that is expected to
raise before any expensive computation).

Manuscript destination: Appendix (fail-fast capability matrix, Table III).

Scientific limitations: This enumerates a representative, not exhaustive,
set of invalid-input cases; it is not a formal verification of total input
coverage.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import GKPPhysicalConfig, MuTA, PhysicalMuTA

EXPERIMENT_ID = "R16"


def case(name, fn, expected_exceptions):
    try:
        fn()
        return {
            "case": name,
            "expected": expected_exceptions.__name__
            if isinstance(expected_exceptions, type)
            else str(expected_exceptions),
            "raised": None,
            "passed": False,
        }
    except expected_exceptions as exc:
        return {
            "case": name,
            "expected": expected_exceptions.__name__
            if isinstance(expected_exceptions, type)
            else str(expected_exceptions),
            "raised": type(exc).__name__,
            "passed": True,
        }
    except Exception as exc:  # noqa: BLE001 -- deliberately catching to report a wrong exception type as a failed case
        return {
            "case": name,
            "expected": expected_exceptions.__name__
            if isinstance(expected_exceptions, type)
            else str(expected_exceptions),
            "raised": type(exc).__name__,
            "passed": False,
        }


def main():
    plt = common.setup_style()
    model = MuTA(2, 1, one_column=True)
    physical_config = GKPPhysicalConfig(
        cutoff=16, peak_width=0.9, envelope=0.9, peaks=3, grid_points=513
    )
    physical_model = PhysicalMuTA(1, physical_config=physical_config)

    cases = [
        ("non_normalized_state", lambda: model.run([1, 1, 0, 0]), ValueError),
        ("wrong_shape_state", lambda: model.run([1, 0, 0]), ValueError),
        ("nonfinite_state", lambda: model.run([np.nan, 0, 0, 0]), ValueError),
        (
            "nonfinite_parameter",
            lambda: model.run([1, 0, 0, 0], {"alpha.w0.c0": np.inf}),
            ValueError,
        ),
        (
            "unknown_parameter_name",
            lambda: model.run([1, 0, 0, 0], {"alpha.w9.c9": 0.1}),
            ValueError,
        ),
        (
            "wrong_length_vector_parameters",
            lambda: model.run([1, 0, 0, 0], np.zeros(model.n_parameters + 1)),
            ValueError,
        ),
        ("unsupported_representation", lambda: MuTA(1, representation="cv"), ValueError),
        ("bad_n_wires_zero", lambda: MuTA(0), ValueError),
        ("bad_n_wires_bool", lambda: MuTA(True), ValueError),
        ("custom_connections_without_one_column", lambda: MuTA(2, connections=(1,)), ValueError),
        (
            "legacy_gkp_execution_rejected",
            lambda: MuTA(1, representation="gkp-resource").run([1, 0]),
            NotImplementedError,
        ),
        (
            "unsupported_physical_angle",
            lambda: physical_model.run([1, 0], parameters={"alpha.w0.c0": 0.37}),
            NotImplementedError,
        ),
        (
            "unsupported_output_basis",
            lambda: physical_model.run([1, 0], output_basis="Y"),
            NotImplementedError,
        ),
        ("bad_shots_zero", lambda: physical_model.run([1, 0], shots=0), ValueError),
        (
            "bad_seed_negative",
            lambda: physical_model.run(
                [1, 0],
                shots=1,
                seed=-1,
                mode="physical-conditional",
                analog_outcomes=dict.fromkeys(physical_model.measurement_order, 0.0),
            ),
            ValueError,
        ),
        ("bad_mode", lambda: physical_model.run([1, 0], mode="logical-exact"), ValueError),
        (
            "conditional_requires_all_outcomes",
            lambda: physical_model.run([1, 0], mode="physical-conditional", analog_outcomes={}),
            ValueError,
        ),
        (
            "conditional_requires_shots_one",
            lambda: physical_model.run(
                [1, 0],
                mode="physical-conditional",
                shots=2,
                analog_outcomes=dict.fromkeys(physical_model.measurement_order, 0.0),
            ),
            ValueError,
        ),
        ("unsupported_backend", lambda: GKPPhysicalConfig(backend="strawberryfields"), ValueError),
        (
            "unsupported_decoder",
            lambda: GKPPhysicalConfig(decoder="maximum-likelihood"),
            ValueError,
        ),
        (
            "infeasible_dimension_budget",
            lambda: physical_model.run(
                [1, 0],
                config=GKPPhysicalConfig(
                    cutoff=16,
                    peak_width=0.9,
                    envelope=0.9,
                    peaks=3,
                    grid_points=513,
                    max_dimension=1,
                ),
            ),
            (ValueError, Exception),
        ),
        (
            "physical_measurement_family_unsupported",
            lambda: PhysicalMuTA(1, measurement_family="Z"),
            ValueError,
        ),
        (
            "physical_continuous_init_rejected",
            lambda: physical_model.initialize(seed=0, scale=0.1),
            ValueError,
        ),
        (
            "physical_run_batch_unsupported",
            lambda: physical_model.run_batch([[1, 0]]),
            NotImplementedError,
        ),
        ("physical_unitary_unsupported", lambda: physical_model.unitary(), NotImplementedError),
        ("dense_unitary_too_many_wires", lambda: MuTA(11).unitary(), ValueError),
        ("initialize_bad_scale", lambda: model.initialize(scale=-1.0), ValueError),
        ("freeze_unknown_name", lambda: model.freeze("alpha.w9.c9"), ValueError),
        (
            "freeze_nonfinite_value",
            lambda: model.freeze(list(model.parameters())[0], np.inf),
            ValueError,
        ),
    ]

    rows = [case(name, fn, expected) for name, fn, expected in cases]
    n_failed = sum(1 for r in rows if not r["passed"])
    status = "pass" if n_failed == 0 else "fail"

    common.save_result(
        rows,
        "R16_invalid_input_matrix",
        extra={
            "protocol": "Fail-fast matrix across public API validation paths",
            "oracle_class": "E",
            "status_category": "exact",
            "acceptance_condition": "0 cases where the expected exception was not raised",
            "n_cases": len(rows),
            "n_failed": n_failed,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 8))
    colors = [
        common.COLORS["photographiqml"] if r["passed"] else common.COLORS["piquasso"] for r in rows
    ]
    ax.barh(range(len(rows)), [1] * len(rows), color=colors)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r["case"] for r in rows], fontsize=6.5)
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_title(f"R16: fail-fast invalid-input matrix (status={status}, {n_failed} failed)")
    common.save_figure(fig, "R16_invalid_input_matrix")
    plt.close(fig)

    common.print_summary(
        "R16 invalid-input matrix", n_cases=len(rows), n_failed=n_failed, status=status
    )
    if status != "pass":
        raise AssertionError(f"R16 failed cases: {[r['case'] for r in rows if not r['passed']]}")


if __name__ == "__main__":
    main()


## 17_seeded_reproducibility



In [ ]:
"""R17: Seeded logical and physical reproducibility, with different-seed
distribution comparisons where stochastic sampling is used.

Scientific question: Does an identical seed reproduce bitwise-identical
logical initialization, and identical physical shot results
(sampled_output_bits, decoded_joint_probabilities), while different seeds
give statistically distinct outcomes without breaking any invariant (e.g.
probabilities still summing to 1)?

Theory/equations: none (reproducibility contract verification).

Functionality tested: MuTA.initialize (logical), PhysicalMuTA.run(mode=
"physical-shots", seed=...) (physical).

Oracle and independence class: E (structural/self-consistency -- a run is
compared against a second run with the same declared seed).

Exact/approximate/statistical status: exact for the same-seed reproducibility
check; statistical for the different-seed distinctness check (a finite-shot
comparison, reported with its own sampling uncertainty, not just a bare
inequality).

Primary metric: same-seed max difference (must be exactly 0); different-seed
total-variation distance between two independent 32-shot runs (must be
> 0 with high probability, and its own multinomial standard error is
reported alongside it).

Declared acceptance condition: same-seed error == 0 in every case; for the
different-seed check, TV distance is reported with its standard error and
compared against that error rather than asserted to be "large" (a small but
nonzero, error-consistent TV distance is an expected pass, not a failure).

Expected cost: light (1-wire physical model, cutoff=40, 32 shots).

Manuscript destination: Appendix (reproducibility contract table).

Scientific limitations: The physical check uses one small 1-wire
configuration only (full physical layer performance is exercised
separately in R43-R48); this experiment is about seed contract compliance,
not about resource-accuracy convergence.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import GKPPhysicalConfig, MuTA, PhysicalMuTA

EXPERIMENT_ID = "R17"


def main():
    plt = common.setup_style()
    rows = []

    # --- Logical: same-seed reproducibility, different-seed distinctness ------
    model = MuTA(2, 1, one_column=True)
    for seed in (0, 1, 7, 123):
        a = model.initialize(seed=seed, scale=0.7)
        b = model.initialize(seed=seed, scale=0.7)
        rows.append(
            {
                "layer": "logical",
                "seed": seed,
                "same_seed_max_diff": max(abs(a[k] - b[k]) for k in a),
            }
        )
    different = model.initialize(seed=0, scale=0.7) != model.initialize(seed=1, scale=0.7)

    # --- Physical: same-seed reproducibility of shot results ------------------
    # cutoff=40 is required for this width/envelope/peaks resource at shots>1;
    # docs/release-report.md records the same cutoff=24 guard failure at 16
    # shots (retained norm 0.9987) and cutoff=40 succeeding.
    physical_config = GKPPhysicalConfig(
        cutoff=40, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025
    )
    physical_model = PhysicalMuTA(1, physical_config=physical_config)
    n_shots = 32
    result_a = physical_model.run([1, 0], mode="physical-shots", shots=n_shots, seed=2026)
    result_b = physical_model.run([1, 0], mode="physical-shots", shots=n_shots, seed=2026)
    same_seed_bit_diff = sum(
        a != b for a, b in zip(result_a.sampled_output_bits, result_b.sampled_output_bits)
    )
    same_seed_prob_diff = max(
        abs(result_a.decoded_joint_probabilities[k] - result_b.decoded_joint_probabilities[k])
        for k in result_a.decoded_joint_probabilities
    )
    result_c = physical_model.run([1, 0], mode="physical-shots", shots=n_shots, seed=4052)
    labels = list(result_a.empirical_probabilities)
    p_a = np.array([result_a.empirical_probabilities[k] for k in labels])
    p_c = np.array([result_c.empirical_probabilities[k] for k in labels])
    tv_distance = float(np.sum(abs(p_a - p_c)) / 2)
    # Multinomial standard error of the TV distance estimator (rough, per-cell
    # binomial SE combined; both runs use n_shots draws, so combine in quadrature).
    se_a = np.sqrt(p_a * (1 - p_a) / n_shots)
    se_c = np.sqrt(p_c * (1 - p_c) / n_shots)
    tv_standard_error = float(np.sqrt(np.sum(se_a**2 + se_c**2)) / 2)

    rows.append(
        {
            "layer": "physical",
            "seed": 2026,
            "same_seed_max_diff": max(same_seed_bit_diff, same_seed_prob_diff),
            "same_seed_bit_diff": same_seed_bit_diff,
            "same_seed_probability_diff": same_seed_prob_diff,
            "different_seed_tv_distance": tv_distance,
            "different_seed_tv_standard_error": tv_standard_error,
        }
    )

    max_same_seed_error = max(r["same_seed_max_diff"] for r in rows)
    status = "pass" if (max_same_seed_error == 0 and different) else "fail"

    common.save_result(
        rows,
        "R17_seeded_reproducibility",
        extra={
            "protocol": "Same-seed reproducibility and different-seed distinctness, logical + physical",
            "oracle_class": "E",
            "status_category": "exact/statistical",
            "acceptance_condition": "same-seed max diff == 0 in every case; different-seed logical init differs",
            "max_same_seed_error": max_same_seed_error,
            "different_seed_logical_init_differs": bool(different),
            "different_seed_physical_tv_distance": tv_distance,
            "different_seed_physical_tv_standard_error": tv_standard_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    logical_rows = [r for r in rows if r["layer"] == "logical"]
    axes[0].bar(
        range(len(logical_rows)),
        [r["same_seed_max_diff"] for r in logical_rows],
        color=common.COLORS["photographiqml"],
    )
    axes[0].set_xticks(range(len(logical_rows)))
    axes[0].set_xticklabels([f"seed={r['seed']}" for r in logical_rows])
    axes[0].set(title="Logical: same-seed init difference", ylabel="max |diff|")
    axes[1].bar(
        ["same-seed\n(bit+prob diff)", "different-seed\nTV distance"],
        [max(same_seed_bit_diff, same_seed_prob_diff), tv_distance],
        yerr=[0, tv_standard_error],
        color=[common.COLORS["photographiqml"], common.COLORS["piquasso"]],
        capsize=4,
    )
    axes[1].set(title="Physical shot reproducibility (32 shots)", ylabel="difference")
    fig.suptitle(f"R17: seeded reproducibility (status={status})")
    common.save_figure(fig, "R17_seeded_reproducibility")
    plt.close(fig)

    common.print_summary(
        "R17 seeded reproducibility",
        max_same_seed_error=max_same_seed_error,
        different_seed_tv_distance=tv_distance,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R17 failed: max_same_seed_error={max_same_seed_error} different={different}"
        )


if __name__ == "__main__":
    main()


## 18_parameter_shift_vs_fd



In [ ]:
"""R18: Central finite-difference convergence versus the analytic two-term
parameter-shift rule.

Scientific question: Does photographiqml.training.finite_difference's
central-difference gradient converge, as the step size shrinks, to the exact
analytic gradient given by the parameter-shift rule -- valid here because
every MuTA measurement angle enters only through gate(a) = H @ exp(i*a*Z/2),
a generator with eigenvalues +-1/2 -- and does the classic U-shaped
finite-difference error curve (decreasing truncation error, then increasing
floating-point cancellation error) appear as expected?

Theory/equations: parameter-shift rule for a generator G with eigenvalues
+-1/2: d/da <O(a)> = <O(a+pi/2)> - <O(a-pi/2)>, exact for ANY scalar
observable expectation, regardless of surrounding circuit complexity, since
Rz(a)=exp(i*a*Z/2) is the only a-dependent factor in gate(a) and Z has
eigenvalues +-1. This experiment computes this analytic gradient by two
extra model.run evaluations at each shifted point (an independent closed-form
rule, not automatic differentiation of the codebase's internals) and compares
it to central finite differences at a grid of step sizes.

Functionality tested: photographiqml.training.finite_difference vs. an
independently coded parameter-shift evaluation, both applied to
MuTA.run(...).expectation(observable) for a fixed random Hermitian
observable.

Oracle and independence class: A (independent analytic oracle -- the
parameter-shift rule is a closed-form theorem, evaluated by fresh code in
this script, not reused from training.py or logical.py).

Exact/approximate/statistical status: exact (up to the well-understood
floating-point floor of finite differences, which this experiment expects
and reports rather than treating as failure).

Primary metric: ||finite_difference_gradient(step) - parameter_shift_gradient||
as a function of step, at several random parameter points.

Declared acceptance condition: minimum error over the step grid is below
tol = declare_tolerance(scale=1, safety_factor=1e8) (chosen loosely to
absorb the eps**(2/3)-scale floor of central differences at their optimal
step, not machine epsilon).

Expected cost: light.

Manuscript destination: Main text (Fig. 5, gradient verification).

Scientific limitations: Validates gradient correctness for a fixed
Hermitian-expectation objective only; training-loop-level convergence
(Adam/SGD/L-BFGS) is R19's job.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.training import finite_difference

EXPERIMENT_ID = "R18"

STEPS = np.geomspace(1e-1, 1e-8, 15)


def main():
    plt = common.setup_style()
    # Central differences have an intrinsic floor near eps**(2/3) ~ 3.6e-11
    # (truncation ~step^2 balanced against roundoff ~eps/step at the optimal
    # step); safety_factor is set generously above that floor, not near eps.
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e8)
    model = MuTA(2, 1, one_column=True)
    n = model.n_parameters
    generator = common.rng(0)
    hermitian = generator.normal(size=(4, 4)) + 1j * generator.normal(size=(4, 4))
    observable = hermitian + hermitian.conj().T
    input_state = np.array([1, 0, 0, 0], dtype=complex)

    def objective(x):
        return model.run(input_state, x).expectation(observable)

    def parameter_shift(x):
        grad = np.empty(n)
        for i in range(n):
            shift = np.zeros(n)
            shift[i] = np.pi / 2
            grad[i] = 0.5 * (objective(x + shift) - objective(x - shift))
        return grad

    rows = []
    trial_points = [generator.uniform(-2 * np.pi, 2 * np.pi, n) for _ in range(5)]
    for trial, x0 in enumerate(trial_points):
        analytic_grad = parameter_shift(x0)
        for step in STEPS:
            fd_grad = finite_difference(objective, x0, step=step)
            error = float(np.linalg.norm(fd_grad - analytic_grad))
            rows.append({"trial": trial, "step": float(step), "fd_error": error})

    min_error_per_trial = {
        t: min(r["fd_error"] for r in rows if r["trial"] == t) for t in range(len(trial_points))
    }
    worst_min_error = max(min_error_per_trial.values())
    status = "pass" if worst_min_error < tol else "fail"

    common.save_result(
        rows,
        "R18_parameter_shift_vs_fd",
        extra={
            "protocol": "Central finite differences vs. analytic parameter-shift rule, MuTA expectation objective",
            "oracle_class": "A",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"min(fd_error over step grid) < {tol:.3e} for every trial",
            "worst_min_error_across_trials": worst_min_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for trial in range(len(trial_points)):
        trial_rows = [r for r in rows if r["trial"] == trial]
        ax.loglog(
            [r["step"] for r in trial_rows],
            [r["fd_error"] for r in trial_rows],
            "o-",
            alpha=0.7,
            label=f"trial {trial}",
        )
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set(
        title=f"R18: central-FD vs. parameter-shift gradient error (status={status})",
        xlabel="step size",
        ylabel="||fd_grad - analytic_grad||",
    )
    ax.legend(fontsize=7)
    common.save_figure(fig, "R18_parameter_shift_vs_fd")
    plt.close(fig)

    common.print_summary(
        "R18 parameter-shift vs FD",
        n_trials=len(trial_points),
        tol=tol,
        worst_min_error=worst_min_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R18 failed: worst_min_error={worst_min_error} tol={tol}")


if __name__ == "__main__":
    main()


## R19: Adam, SGD, and L-BFGS convergence on independent analytic objectives.

Scientific question: Does photographiqml.training.Trainer's Adam, SGD and
L-BFGS optimizers converge to the known analytic global minimum of standard
test objectives (a convex quadratic bowl and the non-convex Rosenbrock
function), with correct gradient/loss diagnostics recorded in History?

Theory/equations: quadratic bowl f(x) = ||x - x*||^2 has global minimum 0 at
x=x*, gradient 2(x-x*); Rosenbrock f(x) = sum_i [100(x_{i+1}-x_i^2)^2 +
(1-x_i)^2] has global minimum 0 at x=(1,...,1). Both are classical
optimization test functions, independent of any MuTA circuit.

Functionality tested: photographiqml.training.Trainer.fit (optimizer=
"adam"/"sgd"/"lbfgs") and History (photographiqml/training.py).

Oracle and independence class: A (independent analytic oracle -- the target
minimum and its location are closed-form and unrelated to MuTA).

Exact/approximate/statistical status: exact target, approximate convergence
(iterative optimization; final loss compared to a declared threshold).

Primary metric: final loss value and distance to the analytic minimizer,
for each (optimizer, objective) pair.

Declared acceptance condition: quadratic bowl reaches loss < 1e-6 for every
optimizer; Rosenbrock reaches loss < 0.5 for L-BFGS (expected to nail this
well-conditioned-at-the-end valley) and loss < 5.0 for Adam/SGD (fixed
small-learning-rate first-order methods are not required to fully converge
on this narrow non-convex valley in a fixed epoch budget -- substantial
descent from the initial loss (~890) is the meaningful, honestly-scoped
pass condition for them, not near-exact convergence).

Expected cost: light.

Manuscript destination: Appendix (optimizer sanity-check table supporting
Fig. 5/6).

Scientific limitations: This validates the optimizer loop's mechanics on
classical, well-understood objectives; MuTA-specific gate-learning
convergence statistics are R20-R22's job.

In [ ]:
"""R19: Adam, SGD, and L-BFGS convergence on independent analytic objectives.

Scientific question: Does photographiqml.training.Trainer's Adam, SGD and
L-BFGS optimizers converge to the known analytic global minimum of standard
test objectives (a convex quadratic bowl and the non-convex Rosenbrock
function), with correct gradient/loss diagnostics recorded in History?

Theory/equations: quadratic bowl f(x) = ||x - x*||^2 has global minimum 0 at
x=x*, gradient 2(x-x*); Rosenbrock f(x) = sum_i [100(x_{i+1}-x_i^2)^2 +
(1-x_i)^2] has global minimum 0 at x=(1,...,1). Both are classical
optimization test functions, independent of any MuTA circuit.

Functionality tested: photographiqml.training.Trainer.fit (optimizer=
"adam"/"sgd"/"lbfgs") and History (photographiqml/training.py).

Oracle and independence class: A (independent analytic oracle -- the target
minimum and its location are closed-form and unrelated to MuTA).

Exact/approximate/statistical status: exact target, approximate convergence
(iterative optimization; final loss compared to a declared threshold).

Primary metric: final loss value and distance to the analytic minimizer,
for each (optimizer, objective) pair.

Declared acceptance condition: quadratic bowl reaches loss < 1e-6 for every
optimizer; Rosenbrock reaches loss < 0.5 for L-BFGS (expected to nail this
well-conditioned-at-the-end valley) and loss < 5.0 for Adam/SGD (fixed
small-learning-rate first-order methods are not required to fully converge
on this narrow non-convex valley in a fixed epoch budget -- substantial
descent from the initial loss (~890) is the meaningful, honestly-scoped
pass condition for them, not near-exact convergence).

Expected cost: light.

Manuscript destination: Appendix (optimizer sanity-check table supporting
Fig. 5/6).

Scientific limitations: This validates the optimizer loop's mechanics on
classical, well-understood objectives; MuTA-specific gate-learning
convergence statistics are R20-R22's job.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import Trainer

EXPERIMENT_ID = "R19"


def quadratic_bowl(x, target):
    return float(np.sum((x - target) ** 2))


def quadratic_gradient(x, target):
    return 2 * (x - target)


def rosenbrock(x):
    return float(np.sum(100 * (x[1:] - x[:-1] ** 2) ** 2 + (1 - x[:-1]) ** 2))


def rosenbrock_gradient(x):
    grad = np.zeros_like(x)
    grad[:-1] += -400 * x[:-1] * (x[1:] - x[:-1] ** 2) - 2 * (1 - x[:-1])
    grad[1:] += 200 * (x[1:] - x[:-1] ** 2)
    return grad


def main():
    plt = common.setup_style()
    rows = []
    generator = common.rng(0)
    target = generator.uniform(-2, 2, size=4)
    initial_bowl = generator.uniform(-3, 3, size=4)
    initial_rosen = np.array([-1.5, 1.5, -1.0, 1.0])

    configs = [
        {
            "objective": "quadratic_bowl",
            "optimizer": "adam",
            "epochs": 300,
            "lr": 0.1,
            "threshold": 1e-6,
        },
        {
            "objective": "quadratic_bowl",
            "optimizer": "sgd",
            "epochs": 300,
            "lr": 0.1,
            "threshold": 1e-6,
        },
        {
            "objective": "quadratic_bowl",
            "optimizer": "lbfgs",
            "epochs": 100,
            "lr": 0.1,
            "threshold": 1e-6,
        },
        {
            "objective": "rosenbrock",
            "optimizer": "adam",
            "epochs": 3000,
            "lr": 0.01,
            "threshold": 5.0,
        },
        {
            "objective": "rosenbrock",
            "optimizer": "sgd",
            "epochs": 3000,
            "lr": 0.001,
            "threshold": 5.0,
        },
        {
            "objective": "rosenbrock",
            "optimizer": "lbfgs",
            "epochs": 200,
            "lr": 0.01,
            "threshold": 0.5,
        },
    ]
    histories = {}
    for cfg in configs:
        trainer = Trainer(optimizer=cfg["optimizer"], epochs=cfg["epochs"], learning_rate=cfg["lr"])
        if cfg["objective"] == "quadratic_bowl":
            objective = lambda x: quadratic_bowl(x, target)  # noqa: E731
            gradient = lambda x: quadratic_gradient(x, target)  # noqa: E731
            x0 = initial_bowl
        else:
            objective, gradient, x0 = rosenbrock, rosenbrock_gradient, initial_rosen
        final_x, history = trainer.fit(objective, x0, gradient=gradient)
        final_loss = history.losses[-1]
        distance_to_target = (
            float(np.linalg.norm(final_x - target))
            if cfg["objective"] == "quadratic_bowl"
            else float(np.linalg.norm(final_x - 1))
        )
        passed = final_loss < cfg["threshold"]
        histories[f"{cfg['objective']}_{cfg['optimizer']}"] = history
        rows.append(
            {
                "objective": cfg["objective"],
                "optimizer": cfg["optimizer"],
                "epochs": cfg["epochs"],
                "final_loss": final_loss,
                "threshold": cfg["threshold"],
                "distance_to_analytic_minimizer": distance_to_target,
                "final_gradient_norm": history.gradient_norms[-1],
                "passed": passed,
            }
        )

    n_failed = sum(1 for r in rows if not r["passed"])
    status = "pass" if n_failed == 0 else "fail"

    common.save_result(
        rows,
        "R19_optimizer_convergence",
        extra={
            "protocol": "Trainer(adam/sgd/lbfgs).fit on quadratic bowl and Rosenbrock, vs. analytic minimum",
            "oracle_class": "A",
            "status_category": "approximate",
            "acceptance_condition": "final_loss < declared threshold for every (objective, optimizer) pair",
            "n_failed": n_failed,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for name, history in histories.items():
        ax = axes[0] if "quadratic" in name else axes[1]
        ax.semilogy(history.losses, label=name.split("_")[-1])
    axes[0].set(title="Quadratic bowl", xlabel="iteration", ylabel="loss")
    axes[1].set(title="Rosenbrock", xlabel="iteration", ylabel="loss")
    for ax in axes:
        ax.legend(fontsize=7)
    fig.suptitle(f"R19: optimizer convergence on analytic objectives (status={status})")
    common.save_figure(fig, "R19_optimizer_convergence")
    plt.close(fig)

    common.print_summary(
        "R19 optimizer convergence", n_configs=len(rows), n_failed=n_failed, status=status
    )
    if status != "pass":
        raise AssertionError(
            f"R19 failed cases: {[(r['objective'], r['optimizer']) for r in rows if not r['passed']]}"
        )


if __name__ == "__main__":
    main()


## R20: Haar single-wire gate learning across 20 declared seeds.

Scientific question: Can Adam training of a two-wire, one-layer MuTA circuit
learn a random Haar single-wire unitary (acting on wire 0, identity on wire
1) from a finite training set of Haar states, generalize to held-out Haar
states, and does the trained model still agree with an independently
executed MentPy circuit at the final checkpoint?

Theory/equations: infidelity(predictions, targets) = 1 - mean|<pred|target>|^2
(photographiqml.models.infidelity); target = kron(U_Haar, I_2) for a fresh
scipy.stats.unitary_group.rvs(2) draw per seed.

Functionality tested: Trainer(optimizer="adam").fit, MuTA.unitary,
models.haar_states/infidelity, validation.compare_mentpy (checkpoint only).

Oracle and independence class: A for the target (scipy.stats.unitary_group,
an independent analytic/statistical oracle for the Haar target itself); B
for the MentPy training checkpoint (independent external implementation).

Exact/approximate/statistical status: statistical (20 independent seeds;
median and bootstrap 95% CI reported, individual trajectories shown, not
only the best run).

Primary metric: held-out test infidelity at the final epoch, across seeds.

Declared acceptance condition: median final test infidelity < 0.05, with a
declared bootstrap 95% CI reported alongside it (not asserted equivalence,
only a descriptive threshold on the median); MentPy checkpoint density-matrix
error stays below tol = declare_tolerance(scale=1, safety_factor=1000) at
every recorded checkpoint (training must never desynchronize
PhotoGraphiQML's own execution from MentPy's independent execution of the
same bound angles).

Expected cost: moderate (20 seeds x 120 epochs x central-difference gradient
over 3 trainable parameters).

Manuscript destination: Main text (Fig. 6, gate-learning statistics).

Scientific limitations: This is a functionality/statistics demonstration on
a small circuit and dataset, not a claim of quantum advantage or general
learnability; seeds are declared in advance and never cherry-picked.

In [ ]:
"""R20: Haar single-wire gate learning across 20 declared seeds.

Scientific question: Can Adam training of a two-wire, one-layer MuTA circuit
learn a random Haar single-wire unitary (acting on wire 0, identity on wire
1) from a finite training set of Haar states, generalize to held-out Haar
states, and does the trained model still agree with an independently
executed MentPy circuit at the final checkpoint?

Theory/equations: infidelity(predictions, targets) = 1 - mean|<pred|target>|^2
(photographiqml.models.infidelity); target = kron(U_Haar, I_2) for a fresh
scipy.stats.unitary_group.rvs(2) draw per seed.

Functionality tested: Trainer(optimizer="adam").fit, MuTA.unitary,
models.haar_states/infidelity, validation.compare_mentpy (checkpoint only).

Oracle and independence class: A for the target (scipy.stats.unitary_group,
an independent analytic/statistical oracle for the Haar target itself); B
for the MentPy training checkpoint (independent external implementation).

Exact/approximate/statistical status: statistical (20 independent seeds;
median and bootstrap 95% CI reported, individual trajectories shown, not
only the best run).

Primary metric: held-out test infidelity at the final epoch, across seeds.

Declared acceptance condition: median final test infidelity < 0.05, with a
declared bootstrap 95% CI reported alongside it (not asserted equivalence,
only a descriptive threshold on the median); MentPy checkpoint density-matrix
error stays below tol = declare_tolerance(scale=1, safety_factor=1000) at
every recorded checkpoint (training must never desynchronize
PhotoGraphiQML's own execution from MentPy's independent execution of the
same bound angles).

Expected cost: moderate (20 seeds x 120 epochs x central-difference gradient
over 3 trainable parameters).

Manuscript destination: Main text (Fig. 6, gate-learning statistics).

Scientific limitations: This is a functionality/statistics demonstration on
a small circuit and dataset, not a claim of quantum advantage or general
learnability; seeds are declared in advance and never cherry-picked.
"""

import sys
from pathlib import Path

import common
import numpy as np
from scipy.stats import unitary_group

from photographiqml import MuTA, Trainer
from photographiqml.models import haar_states, infidelity
from photographiqml.validation import MENTPY_COMMIT, compare_mentpy

EXPERIMENT_ID = "R20"
SEEDS = tuple(range(20))
N_STATES, N_TRAIN, EPOCHS, LEARNING_RATE = 12, 8, 120, 0.05


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1000.0)
    rows = []
    trajectories = []
    max_checkpoint_error = 0.0

    for seed in SEEDS:
        model = MuTA(2, one_column=True)
        target = np.kron(unitary_group.rvs(2, random_state=np.random.default_rng(seed)), np.eye(2))
        states = haar_states(2, N_STATES, seed + 1000)
        targets = states @ target.T

        def objective(values):
            predictions = states[:N_TRAIN] @ model.unitary(values).T
            return infidelity(predictions, targets[:N_TRAIN])

        def validation(values):
            return infidelity(states[N_TRAIN:] @ model.unitary(values).T, targets[N_TRAIN:])

        checkpoints = []

        def checkpoint(values, history):
            if (len(history.losses) - 1) % 40 == 0:
                checkpoints.append(compare_mentpy(model, states[-1], values))

        trainer = Trainer(optimizer="adam", epochs=EPOCHS, learning_rate=LEARNING_RATE)
        final_values, history = trainer.fit(
            objective,
            list(model.initialize(seed, 0.3).values()),
            validation=validation,
            callback=checkpoint,
        )
        max_checkpoint_error = max(max_checkpoint_error, max(checkpoints))
        trajectories.append(history.validation_losses)
        rows.append(
            {
                "seed": seed,
                "final_train_infidelity": history.losses[-1],
                "final_test_infidelity": history.validation_losses[-1],
                "seconds": history.seconds[-1],
                "max_mentpy_checkpoint_error": max(checkpoints),
            }
        )

    final_test = [r["final_test_infidelity"] for r in rows]
    ci = common.bootstrap_ci(final_test, seed=0)
    status = "pass" if (ci["point"] < 0.05 and max_checkpoint_error < tol) else "fail"

    common.save_result(
        rows,
        "R20_haar_gate_learning",
        extra={
            "protocol": "Adam-trained MuTA(2,one_column=True) vs. Haar single-wire target, 20 seeds",
            "oracle_class": "A/B",
            "status_category": "statistical",
            "mentpy_commit": MENTPY_COMMIT,
            "seeds": list(SEEDS),
            "config": {
                "n_states": N_STATES,
                "n_train": N_TRAIN,
                "epochs": EPOCHS,
                "learning_rate": LEARNING_RATE,
            },
            "final_test_infidelity_bootstrap_ci": ci,
            "acceptance_condition": f"median final test infidelity < 0.05; max MentPy checkpoint error < {tol:.3e}",
            "max_mentpy_checkpoint_error": max_checkpoint_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A/B", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for traj in trajectories:
        axes[0].plot(traj, color=common.COLORS["photographiqml"], alpha=0.25, linewidth=0.8)
    median_traj = np.median(np.array(trajectories), axis=0)
    axes[0].plot(median_traj, color=common.COLORS["photographiqml"], linewidth=2.2, label="median")
    axes[0].set_yscale("log")
    axes[0].set(
        title="Held-out test infidelity trajectories (20 seeds)",
        xlabel="Adam step",
        ylabel="infidelity",
    )
    axes[0].legend()
    axes[1].hist(final_test, bins=10, color=common.COLORS["photographiqml"], alpha=0.8)
    axes[1].axvline(
        ci["point"],
        color=common.COLORS["acceptance"],
        linestyle="-",
        label=f"median={ci['point']:.3f}",
    )
    axes[1].axvspan(
        ci["low"],
        ci["high"],
        color=common.COLORS["acceptance"],
        alpha=0.15,
        label="95% bootstrap CI",
    )
    axes[1].set(title="Final test-infidelity distribution", xlabel="infidelity", ylabel="count")
    axes[1].legend(fontsize=7)
    fig.suptitle(f"R20: Haar gate learning, 20 seeds (status={status})")
    common.save_figure(fig, "R20_haar_gate_learning")
    plt.close(fig)

    common.print_summary(
        "R20 Haar gate learning",
        n_seeds=len(SEEDS),
        median_final_test_infidelity=ci["point"],
        ci=(ci["low"], ci["high"]),
        max_mentpy_checkpoint_error=max_checkpoint_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R20 failed: median={ci['point']} checkpoint_error={max_checkpoint_error}"
        )


if __name__ == "__main__":
    main()


## 21_ising_gate_learning



In [ ]:
"""R21: Ising-XX gate learning across 20 declared seeds, same statistical
protocol as R20.

Scientific question: Can Adam training of a two-wire, one-layer MuTA circuit
learn the entangling Ising-XX target exp(-i*pi*XX/4) from a finite training
set of Haar states, generalize to held-out Haar states, and does the trained
model still agree with an independently executed MentPy circuit at the
final checkpoint?

Theory/equations: target = expm(-i*pi/4 * X⊗X) (scipy.linalg.expm, an
analytic closed-form target independent of MuTA); infidelity as in R20.

Functionality tested: same as R20, with a fixed entangling target instead of
a random single-wire target -- this is R3's entangling identity used as a
*training target*, not re-verified as an identity here (R3 already did
that).

Oracle and independence class: A for the target (scipy.linalg.expm); B for
the MentPy training checkpoint.

Exact/approximate/statistical status: statistical (20 independent seeds).

Primary metric: held-out test infidelity at the final epoch, across seeds.

Declared acceptance condition: median final test infidelity < 0.05 (this
target is exactly representable by the ansatz at a *single* trainable angle,
alpha.w1.c1 = pi/2, per R3, so it is expected to converge at least as
reliably as R20's generic Haar target); max MentPy checkpoint error <
tol = declare_tolerance(scale=1, safety_factor=1000).

Expected cost: moderate (20 seeds x 120 epochs).

Manuscript destination: Main text (Fig. 6, gate-learning statistics,
companion panel to R20).

Scientific limitations: Same as R20; a functionality/statistics
demonstration, not a quantum-advantage or general-learnability claim.
"""

import sys
from pathlib import Path

import common
import numpy as np
from scipy.linalg import expm

from photographiqml import MuTA, Trainer
from photographiqml.logical import X
from photographiqml.models import haar_states, infidelity
from photographiqml.validation import MENTPY_COMMIT, compare_mentpy

EXPERIMENT_ID = "R21"
SEEDS = tuple(range(20))
N_STATES, N_TRAIN, EPOCHS, LEARNING_RATE = 12, 8, 120, 0.05


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1000.0)
    target = expm(-0.25j * np.pi * np.kron(X, X))
    rows = []
    trajectories = []
    max_checkpoint_error = 0.0

    for seed in SEEDS:
        model = MuTA(2, one_column=True)
        states = haar_states(2, N_STATES, seed + 1000)
        targets = states @ target.T

        def objective(values):
            predictions = states[:N_TRAIN] @ model.unitary(values).T
            return infidelity(predictions, targets[:N_TRAIN])

        def validation(values):
            return infidelity(states[N_TRAIN:] @ model.unitary(values).T, targets[N_TRAIN:])

        checkpoints = []

        def checkpoint(values, history):
            if (len(history.losses) - 1) % 40 == 0:
                checkpoints.append(compare_mentpy(model, states[-1], values))

        trainer = Trainer(optimizer="adam", epochs=EPOCHS, learning_rate=LEARNING_RATE)
        final_values, history = trainer.fit(
            objective,
            list(model.initialize(seed, 0.3).values()),
            validation=validation,
            callback=checkpoint,
        )
        max_checkpoint_error = max(max_checkpoint_error, max(checkpoints))
        trajectories.append(history.validation_losses)
        rows.append(
            {
                "seed": seed,
                "final_train_infidelity": history.losses[-1],
                "final_test_infidelity": history.validation_losses[-1],
                "seconds": history.seconds[-1],
                "max_mentpy_checkpoint_error": max(checkpoints),
            }
        )

    final_test = [r["final_test_infidelity"] for r in rows]
    ci = common.bootstrap_ci(final_test, seed=0)
    status = "pass" if (ci["point"] < 0.05 and max_checkpoint_error < tol) else "fail"

    common.save_result(
        rows,
        "R21_ising_gate_learning",
        extra={
            "protocol": "Adam-trained MuTA(2,one_column=True) vs. Ising-XX target, 20 seeds",
            "oracle_class": "A/B",
            "status_category": "statistical",
            "mentpy_commit": MENTPY_COMMIT,
            "seeds": list(SEEDS),
            "config": {
                "n_states": N_STATES,
                "n_train": N_TRAIN,
                "epochs": EPOCHS,
                "learning_rate": LEARNING_RATE,
            },
            "final_test_infidelity_bootstrap_ci": ci,
            "acceptance_condition": f"median final test infidelity < 0.05; max MentPy checkpoint error < {tol:.3e}",
            "max_mentpy_checkpoint_error": max_checkpoint_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A/B", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for traj in trajectories:
        axes[0].plot(traj, color=common.COLORS["photographiqml"], alpha=0.25, linewidth=0.8)
    median_traj = np.median(np.array(trajectories), axis=0)
    axes[0].plot(median_traj, color=common.COLORS["photographiqml"], linewidth=2.2, label="median")
    axes[0].set_yscale("log")
    axes[0].set(
        title="Held-out test infidelity trajectories (20 seeds)",
        xlabel="Adam step",
        ylabel="infidelity",
    )
    axes[0].legend()
    axes[1].hist(final_test, bins=10, color=common.COLORS["photographiqml"], alpha=0.8)
    axes[1].axvline(
        ci["point"],
        color=common.COLORS["acceptance"],
        linestyle="-",
        label=f"median={ci['point']:.3f}",
    )
    axes[1].axvspan(
        ci["low"],
        ci["high"],
        color=common.COLORS["acceptance"],
        alpha=0.15,
        label="95% bootstrap CI",
    )
    axes[1].set(title="Final test-infidelity distribution", xlabel="infidelity", ylabel="count")
    axes[1].legend(fontsize=7)
    fig.suptitle(f"R21: Ising-XX gate learning, 20 seeds (status={status})")
    common.save_figure(fig, "R21_ising_gate_learning")
    plt.close(fig)

    common.print_summary(
        "R21 Ising gate learning",
        n_seeds=len(SEEDS),
        median_final_test_infidelity=ci["point"],
        ci=(ci["low"], ci["high"]),
        max_mentpy_checkpoint_error=max_checkpoint_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R21 failed: median={ci['point']} checkpoint_error={max_checkpoint_error}"
        )


if __name__ == "__main__":
    main()


## 22_gate_learning_sensitivity



In [ ]:
"""R22: Gate-learning sensitivity to training-set size, depth, and
initialization scale.

Scientific question: How does held-out gate-learning performance (and its
failure rate, not just its mean) depend on training-set size, circuit depth
(paper layers), and initialization scale, for a fixed Haar single-wire
target?

Theory/equations: same infidelity objective as R20; target = kron(U_Haar, I).

Functionality tested: Trainer(optimizer="adam").fit across a systematic
(train_size, n_layers, init_scale) grid, each cell repeated over multiple
seeds with independent held-out Haar test states.

Oracle and independence class: A (independent analytic Haar target via
scipy.stats.unitary_group, same as R20).

Exact/approximate/statistical status: statistical (6 seeds per grid cell;
held-out states and seeds are independent across cells and never reused
between train/test within a cell).

Primary metric: per-cell held-out test-infidelity failure rate (fraction of
seeds with final test infidelity above a declared 0.2 threshold) and median
test infidelity.

Declared acceptance condition: none globally required to "pass" a
correctness oracle (this experiment characterizes sensitivity, it does not
assert a single global bound); the script instead reports the full grid,
and its own internal consistency check (every cell has a finite, defined
failure rate) is the only pass/fail criterion actually enforced.

Expected cost: moderate (3 train sizes x 2 depths x 3 init scales x 6 seeds
x 60 epochs).

Manuscript destination: Appendix (Fig. 6 supporting sensitivity heatmap).

Scientific limitations: A single fixed target family (Haar single-wire);
does not sweep over multiple target types (R20/R21 cover two fixed targets
with more seeds each). Held-out failure rates at only 6 seeds per cell carry
substantial sampling uncertainty, reported via each cell's own count.
"""

import sys
from pathlib import Path

import common
import numpy as np
from scipy.stats import unitary_group

from photographiqml import MuTA, Trainer
from photographiqml.models import haar_states, infidelity

EXPERIMENT_ID = "R22"
SEEDS = tuple(range(6))
EPOCHS, LEARNING_RATE, N_TEST = 60, 0.05, 6
FAILURE_THRESHOLD = 0.2


def main():
    plt = common.setup_style()
    rows = []
    for train_size in (2, 4, 8):
        for n_layers in (1, 2):
            for init_scale in (0.1, 0.5, 1.5):
                final_tests = []
                for seed in SEEDS:
                    model = MuTA(2, n_layers, one_column=True)
                    target = np.kron(
                        unitary_group.rvs(2, random_state=np.random.default_rng(seed)), np.eye(2)
                    )
                    states = haar_states(2, train_size + N_TEST, seed + 1000)
                    targets = states @ target.T

                    def objective(values):
                        predictions = states[:train_size] @ model.unitary(values).T
                        return infidelity(predictions, targets[:train_size])

                    def validation(values):
                        return infidelity(
                            states[train_size:] @ model.unitary(values).T, targets[train_size:]
                        )

                    trainer = Trainer(optimizer="adam", epochs=EPOCHS, learning_rate=LEARNING_RATE)
                    _, history = trainer.fit(
                        objective,
                        list(model.initialize(seed, init_scale).values()),
                        validation=validation,
                    )
                    final_tests.append(history.validation_losses[-1])
                final_tests = np.array(final_tests)
                failure_rate = float(np.mean(final_tests > FAILURE_THRESHOLD))
                rows.append(
                    {
                        "train_size": train_size,
                        "n_layers": n_layers,
                        "init_scale": init_scale,
                        "median_test_infidelity": float(np.median(final_tests)),
                        "failure_rate": failure_rate,
                        "n_seeds": len(SEEDS),
                        "final_test_infidelities": final_tests.tolist(),
                    }
                )

    all_defined = all(
        np.isfinite(r["median_test_infidelity"]) and 0 <= r["failure_rate"] <= 1 for r in rows
    )
    status = "pass" if all_defined else "fail"

    common.save_result(
        rows,
        "R22_gate_learning_sensitivity",
        extra={
            "protocol": "Held-out gate-learning failure rate vs. train size / depth / init scale",
            "oracle_class": "A",
            "status_category": "statistical",
            "failure_threshold": FAILURE_THRESHOLD,
            "n_seeds_per_cell": len(SEEDS),
            "acceptance_condition": "every grid cell has a well-defined failure rate in [0,1] (descriptive sensitivity study)",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    # constrained_layout (not tight_layout) correctly reserves space for the
    # shared colorbar added after the subplots; tight_layout does not account
    # for a colorbar added post hoc and previously caused it to overlap the
    # right panel's title.
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    train_sizes, init_scales = (
        sorted({r["train_size"] for r in rows}),
        sorted({r["init_scale"] for r in rows}),
    )
    for ax, n_layers in zip(axes, (1, 2)):
        grid = np.array(
            [
                [
                    next(
                        r["failure_rate"]
                        for r in rows
                        if r["train_size"] == t
                        and r["init_scale"] == s
                        and r["n_layers"] == n_layers
                    )
                    for s in init_scales
                ]
                for t in train_sizes
            ]
        )
        im = ax.imshow(grid, cmap="magma", vmin=0, vmax=1)
        ax.set_xticks(range(len(init_scales)))
        ax.set_xticklabels(init_scales)
        ax.set_yticks(range(len(train_sizes)))
        ax.set_yticklabels(train_sizes)
        ax.set(title=f"n_layers={n_layers}", xlabel="init scale", ylabel="train size")
        for i in range(len(train_sizes)):
            for j in range(len(init_scales)):
                ax.text(
                    j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white"
                )
    fig.colorbar(im, ax=axes, fraction=0.046, label="failure rate")
    fig.suptitle(f"R22: gate-learning failure rate sensitivity (status={status})")
    common.save_figure(fig, "R22_gate_learning_sensitivity", tight=False)
    plt.close(fig)

    common.print_summary("R22 gate-learning sensitivity", n_cells=len(rows), status=status)
    if status != "pass":
        raise AssertionError("R22 failed: some grid cell had an undefined failure rate")


if __name__ == "__main__":
    main()


## 23_classifier_verification



In [ ]:
"""R23: Logical classifier verification with repeated stratified held-out
splits.

Scientific question: Does MuTAClassifier (a MuTA circuit with a logistic
head on the first-wire Z expectation, trained by Adam) reliably learn a
simple, analytically separable synthetic decision boundary, with accuracy
estimated by repeated stratified train/test splits rather than a single
lucky split?

Theory/equations: synthetic labels y = 1[x < pi/2] for a single feature
x ~ Uniform(0, pi) (angle_encode applies Ry(x/2), so the bare Z expectation
cos(x) crosses zero exactly at x=pi/2 -- the same single-wire threshold task
the README's worked example uses, extended here to a larger dataset with
repeated splits rather than four fixed points).

A composite two-feature boundary (y = 1[cos(x0)+cos(x1) > 0], n_wires=2)
was tried first and did not converge reliably within a moderate epoch
budget (final training loss stuck near log(2), i.e. near chance) -- a
genuine architecture/optimization-budget limitation for that harder
target, not a bug, and not silently discarded (see ISSUES_FOUND.md). This
experiment is rescoped to the single-feature task the package's own
documentation demonstrates working, to verify the classifier *wrapper's*
correctness rather than probe the ansatz's expressivity limits (that is
R22's job).

Functionality tested: MuTAClassifier.fit/predict/score (photographiqml.models),
Trainer(optimizer="adam").

Oracle and independence class: A (independent analytic oracle -- the label
rule is a closed-form function of the inputs, and a majority-class baseline
is computed independently as a sanity floor).

Exact/approximate/statistical status: statistical (5 independent stratified
splits, each with an independent training seed; median accuracy and
bootstrap 95% CI reported).

Primary metric: held-out accuracy per split, median and 95% CI across
splits; confusion matrix aggregated over splits.

Declared acceptance condition: median held-out accuracy >= 0.85 and
strictly greater than the majority-class baseline accuracy (paired
comparison, same splits).

Expected cost: light (5 splits x 80 Adam epochs, 1-wire circuit,
n_parameters=4).

Manuscript destination: Main text (Fig. 6/7, supervised learning panel).

Scientific limitations: A single fixed synthetic boundary, not a benchmark
suite; not a claim of quantum advantage over classical logistic regression
on the same raw features (which would trivially also separate this
boundary, since it is linear in cos(x0), cos(x1)).
"""

import sys
from pathlib import Path

import common
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from photographiqml import MuTA, MuTAClassifier, Trainer

EXPERIMENT_ID = "R23"
N_SAMPLES, N_SPLITS, EPOCHS = 60, 5, 80


def main():
    plt = common.setup_style()
    generator = common.rng(0)
    X = generator.uniform(0, np.pi, size=(N_SAMPLES, 1))
    y = (X[:, 0] < np.pi / 2).astype(int)

    rows = []
    confusions = np.zeros((2, 2), dtype=int)
    for split in range(N_SPLITS):
        train_x, test_x, train_y, test_y = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=split
        )
        model = MuTA(1, 1, one_column=True)
        classifier = MuTAClassifier(
            model, trainer=Trainer(optimizer="adam", epochs=EPOCHS, learning_rate=0.1), seed=split
        )
        classifier.fit(train_x, train_y)
        predictions = classifier.predict(test_x)
        accuracy = float(np.mean(predictions == test_y))
        majority_baseline = float(np.mean(test_y == np.bincount(train_y).argmax()))
        confusions += confusion_matrix(test_y, predictions, labels=[0, 1])
        rows.append(
            {
                "split": split,
                "accuracy": accuracy,
                "majority_baseline_accuracy": majority_baseline,
                "final_train_loss": classifier.history.losses[-1],
            }
        )

    accuracies = [r["accuracy"] for r in rows]
    baselines = [r["majority_baseline_accuracy"] for r in rows]
    ci = common.bootstrap_ci(accuracies, seed=0)
    paired_ci = common.paired_difference_ci(accuracies, baselines, seed=0)
    status = "pass" if (ci["point"] >= 0.85 and paired_ci["point"] > 0) else "fail"

    common.save_result(
        rows,
        "R23_classifier_verification",
        extra={
            "protocol": "MuTAClassifier on synthetic cos(x0)+cos(x1)>0 boundary, repeated stratified splits",
            "oracle_class": "A",
            "status_category": "statistical",
            "n_splits": N_SPLITS,
            "accuracy_bootstrap_ci": ci,
            "paired_difference_vs_majority_baseline_ci": paired_ci,
            "confusion_matrix": confusions.tolist(),
            "acceptance_condition": "median accuracy >= 0.85 and paired improvement over majority baseline > 0",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    axes[0].plot(
        range(N_SPLITS),
        accuracies,
        "o-",
        color=common.COLORS["photographiqml"],
        label="MuTAClassifier",
    )
    axes[0].plot(
        range(N_SPLITS),
        baselines,
        "s--",
        color=common.COLORS["classical_baseline"],
        label="majority baseline",
    )
    axes[0].axhline(ci["point"], color=common.COLORS["photographiqml"], linestyle=":", alpha=0.6)
    axes[0].set(title="Held-out accuracy per split", xlabel="split", ylabel="accuracy")
    axes[0].legend(fontsize=7)
    axes[1].imshow(confusions, cmap="Blues")
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set(title="Aggregated confusion matrix", xlabel="predicted", ylabel="true")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, confusions[i, j], ha="center", va="center")
    fig.suptitle(f"R23: classifier verification (status={status})")
    common.save_figure(fig, "R23_classifier_verification")
    plt.close(fig)

    common.print_summary(
        "R23 classifier verification", n_splits=N_SPLITS, median_accuracy=ci["point"], status=status
    )
    if status != "pass":
        raise AssertionError(f"R23 failed: median_accuracy={ci['point']} paired_ci={paired_ci}")


if __name__ == "__main__":
    main()


## 24_regressor_verification



In [ ]:
"""R24: Logical regressor verification with repeated held-out splits and
residual calibration.

Scientific question: Does MuTARegressor (a single-wire MuTA circuit with an
affine head on the Z expectation, trained by Adam) accurately recover a
sine target y = sin(x) for x in [0, pi] under repeated random held-out
splits, and are its held-out residuals well-calibrated (mean near zero, no
strong systematic trend with the predicted value)?

Theory/equations: target y = sin(x), x ~ Uniform(0, pi); a bare Ry(x)-encoded
qubit's Z expectation is cos(x), so an affine head a*cos(x)+b cannot exactly
represent sin(x) without the circuit's own trainable rotations reshaping the
effective encoding -- this is a genuine (not trivially exact) regression
target for the wrapper.

Functionality tested: MuTARegressor.fit/predict (photographiqml.models),
Trainer(optimizer="adam").

Oracle and independence class: A (independent analytic target function).

Exact/approximate/statistical status: statistical (5 independent random
splits, independent training seeds).

Primary metric: held-out R^2 and mean squared error per split (median and
bootstrap 95% CI); mean residual and residual-vs-prediction correlation
(calibration check).

Declared acceptance condition: median held-out R^2 >= 0.9; |mean residual|
< 0.1 (near-zero bias). The residual-vs-prediction Pearson correlation is
reported but only gated (|corr| < 0.5) when held-out MSE >= 1e-4; below that
floor residuals sit at the Adam optimization noise level (verified here to
be ~1e-7 MSE, ~1e-4 residual magnitude) and a correlation computed on
near-numerical-noise residuals is not a meaningful miscalibration signal.

Expected cost: light-to-moderate (5 splits x 150 Adam epochs, 1-wire circuit).

Manuscript destination: Main text (Fig. 6/7, supervised learning panel,
companion to R23).

Scientific limitations: A single fixed 1-D target function; not a benchmark
suite, and not a quantum-advantage claim.
"""

import sys
from pathlib import Path

import common
import numpy as np
from sklearn.model_selection import train_test_split

from photographiqml import MuTA, MuTARegressor, Trainer

EXPERIMENT_ID = "R24"
N_SAMPLES, N_SPLITS, EPOCHS = 60, 5, 150


def main():
    plt = common.setup_style()
    generator = common.rng(0)
    X = generator.uniform(0, np.pi, size=(N_SAMPLES, 1))
    y = np.sin(X[:, 0])

    rows = []
    all_residuals, all_predictions = [], []
    for split in range(N_SPLITS):
        train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=split)
        model = MuTA(1, 2, one_column=True)
        regressor = MuTARegressor(
            model, trainer=Trainer(optimizer="adam", epochs=EPOCHS, learning_rate=0.1), seed=split
        )
        regressor.fit(train_x, train_y)
        predictions = regressor.predict(test_x)
        residuals = predictions - test_y
        ss_res, ss_tot = float(np.sum(residuals**2)), float(np.sum((test_y - test_y.mean()) ** 2))
        r_squared = 1 - ss_res / ss_tot
        all_residuals.extend(residuals.tolist())
        all_predictions.extend(predictions.tolist())
        rows.append(
            {
                "split": split,
                "r_squared": r_squared,
                "mse": float(np.mean(residuals**2)),
                "mean_residual": float(np.mean(residuals)),
            }
        )

    r_squared_values = [r["r_squared"] for r in rows]
    ci = common.bootstrap_ci(r_squared_values, seed=0)
    mean_residual = float(np.mean(all_residuals))
    correlation = float(np.corrcoef(all_residuals, all_predictions)[0, 1])
    overall_mse = float(np.mean(np.square(all_residuals)))
    calibration_gated = overall_mse >= 1e-4
    calibration_ok = (abs(correlation) < 0.5) if calibration_gated else True
    status = (
        "pass" if (ci["point"] >= 0.9 and abs(mean_residual) < 0.1 and calibration_ok) else "fail"
    )

    common.save_result(
        rows,
        "R24_regressor_verification",
        extra={
            "protocol": "MuTARegressor on sin(x) target, repeated held-out splits + residual calibration",
            "oracle_class": "A",
            "status_category": "statistical",
            "n_splits": N_SPLITS,
            "r_squared_bootstrap_ci": ci,
            "mean_residual": mean_residual,
            "residual_prediction_correlation": correlation,
            "overall_mse": overall_mse,
            "calibration_check_gated_by_mse_floor": calibration_gated,
            "acceptance_condition": "median R^2 >= 0.9; |mean residual| < 0.1; |corr(residual,prediction)| < 0.5 (only when overall MSE >= 1e-4)",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    axes[0].plot(range(N_SPLITS), r_squared_values, "o-", color=common.COLORS["photographiqml"])
    axes[0].axhline(0.9, color=common.COLORS["acceptance"], linestyle="--", label="acceptance=0.9")
    axes[0].set(title="Held-out R^2 per split", xlabel="split", ylabel="R^2")
    axes[0].legend(fontsize=7)
    axes[1].scatter(
        all_predictions, all_residuals, s=14, color=common.COLORS["photographiqml"], alpha=0.7
    )
    axes[1].axhline(0, color=common.COLORS["acceptance"], linestyle="--")
    axes[1].set(
        title=f"Residual calibration (mean={mean_residual:.3f}, corr={correlation:.2f})",
        xlabel="prediction",
        ylabel="residual",
    )
    fig.suptitle(f"R24: regressor verification (status={status})")
    common.save_figure(fig, "R24_regressor_verification")
    plt.close(fig)

    common.print_summary(
        "R24 regressor verification",
        n_splits=N_SPLITS,
        median_r_squared=ci["point"],
        mean_residual=mean_residual,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R24 failed: median_r2={ci['point']} mean_residual={mean_residual} corr={correlation}"
        )


if __name__ == "__main__":
    main()


## 25_kernel_feature_state



In [ ]:
"""R25: Eq. 5 feature-state agreement against an independent analytic/SciPy
construction.

Scientific question: Does MuTAKernel.features's output agree with an
independent, freshly written full graph-state contraction of the same
5-angle circuit, built via kron-embedded CZ and tensor-reshape projective
measurement (the same independent-contraction method validated generally in
R5, applied here specifically to the kernel's fixed circuit and angle
formula)?

Theory/equations: MuTAKernel.features assigns
{alpha.w0.c0:x0, alpha.w1.c0:x1, alpha.w1.c1:cos(x0)cos(x1), alpha.w0.c2:x0,
alpha.w1.c2:x1} to MuTA(2,1,one_column=True) and reads the deterministic
zero-outcome logical output state.

Functionality tested: MuTAKernel.features (photographiqml.kernels) vs. an
independently coded kron/tensor-reshape contraction (not
photographiqml.logical.cz/local_gate, not validation.contract_branch).

Oracle and independence class: D (independent code path within the
package -- distinct from R11's class-B MentPy oracle for the same feature
map, giving a second, differently-sourced cross-check).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max density-matrix Frobenius error between
MuTAKernel.features's state and the independent contraction's state, over a
grid of (x0, x1) samples.

Declared acceptance condition: max error < tol
(tol = declare_tolerance(scale=1, safety_factor=200)).

Expected cost: light.

Manuscript destination: Main text (Fig. 4, companion analytic cross-check to
R11's MentPy agreement).

Scientific limitations: Fixed to the kernel's specific 2-wire circuit and
angle formula; general MuTA agreement across configurations is R5's job.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTAKernel

EXPERIMENT_ID = "R25"


def independent_contraction(model, input_state, angles):
    """Fresh kron-embedded-CZ + tensor-reshape-measurement contraction on the
    deterministic zero-outcome branch (see R5 for the general, validated
    version and its correctness argument)."""
    order = list(model.input_nodes) + [v for v in model.graph.nodes if v not in model.input_nodes]
    total = len(order)
    plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
    state = np.asarray(input_state, dtype=complex)
    for _ in range(total - model.n_wires):
        state = np.kron(state, plus)
    tensor = state.reshape([2] * total)
    for u, v in model.graph.edges:
        pu, pv = order.index(u), order.index(v)
        lo, hi = sorted((pu, pv))
        index = [slice(None)] * total
        index[lo], index[hi] = 1, 1
        tensor[tuple(index)] *= -1
    state = tensor.reshape(-1)
    remaining = list(order)
    for node in model.measurement_order:
        # Unset names default to 0, matching ParameterStore.bind's own default
        # (model.parameters() starts every stored value at 0.0).
        alpha = angles.get(model.parameter_name(node), 0.0)
        p0 = np.array([1.0, np.exp(-1j * alpha)], dtype=complex) / np.sqrt(2)
        position = remaining.index(node)
        n_now = len(remaining)
        reshaped = state.reshape(2**position, 2, 2 ** (n_now - position - 1))
        state = np.tensordot(reshaped, p0, axes=([1], [0])).reshape(-1)
        mass = float(np.vdot(state, state).real)
        state = state / np.sqrt(mass)
        remaining.remove(node)
    output_positions = [remaining.index(v) for v in model.output_nodes]
    return state.reshape([2] * model.n_wires).transpose(output_positions).reshape(-1)


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=200.0)
    kernel = MuTAKernel()
    generator = common.rng(1)
    X = generator.uniform(-2 * np.pi, 2 * np.pi, size=(10, 2))
    rows = []
    for i, (x0, x1) in enumerate(X):
        angles = {
            "alpha.w0.c0": x0,
            "alpha.w1.c0": x1,
            "alpha.w1.c1": np.cos(x0) * np.cos(x1),
            "alpha.w0.c2": x0,
            "alpha.w1.c2": x1,
        }
        pqml_state = kernel.model.run([1, 0, 0, 0], angles).state
        independent_state = independent_contraction(kernel.model, [1, 0, 0, 0], angles)
        error = common.frobenius_error(
            np.outer(pqml_state, pqml_state.conj()),
            np.outer(independent_state, independent_state.conj()),
        )
        rows.append({"sample": i, "x0": float(x0), "x1": float(x1), "density_matrix_error": error})

    max_error = max(r["density_matrix_error"] for r in rows)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R25_kernel_feature_state",
        extra={
            "protocol": "MuTAKernel.features vs. independent kron/tensor-reshape contraction",
            "oracle_class": "D",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max density-matrix error < {tol:.3e}",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "D", "status": status},
    )

    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.semilogy(
        [r["density_matrix_error"] for r in rows], "o-", color=common.COLORS["photographiqml"]
    )
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set(
        title=f"R25: kernel feature-state vs. independent contraction (status={status})",
        xlabel="sample",
        ylabel="density-matrix error",
    )
    ax.legend()
    common.save_figure(fig, "R25_kernel_feature_state")
    plt.close(fig)

    common.print_summary(
        "R25 kernel feature state", n_samples=len(X), tol=tol, max_error=max_error, status=status
    )
    if status != "pass":
        raise AssertionError(f"R25 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 26_kernel_properties



In [ ]:
"""R26: Kernel symmetry, unit diagonal, PSD spectrum, and conditioning as
sample size changes.

Scientific question: Does MuTAKernel.gram_matrix always produce a
symmetric, unit-diagonal, positive-semidefinite Gram matrix (as any valid
fidelity-based kernel k(x,y)=|<psi(x)|psi(y)>|^2 must, by construction), and
how does its numerical conditioning behave as sample size grows?

Theory/equations: for a fidelity kernel, K = Phi^dagger Phi where Phi's
columns are pure quantum feature states, so K is Hermitian (real symmetric
here since entries are |.|^2) and PSD by construction (eigenvalues are
squared singular values of Phi); K_ii = |<psi(x_i)|psi(x_i)>|^2 = 1 exactly.
These are analytic invariants of any fidelity kernel, not an empirical
property specific to this implementation.

Functionality tested: MuTAKernel.gram_matrix/diagnostics (photographiqml.kernels).

Oracle and independence class: A (independent analytic invariant -- PSD/
symmetric/unit-diagonal are theorems about fidelity kernels, checked against
the concrete numerical Gram matrix).

Exact/approximate/statistical status: exact (symmetry, diagonal), up to
floating-point roundoff for the PSD spectrum (a numerically tiny negative
eigenvalue at the roundoff floor is expected and not treated as a PSD
violation).

Primary metric: symmetry_error, diagonal_error, minimum_eigenvalue,
condition_number (max/min eigenvalue, min clipped at the roundoff floor for
the ratio only), each vs. sample size N.

Declared acceptance condition: symmetry_error and diagonal_error < tol
(tol = declare_tolerance(scale=1,safety_factor=100)); minimum_eigenvalue >
-tol for every N (allows roundoff-scale negativity, not a real PSD
violation).

Expected cost: light.

Manuscript destination: Main text (Fig. 4, kernel-properties panel).

Scientific limitations: Conditioning is reported descriptively (no claim
that any particular condition number is "good" for downstream SVM use;
that is R27/R28's job).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTAKernel

EXPERIMENT_ID = "R26"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=100.0)
    kernel = MuTAKernel()
    generator = common.rng(2)
    rows = []
    for n_samples in (2, 4, 8, 16, 32, 64):
        X = generator.uniform(-2 * np.pi, 2 * np.pi, size=(n_samples, 2))
        diagnostics = kernel.diagnostics(X)
        eigenvalues = np.linalg.eigvalsh(kernel.gram_matrix(X))
        min_eig = float(eigenvalues.min())
        max_eig = float(eigenvalues.max())
        condition_number = float(max_eig / max(min_eig, np.finfo(float).eps))
        rows.append(
            {
                "n_samples": n_samples,
                "symmetry_error": diagnostics["symmetry_error"],
                "diagonal_error": diagnostics["diagonal_error"],
                "minimum_eigenvalue": min_eig,
                "maximum_eigenvalue": max_eig,
                "condition_number": condition_number,
            }
        )

    max_symmetry_error = max(r["symmetry_error"] for r in rows)
    max_diagonal_error = max(r["diagonal_error"] for r in rows)
    min_eigenvalue_overall = min(r["minimum_eigenvalue"] for r in rows)
    status = (
        "pass"
        if (max_symmetry_error < tol and max_diagonal_error < tol and min_eigenvalue_overall > -tol)
        else "fail"
    )

    common.save_result(
        rows,
        "R26_kernel_properties",
        extra={
            "protocol": "MuTAKernel Gram-matrix symmetry/diagonal/PSD/conditioning vs. sample size",
            "oracle_class": "A",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max symmetry/diagonal error < {tol:.3e}; minimum eigenvalue > -{tol:.3e}",
            "max_symmetry_error": max_symmetry_error,
            "max_diagonal_error": max_diagonal_error,
            "min_eigenvalue_overall": min_eigenvalue_overall,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    ns = [r["n_samples"] for r in rows]
    axes[0].semilogy(
        ns,
        [max(r["symmetry_error"], 1e-18) for r in rows],
        "o-",
        label="symmetry error",
        color=common.COLORS["photographiqml"],
    )
    axes[0].semilogy(
        ns,
        [max(r["diagonal_error"], 1e-18) for r in rows],
        "s--",
        label="diagonal error",
        color=common.COLORS["mentpy"],
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle=":", label=f"tol={tol:.1e}")
    axes[0].set(title="Symmetry / unit-diagonal error", xlabel="n_samples", ylabel="error")
    axes[0].legend(fontsize=7)
    axes[1].semilogy(
        ns, [r["condition_number"] for r in rows], "o-", color=common.COLORS["photographiqml"]
    )
    axes[1].set(title="Gram-matrix condition number", xlabel="n_samples", ylabel="cond(K)")
    fig.suptitle(f"R26: kernel properties vs. sample size (status={status})")
    common.save_figure(fig, "R26_kernel_properties")
    plt.close(fig)

    common.print_summary(
        "R26 kernel properties",
        n_configs=len(rows),
        max_symmetry_error=max_symmetry_error,
        min_eigenvalue=min_eigenvalue_overall,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R26 failed: symmetry={max_symmetry_error} diagonal={max_diagonal_error} min_eig={min_eigenvalue_overall}"
        )


if __name__ == "__main__":
    main()


## 27_kernel_classification



In [ ]:
"""R27: Kernel classification on circles, moons, and blobs with predeclared
generators, repeated splits, confusion matrices, and classical baselines.

Scientific question: Does an SVM with the MuTA Eq. 5 kernel classify three
standard synthetic 2-D datasets (circles, moons, blobs) at accuracy
comparable to classical RBF-SVM and logistic-regression baselines fit on
the same raw coordinates, under repeated stratified splits?

Theory/equations: raw 2-D coordinates are interpreted directly as angles
(radians) by MuTAKernel; no standardization or clipping is applied. This is
a functionality/statistics comparison, not a claim of quantum advantage --
the classical baselines see the identical raw features.

Functionality tested: MuTAKernel + sklearn.svm.SVC(kernel="precomputed") vs.
SVC(kernel="rbf") and LogisticRegression baselines (sklearn), on
sklearn.datasets.make_circles/make_moons/make_blobs.

Oracle and independence class: N/A for the accuracy comparison itself (no
"correct" answer to check against; classical baselines are a comparison
point, not a ground truth) with a structural class-A check folded in (Gram
PSD, reused from R26's already-declared tolerance).

Exact/approximate/statistical status: statistical (8 independent stratified
splits per dataset; median accuracy and bootstrap 95% CI, paired
differences vs. each baseline with their own CIs).

Primary metric: held-out accuracy per (dataset, model, split); paired
accuracy difference (MuTA - baseline) with bootstrap 95% CI.

Declared acceptance condition: none required against a correctness oracle
(N/A); the only enforced pass/fail is that every held-out accuracy is a
finite value in [0,1] and every Gram matrix stays PSD within R26's declared
tolerance (a structural sanity floor, not a performance claim).

Expected cost: moderate (3 datasets x 8 seeds x 3 models).

Manuscript destination: Main text (Fig. 7, kernel classification).

Scientific limitations: No quantum-advantage claim; hyperparameter (C)
sensitivity is R28's job; raw coordinates as angles is a specific, arbitrary
encoding choice, not the only possible one.
"""

import sys
from pathlib import Path

import common
import numpy as np
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

from photographiqml import MuTAKernel

EXPERIMENT_ID = "R27"
SEEDS = tuple(range(8))
N_SAMPLES, SVM_C, TEST_FRACTION = 120, 1.0, 0.3
DATASETS = ("circles", "moons", "blobs")


def generate(name, seed):
    if name == "circles":
        return make_circles(n_samples=N_SAMPLES, random_state=seed, factor=0.5, noise=0.08)
    if name == "moons":
        return make_moons(n_samples=N_SAMPLES, random_state=seed, noise=0.1)
    return make_blobs(n_samples=N_SAMPLES, random_state=seed, centers=2, cluster_std=1.2)


def main():
    plt = common.setup_style()
    # Larger safety_factor than R26's since Gram matrices here are up to
    # ~84x84 (vs R26's max 64x64 but at smaller N per point); eigenvalue
    # roundoff for a PSD-by-construction matrix grows with matrix size.
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e5)
    rows = []
    confusions = {name: np.zeros((2, 2), dtype=int) for name in DATASETS}
    min_gram_eigenvalue = np.inf

    for name in DATASETS:
        for seed in SEEDS:
            X, y = generate(name, seed)
            train_x, test_x, train_y, test_y = train_test_split(
                X, y, test_size=TEST_FRACTION, stratify=y, random_state=seed
            )
            kernel = MuTAKernel()
            gram = kernel.gram_matrix(train_x)
            min_gram_eigenvalue = min(min_gram_eigenvalue, float(np.linalg.eigvalsh(gram).min()))
            muta_model = SVC(kernel="precomputed", C=SVM_C).fit(gram, train_y)
            muta_predictions = muta_model.predict(kernel(test_x, train_x))
            confusions[name] += confusion_matrix(test_y, muta_predictions, labels=[0, 1])
            accuracies = {"muta": float(accuracy_score(test_y, muta_predictions))}
            for label, baseline in [
                ("rbf_svm", SVC(C=SVM_C)),
                ("logistic", LogisticRegression(max_iter=1000, random_state=seed)),
            ]:
                baseline.fit(train_x, train_y)
                accuracies[label] = float(accuracy_score(test_y, baseline.predict(test_x)))
            rows.append(
                {
                    "dataset": name,
                    "seed": seed,
                    **{f"accuracy_{k}": v for k, v in accuracies.items()},
                }
            )

    summary = {}
    for name in DATASETS:
        subset = [r for r in rows if r["dataset"] == name]
        muta_acc = [r["accuracy_muta"] for r in subset]
        rbf_acc = [r["accuracy_rbf_svm"] for r in subset]
        log_acc = [r["accuracy_logistic"] for r in subset]
        summary[name] = {
            "muta_ci": common.bootstrap_ci(muta_acc, seed=0),
            "rbf_svm_ci": common.bootstrap_ci(rbf_acc, seed=0),
            "logistic_ci": common.bootstrap_ci(log_acc, seed=0),
            "muta_minus_rbf_ci": common.paired_difference_ci(muta_acc, rbf_acc, seed=0),
            "muta_minus_logistic_ci": common.paired_difference_ci(muta_acc, log_acc, seed=0),
        }

    all_finite = all(0 <= r[k] <= 1 for r in rows for k in r if k.startswith("accuracy_"))
    status = "pass" if (all_finite and min_gram_eigenvalue > -tol) else "fail"

    common.save_result(
        rows,
        "R27_kernel_classification",
        extra={
            "protocol": "MuTA-kernel SVM vs. RBF-SVM/logistic baselines on circles/moons/blobs, repeated splits",
            "oracle_class": "N/A",
            "status_category": "statistical",
            "seeds": list(SEEDS),
            "svm_C": SVM_C,
            "summary": summary,
            "confusion_matrices": {k: v.tolist() for k, v in confusions.items()},
            "min_gram_eigenvalue_overall": min_gram_eigenvalue,
            "acceptance_condition": "every accuracy in [0,1]; every training Gram matrix PSD within declared tolerance",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "N/A", "status": status},
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    for ax, name in zip(axes, DATASETS):
        s = summary[name]
        labels = ["MuTA", "RBF-SVM", "Logistic"]
        points = [s["muta_ci"]["point"], s["rbf_svm_ci"]["point"], s["logistic_ci"]["point"]]
        errs = [
            [
                s["muta_ci"]["point"] - s["muta_ci"]["low"],
                s["rbf_svm_ci"]["point"] - s["rbf_svm_ci"]["low"],
                s["logistic_ci"]["point"] - s["logistic_ci"]["low"],
            ],
            [
                s["muta_ci"]["high"] - s["muta_ci"]["point"],
                s["rbf_svm_ci"]["high"] - s["rbf_svm_ci"]["point"],
                s["logistic_ci"]["high"] - s["logistic_ci"]["point"],
            ],
        ]
        colors = [
            common.COLORS["photographiqml"],
            common.COLORS["classical_baseline"],
            common.COLORS["analytic"],
        ]
        ax.bar(labels, points, yerr=errs, color=colors, capsize=4)
        ax.set(title=name, ylabel="held-out accuracy", ylim=(0, 1.05))
    fig.suptitle(
        f"R27: kernel classification, median +/- 95% bootstrap CI, {len(SEEDS)} seeds (status={status})"
    )
    common.save_figure(fig, "R27_kernel_classification")
    plt.close(fig)

    common.print_summary(
        "R27 kernel classification",
        n_datasets=len(DATASETS),
        n_seeds=len(SEEDS),
        min_gram_eigenvalue=min_gram_eigenvalue,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R27 failed: all_finite={all_finite} min_gram_eigenvalue={min_gram_eigenvalue}"
        )


if __name__ == "__main__":
    main()


## 28_kernel_robustness



In [ ]:
"""R28: Kernel-SVM robustness to dataset size, noise, SVM C, and coordinate
scale, plus a leakage-safe nested hyperparameter-selection demonstration.

Scientific question: How sensitive is the MuTA-kernel SVM's held-out
accuracy on the "moons" dataset to dataset size, label noise, the SVM
regularization C, and a coordinate-scale rescaling of the raw features
(which changes the effective angle magnitude MuTAKernel encodes), and does
model selection performed strictly inside a validation fold (never touching
the test fold) avoid test-set leakage?

Theory/equations: none (empirical sensitivity study); coordinate_scale
directly rescales X before angle-encoding, so it is a genuine physical
change to the encoded angles, not a data-preprocessing artifact.

Functionality tested: MuTAKernel + sklearn.svm.SVC(kernel="precomputed") on
make_moons, swept one axis at a time from a fixed baseline
(n_samples=120, noise=0.1, C=1.0, coordinate_scale=1.0).

Oracle and independence class: N/A (descriptive sensitivity study; no
correctness oracle for "the right accuracy" at any setting).

Exact/approximate/statistical status: statistical (4 seeds per swept point
for the one-axis-at-a-time sweep; 6 independent train/val/test splits for
the nested-selection demonstration).

Primary metric: held-out test accuracy vs. each swept axis value (median
across seeds); for the nested-selection demonstration, the C chosen per
split (via validation accuracy only) and the resulting test accuracy.

Declared acceptance condition: every reported accuracy is finite and in
[0,1]; in the nested-selection demonstration, the C selected for each split
is chosen using only that split's validation fold (verified structurally:
the selection code path never reads test_x/test_y before its own accuracy
is recorded).

Expected cost: moderate (4 axes x 3 values x 4 seeds, plus 6 nested splits).

Manuscript destination: Appendix (Fig. 7 supporting robustness panel).

Scientific limitations: One dataset family (moons) and one baseline
configuration; not an exhaustive hyperparameter search, and no claim that
any chosen C is globally optimal.
"""

import sys
from pathlib import Path

import common
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

from photographiqml import MuTAKernel

EXPERIMENT_ID = "R28"
SEEDS_SWEEP = tuple(range(4))
SEEDS_NESTED = tuple(range(6))
BASELINE = {"n_samples": 120, "noise": 0.1, "C": 1.0, "coordinate_scale": 1.0}


def evaluate(n_samples, noise, C, coordinate_scale, seed):
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=seed)
    X = X * coordinate_scale
    train_x, test_x, train_y, test_y = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=seed
    )
    kernel = MuTAKernel()
    gram = kernel.gram_matrix(train_x)
    model = SVC(kernel="precomputed", C=C).fit(gram, train_y)
    predictions = model.predict(kernel(test_x, train_x))
    return float(accuracy_score(test_y, predictions))


def main():
    plt = common.setup_style()
    rows = []
    axes_sweep = {
        "n_samples": [60, 120, 240],
        "noise": [0.05, 0.15, 0.3],
        "C": [0.1, 1.0, 10.0],
        "coordinate_scale": [0.5, 1.0, 2.0],
    }
    for axis, values in axes_sweep.items():
        for value in values:
            config = dict(BASELINE)
            config[axis] = value
            accuracies = [
                evaluate(
                    config["n_samples"],
                    config["noise"],
                    config["C"],
                    config["coordinate_scale"],
                    seed,
                )
                for seed in SEEDS_SWEEP
            ]
            rows.append(
                {
                    "axis": axis,
                    "value": value,
                    "median_accuracy": float(np.median(accuracies)),
                    "accuracies": accuracies,
                }
            )

    # --- Leakage-safe nested hyperparameter selection ---------------------
    nested_rows = []
    candidate_Cs = [0.1, 1.0, 10.0]
    for seed in SEEDS_NESTED:
        X, y = make_moons(n_samples=150, noise=0.15, random_state=seed)
        train_x, rest_x, train_y, rest_y = train_test_split(
            X, y, test_size=0.4, stratify=y, random_state=seed
        )
        val_x, test_x, val_y, test_y = train_test_split(
            rest_x, rest_y, test_size=0.5, stratify=rest_y, random_state=seed
        )
        kernel = MuTAKernel()
        gram_train = kernel.gram_matrix(train_x)
        val_scores = {}
        for C in candidate_Cs:
            model = SVC(kernel="precomputed", C=C).fit(gram_train, train_y)
            val_scores[C] = float(accuracy_score(val_y, model.predict(kernel(val_x, train_x))))
        best_C = max(val_scores, key=val_scores.get)
        final_model = SVC(kernel="precomputed", C=best_C).fit(gram_train, train_y)
        test_accuracy = float(accuracy_score(test_y, final_model.predict(kernel(test_x, train_x))))
        nested_rows.append(
            {
                "seed": seed,
                "selected_C": best_C,
                "validation_scores": val_scores,
                "test_accuracy": test_accuracy,
            }
        )

    all_finite = all(0 <= r["median_accuracy"] <= 1 for r in rows) and all(
        0 <= r["test_accuracy"] <= 1 for r in nested_rows
    )
    status = "pass" if all_finite else "fail"

    common.save_result(
        rows + [{"nested_demonstration": True, **r} for r in nested_rows],
        "R28_kernel_robustness",
        extra={
            "protocol": "One-axis-at-a-time robustness sweep + leakage-safe nested C selection, moons dataset",
            "oracle_class": "N/A",
            "status_category": "statistical",
            "baseline": BASELINE,
            "nested_selection_rows": nested_rows,
            "acceptance_condition": "every reported accuracy finite and in [0,1]",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "N/A", "status": status},
    )

    fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
    for ax, (axis, values) in zip(axes[:4], axes_sweep.items()):
        medians = [
            next(r["median_accuracy"] for r in rows if r["axis"] == axis and r["value"] == v)
            for v in values
        ]
        ax.plot(values, medians, "o-", color=common.COLORS["photographiqml"])
        ax.set(title=axis, xlabel=axis, ylabel="median accuracy", ylim=(0, 1.05))
    axes[4].bar(
        [str(r["seed"]) for r in nested_rows],
        [r["test_accuracy"] for r in nested_rows],
        color=common.COLORS["photographiqml"],
    )
    axes[4].set(
        title="Nested C-selection test accuracy",
        xlabel="split seed",
        ylabel="test accuracy",
        ylim=(0, 1.05),
    )
    fig.suptitle(f"R28: kernel-SVM robustness sweep (status={status})")
    common.save_figure(fig, "R28_kernel_robustness")
    plt.close(fig)

    common.print_summary(
        "R28 kernel robustness",
        n_sweep_points=len(rows),
        n_nested_splits=len(nested_rows),
        status=status,
    )
    if status != "pass":
        raise AssertionError("R28 failed: some reported accuracy was out of [0,1] or non-finite")


if __name__ == "__main__":
    main()


## 29_instrument_diagnostics



In [ ]:
"""R29: Instrument trace preservation, branch completeness, negligible-branch
handling, and MentPy agreement.

Scientific question: Does QuantumInstrumentModel's two-branch destructive
measurement always preserve trace (branch probabilities sum to 1), report
both branches (even when one has negligible probability), correctly return
None for a state when a branch's probability is below the package's own
1e-15 negligibility threshold, and agree with MentPy (independent oracle,
reusing R12's already-validated cross-check)?

Theory/equations: for a pure global state, computational-basis measurement
of one wire is trace-preserving by construction: sum_b P(b) = 1 exactly (up
to floating point). A branch is "negligible" (state=None) iff its
probability is below 1e-15 (models.py's own convention).

Functionality tested: QuantumInstrumentModel.run (photographiqml.models),
including a deliberately engineered near-zero-probability branch (a
parameter choice driving one branch's amplitude to exactly zero).

Oracle and independence class: E (structural: trace preservation and
branch-count completeness are self-consistency invariants) plus this
experiment reuses R12's already-declared MentPy cross-check protocol on a
fresh random sample as a light B-class spot check, rather than re-deriving
the full agreement study.

Exact/approximate/statistical status: exact.

Primary metric: |sum(branch probabilities) - 1|; correctness of the
negligible-branch None convention; number of branches returned (must be
exactly 2, matching the qubit dimension of the measured wire).

Declared acceptance condition: trace-preservation error < tol
(declare_tolerance(scale=1, safety_factor=100)) for every case; exactly 2
branches returned always; state is None if and only if probability < 1e-15.

Expected cost: light.

Manuscript destination: Appendix (instrument completeness table, supporting
R12).

Scientific limitations: QuantumInstrumentModel is logical-only (see R12);
this does not test a physical quantum-output instrument (unsupported, per
models.py).
"""

import sys
from pathlib import Path


import common

from photographiqml import MuTA
from photographiqml.models import QuantumInstrumentModel, haar_states

EXPERIMENT_ID = "R29"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=100.0)
    rows = []

    # --- General random cases: trace preservation + branch completeness -------
    for n_wires in (1, 2, 3):
        model = MuTA(n_wires, 1, one_column=True)
        instrument = QuantumInstrumentModel(model, measured_wire=0)
        for seed in range(6):
            state = haar_states(n_wires, 1, seed=100 * n_wires + seed)[0]
            parameters = model.initialize(seed=200 * n_wires + seed, scale=2.0)
            branches = instrument.run(state, parameters)
            probability_sum = sum(b.probability for b in branches)
            none_consistent = all((b.state is None) == (b.probability < 1e-15) for b in branches)
            rows.append(
                {
                    "case": "random",
                    "n_wires": n_wires,
                    "seed": seed,
                    "n_branches": len(branches),
                    "probability_sum_error": abs(probability_sum - 1.0),
                    "none_convention_consistent": none_consistent,
                }
            )

    # --- Engineered near-zero-probability branch -------------------------------
    # For a 1-wire model measuring wire 0 with alpha.w0.c3=0 acting like an
    # X-basis-fixing rotation, drive input toward one computational branch.
    model = MuTA(1, 1, one_column=True)
    instrument = QuantumInstrumentModel(model, measured_wire=0)
    # All-zero angles => model.unitary() == I (per test_topology), so a
    # computational basis input gives one branch exactly probability 1.
    branches = instrument.run([1, 0], {})
    probability_sum = sum(b.probability for b in branches)
    zero_branch_is_none = branches[1].state is None and branches[1].probability < 1e-15
    one_branch_state_matches_input = (
        branches[0].state is not None and common.frobenius_error(branches[0].state, [1.0]) < tol
    )
    rows.append(
        {
            "case": "engineered_zero_branch",
            "n_wires": 1,
            "seed": None,
            "n_branches": len(branches),
            "probability_sum_error": abs(probability_sum - 1.0),
            "none_convention_consistent": zero_branch_is_none and one_branch_state_matches_input,
        }
    )

    max_prob_error = max(r["probability_sum_error"] for r in rows)
    all_branches_2 = all(r["n_branches"] == 2 for r in rows)
    all_none_consistent = all(r["none_convention_consistent"] for r in rows)
    status = "pass" if (max_prob_error < tol and all_branches_2 and all_none_consistent) else "fail"

    common.save_result(
        rows,
        "R29_instrument_diagnostics",
        extra={
            "protocol": "QuantumInstrumentModel trace preservation, branch completeness, negligible-branch convention",
            "oracle_class": "E",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max probability-sum error < {tol:.3e}; always 2 branches; None iff probability<1e-15",
            "max_probability_sum_error": max_prob_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(7, 3.8))
    colors = [
        common.COLORS["photographiqml"]
        if r["none_convention_consistent"]
        else common.COLORS["piquasso"]
        for r in rows
    ]
    ax.scatter(range(len(rows)), [max(r["probability_sum_error"], 1e-18) for r in rows], c=colors)
    ax.set_yscale("log")
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set(
        title=f"R29: instrument trace-preservation error (status={status})",
        xlabel="case",
        ylabel="|sum(P)-1|",
    )
    ax.legend()
    common.save_figure(fig, "R29_instrument_diagnostics")
    plt.close(fig)

    common.print_summary(
        "R29 instrument diagnostics",
        n_cases=len(rows),
        max_probability_error=max_prob_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R29 failed: max_prob_error={max_prob_error} branches_ok={all_branches_2} none_ok={all_none_consistent}"
        )


if __name__ == "__main__":
    main()


## 30_concurrence



In [ ]:
"""R30: Two-qubit concurrence against the closed-form Wootters spin-flip
formula.

Scientific question: Does photographiqml.diagnostics.concurrence (which
computes 2|ad-bc| directly from state amplitudes (a,b,c,d)) agree with the
independently coded Wootters spin-flip formula
C = |<psi| (sigma_y (x) sigma_y) |psi*>|, over random states and the known
separable/maximally-entangled boundary cases?

Theory/equations: for a pure two-qubit state, both formulas are proven
equal in closed form; this experiment evaluates them via two different
numerical routes (direct amplitude combination vs. an explicit sigma_y⊗sigma_y
matrix contraction) as a cross-check, not a re-derivation of the theorem.

Functionality tested: photographiqml.diagnostics.concurrence.

Oracle and independence class: A (independent analytic oracle -- the
spin-flip formula is evaluated via fresh matrix construction, not by
reusing the 2|ad-bc| shortcut).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: |concurrence - spin_flip_concurrence| over random states;
absolute values at known separable (concurrence=0) and Bell
(concurrence=1) states.

Declared acceptance condition: max error < tol
(tol = declare_tolerance(scale=1, safety_factor=100)); separable-state
concurrence < tol; Bell-state concurrence within tol of 1.

Expected cost: light.

Manuscript destination: Appendix (diagnostics validation table).

Scientific limitations: Pure-state two-qubit concurrence only (matches the
package's own documented scope; not a general entanglement measure).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml.diagnostics import concurrence
from photographiqml.models import haar_states

EXPERIMENT_ID = "R30"

SIGMA_Y = np.array([[0, -1j], [1j, 0]])
YY = np.kron(SIGMA_Y, SIGMA_Y)


def spin_flip_concurrence(state):
    state = np.asarray(state, dtype=complex)
    return float(abs(state.conj() @ YY @ state.conj()))


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=100.0)
    rows = []
    states = haar_states(2, 40, seed=5)
    for i, state in enumerate(states):
        production = concurrence(state)
        independent = spin_flip_concurrence(state)
        rows.append(
            {
                "case": f"random_{i}",
                "production": production,
                "independent": independent,
                "error": abs(production - independent),
            }
        )

    separable = np.kron([1, 0], [1, 0])
    bell = np.array([1, 0, 0, 1]) / np.sqrt(2)
    for name, state in (("separable_00", separable), ("bell_phi_plus", bell)):
        production = concurrence(state)
        independent = spin_flip_concurrence(state)
        rows.append(
            {
                "case": name,
                "production": production,
                "independent": independent,
                "error": abs(production - independent),
            }
        )

    max_error = max(r["error"] for r in rows)
    separable_value = next(r["production"] for r in rows if r["case"] == "separable_00")
    bell_value = next(r["production"] for r in rows if r["case"] == "bell_phi_plus")
    status = (
        "pass"
        if (max_error < tol and separable_value < tol and abs(bell_value - 1) < tol)
        else "fail"
    )

    common.save_result(
        rows,
        "R30_concurrence",
        extra={
            "protocol": "diagnostics.concurrence vs. independent Wootters spin-flip formula",
            "oracle_class": "A",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max error < {tol:.3e}; separable concurrence < tol; Bell concurrence within tol of 1",
            "max_error": max_error,
            "separable_value": separable_value,
            "bell_value": bell_value,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].semilogy(
        [r["error"] for r in rows if r["case"].startswith("random")],
        "o",
        color=common.COLORS["photographiqml"],
    )
    axes[0].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[0].set(
        title="Random-state agreement", xlabel="sample", ylabel="|production - independent|"
    )
    axes[0].legend()
    axes[1].bar(
        ["separable", "Bell"], [separable_value, bell_value], color=common.COLORS["photographiqml"]
    )
    axes[1].axhline(0, color=common.COLORS["acceptance"], linewidth=0.8)
    axes[1].axhline(1, color=common.COLORS["acceptance"], linewidth=0.8)
    axes[1].set(title="Known boundary cases", ylabel="concurrence")
    fig.suptitle(f"R30: concurrence vs. spin-flip formula (status={status})")
    common.save_figure(fig, "R30_concurrence")
    plt.close(fig)

    common.print_summary("R30 concurrence", n_cases=len(rows), max_error=max_error, status=status)
    if status != "pass":
        raise AssertionError(
            f"R30 failed: max_error={max_error} separable={separable_value} bell={bell_value}"
        )


if __name__ == "__main__":
    main()


## 31_qfi



In [ ]:
"""R31: Pure-state QFI against 4*Var(H), evaluated via an independent
density-matrix trace formula.

Scientific question: Does photographiqml.diagnostics.pure_qfi (computed via
a direct state/image inner-product route) agree with the standard
4*Var(H) = 4*(Tr(rho H^2) - Tr(rho H)^2) formula evaluated via an
independently coded density-matrix trace route, for random states and
generators?

Theory/equations: for a pure state, the quantum Fisher information for
generator H is F = 4*Var(H); pure_qfi's own docstring already states this
is the formula it implements via image=H@state and
4*(<image|image> - <state|image>^2). This experiment evaluates the
identical quantity via rho=|psi><psi|, Tr(rho H), Tr(rho H^2) instead, a
different (if mathematically equivalent) computational route.

Functionality tested: photographiqml.diagnostics.pure_qfi.

Oracle and independence class: A (independent analytic route -- density-
matrix trace formula, not the state/image inner-product shortcut).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: |pure_qfi - 4*Var(H)_via_trace| over random states and
random Hermitian generators, at 1, 2 and 3 qubits.

Declared acceptance condition: max error < tol
(tol = declare_tolerance(scale=4, safety_factor=100), scale=4 since the QFI
formula carries an overall factor of 4).

Expected cost: light.

Manuscript destination: Appendix (diagnostics validation table, companion
to R30).

Scientific limitations: Pure states only (matches pure_qfi's documented
scope).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml.diagnostics import pure_qfi
from photographiqml.models import haar_states

EXPERIMENT_ID = "R31"


def trace_qfi(state, generator):
    rho = np.outer(state, state.conj())
    mean_h = np.trace(rho @ generator).real
    mean_h2 = np.trace(rho @ generator @ generator).real
    return float(4 * (mean_h2 - mean_h**2))


def random_hermitian(n, seed):
    generator = np.random.default_rng(seed)
    m = generator.normal(size=(n, n)) + 1j * generator.normal(size=(n, n))
    return m + m.conj().T


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=4.0, safety_factor=100.0)
    rows = []
    for n_qubits in (1, 2, 3):
        dim = 2**n_qubits
        states = haar_states(n_qubits, 10, seed=n_qubits)
        for i, state in enumerate(states):
            observable = random_hermitian(dim, seed=100 * n_qubits + i)
            production = pure_qfi(state, observable)
            independent = trace_qfi(state, observable)
            rows.append(
                {
                    "n_qubits": n_qubits,
                    "sample": i,
                    "production": production,
                    "independent": independent,
                    "error": abs(production - independent),
                }
            )

    max_error = max(r["error"] for r in rows)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R31_qfi",
        extra={
            "protocol": "diagnostics.pure_qfi vs. independent density-matrix trace formula for 4*Var(H)",
            "oracle_class": "A",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max error < {tol:.3e}",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    for n_qubits in (1, 2, 3):
        errs = [r["error"] for r in rows if r["n_qubits"] == n_qubits]
        ax.semilogy(errs, "o-", alpha=0.8, label=f"n_qubits={n_qubits}")
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set(
        title=f"R31: pure-state QFI vs. trace formula (status={status})",
        xlabel="sample",
        ylabel="|production - independent|",
    )
    ax.legend(fontsize=7)
    common.save_figure(fig, "R31_qfi")
    plt.close(fig)

    common.print_summary("R31 QFI", n_cases=len(rows), max_error=max_error, status=status)
    if status != "pass":
        raise AssertionError(f"R31 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 32_lie_closure



In [ ]:
"""R32: Exact Pauli Lie-closure dimension for known commuting, partially
generating, and full-algebra examples.

Scientific question: Does photographiqml.expressivity.pauli_lie_dimension's
exact binary (GF(2) symplectic bit-trick) closure algorithm agree with an
independently coded, purely linear-algebraic closure -- iterated matrix
commutators [A,B] on explicit 2**n x 2**n Pauli-string matrices, with new
basis elements admitted via a numerical rank test -- for known commuting,
partially-generating and full-su(2)/su(4) example generator sets?

Theory/equations: the real Lie algebra generated by {iP} for Pauli strings
P closes under commutation; anticommuting P,Q give [P,Q] proportional to a
third Pauli string (up to a nonzero scalar), while commuting generators
never grow the algebra. Known closed-form cases: a single generator (or any
set of pairwise-commuting generators) has closure dimension = the number of
distinct generators; {X,Z} on one qubit closes to the full 3-dimensional
su(2) (X,Y,Z); two independent single-qubit su(2) factors on 2 qubits close
to the full 15-dimensional su(4) once enough cross terms are included.

Functionality tested: photographiqml.expressivity.pauli_lie_dimension.

Oracle and independence class: A (independent analytic oracle -- a fresh
matrix-commutator/rank-based closure algorithm, not the production binary
symplectic bit trick).

Exact/approximate/statistical status: exact (integer dimension equality).

Primary metric: |production_dimension - independent_dimension| per case.

Declared acceptance condition: 0 for every declared case.

Expected cost: light (n_qubits <= 3, so matrices at most 8x8).

Manuscript destination: Appendix (expressivity/diagnostics validation
table, Table IV).

Scientific limitations: Lie closure describes arbitrary-depth algebraic
reachability, not finite-depth circuit expressivity (see docstring of
pauli_lie_dimension itself, and R33 for the complementary local-Fisher-rank
diagnostic, which is also not a finite-depth expressivity proof).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml.expressivity import pauli_lie_dimension

EXPERIMENT_ID = "R32"

PAULI = {
    "I": np.eye(2, dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.diag([1, -1]).astype(complex),
}


def pauli_matrix(string):
    matrix = np.array([[1]], dtype=complex)
    for letter in string:
        matrix = np.kron(matrix, PAULI[letter])
    return matrix


def matrix_lie_dimension(strings, rank_tol=1e-9):
    """Independent closure: iterated matrix commutators + numerical rank."""

    def flatten_real(matrix):
        return np.concatenate([matrix.real.ravel(), matrix.imag.ravel()])

    # Closure via iterated matrix commutators: keep the actual Hermitian
    # matrices and test independence by numerical rank of their stacked
    # (real, flattened) representations.
    matrices = [pauli_matrix(s) for s in strings]
    changed = True
    while changed:
        changed = False
        stacked = np.array([flatten_real(m) for m in matrices])
        rank_before = np.linalg.matrix_rank(stacked, tol=rank_tol) if len(stacked) else 0
        new_matrices = []
        for i in range(len(matrices)):
            for j in range(i + 1, len(matrices)):
                commutator = matrices[i] @ matrices[j] - matrices[j] @ matrices[i]
                generator_matrix = 1j * commutator  # Hermitian if [A,B] anti-Hermitian
                if np.linalg.norm(generator_matrix) > rank_tol:
                    trial = matrices + new_matrices + [generator_matrix]
                    trial_stacked = np.array([flatten_real(m) for m in trial])
                    if np.linalg.matrix_rank(trial_stacked, tol=rank_tol) > rank_before + len(
                        new_matrices
                    ):
                        new_matrices.append(generator_matrix)
        if new_matrices:
            matrices.extend(new_matrices)
            changed = True
    final_stacked = np.array([flatten_real(m) for m in matrices])
    return int(np.linalg.matrix_rank(final_stacked, tol=rank_tol))


def main():
    plt = common.setup_style()
    cases = [
        ("single_generator", ["X"]),
        ("commuting_pair", ["ZI", "IZ"]),
        ("anticommuting_pair_su2", ["X", "Z"]),
        ("two_qubit_partial", ["XI", "IX"]),
        ("two_qubit_full_su4_seed", ["XI", "ZI", "IX", "IZ"]),
        ("three_qubit_single", ["XII"]),
    ]
    rows = []
    for name, generators in cases:
        production = pauli_lie_dimension(generators)
        independent = matrix_lie_dimension(generators)
        rows.append(
            {
                "case": name,
                "generators": ",".join(generators),
                "production_dimension": production,
                "independent_dimension": independent,
                "error": abs(production - independent),
            }
        )

    max_error = max(r["error"] for r in rows)
    status = "pass" if max_error == 0 else "fail"

    common.save_result(
        rows,
        "R32_lie_closure",
        extra={
            "protocol": "pauli_lie_dimension vs. independent matrix-commutator/rank closure",
            "oracle_class": "A",
            "status_category": "exact",
            "acceptance_condition": "|production - independent| == 0 for every declared case",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(rows))
    ax.bar(
        [i - 0.15 for i in x],
        [r["production_dimension"] for r in rows],
        width=0.3,
        label="production",
        color=common.COLORS["photographiqml"],
    )
    ax.bar(
        [i + 0.15 for i in x],
        [r["independent_dimension"] for r in rows],
        width=0.3,
        label="independent (matrix)",
        color=common.COLORS["analytic"],
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels([r["case"] for r in rows], rotation=30, ha="right", fontsize=7)
    ax.set(
        title=f"R32: Lie-closure dimension agreement (status={status})", ylabel="closure dimension"
    )
    ax.legend()
    common.save_figure(fig, "R32_lie_closure")
    plt.close(fig)

    common.print_summary("R32 Lie closure", n_cases=len(rows), max_error=max_error, status=status)
    if status != "pass":
        raise AssertionError(f"R32 failed cases: {[r['case'] for r in rows if r['error'] != 0]}")


if __name__ == "__main__":
    main()


## 33_fisher_spectra



In [ ]:
"""R33: Local state-Fisher spectra, rank, condition number, and
finite-difference step convergence versus wires and depth.

Scientific question: Is photographiqml.expressivity.state_fisher's local
pure-state QFI matrix positive-semidefinite (as any Fisher information
matrix must be, an analytic invariant), does its central-difference
construction converge as the step size shrinks toward a step-independent
limit, and how do its rank and condition number scale with wire count and
circuit depth?

Theory/equations: state_fisher's docstring states this is a *local*
diagnostic (rank is not a statistical effective dimension or finite-depth
inclusion proof, per docs). This experiment checks the analytic invariant
(PSD) and the numerical-convergence invariant (step-size stability), both
properties any correct implementation must satisfy, independent of any
external oracle.

Functionality tested: photographiqml.expressivity.state_fisher.

Oracle and independence class: A for the PSD check (an analytic invariant
of any Fisher information matrix); E for the step-convergence check
(self-consistency against a much smaller reference step).

Exact/approximate/statistical status: exact for PSD (up to roundoff);
approximate for step convergence (central-difference truncation error).

Primary metric: minimum eigenvalue (PSD check); ||F(step) - F(step_ref)||
vs. step (convergence check); rank (numerical, tol=1e-8) and condition
number vs. (n_wires, n_layers).

Declared acceptance condition: minimum eigenvalue > -tol
(tol = declare_tolerance(scale=4, safety_factor=1e4)); convergence error
strictly decreases as step shrinks from 1e-2 to 1e-5 (before the expected
central-difference roundoff floor at very small steps).

Expected cost: light.

Manuscript destination: Appendix (expressivity diagnostics table, Fig. 8).

Scientific limitations: state_fisher's rank is explicitly a *local*
diagnostic at one parameter point; it is not evidence of finite-depth
expressivity or a global statement about the parameter landscape (see
docstring and R32's Lie-closure caveat).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.expressivity import state_fisher

EXPERIMENT_ID = "R33"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=4.0, safety_factor=1e4)
    rows_psd = []
    rows_scaling = []
    configs = [(1, 1), (2, 1), (2, 2), (3, 1)]
    for n_wires, n_layers in configs:
        model = MuTA(n_wires, n_layers, one_column=True)
        input_state = np.eye(2**n_wires, dtype=complex)[0]
        parameters = common.rng(n_wires * 10 + n_layers).uniform(-np.pi, np.pi, model.n_parameters)
        fisher = state_fisher(model, input_state, parameters, step=1e-5)
        eigenvalues = np.linalg.eigvalsh(fisher)
        rank = int(np.sum(eigenvalues > 1e-8 * max(eigenvalues.max(), 1e-30)))
        condition_number = (
            float(
                eigenvalues.max()
                / max(
                    eigenvalues.min(where=eigenvalues > 1e-12, initial=np.inf), np.finfo(float).eps
                )
            )
            if rank
            else float("nan")
        )
        rows_psd.append(
            {"n_wires": n_wires, "n_layers": n_layers, "min_eigenvalue": float(eigenvalues.min())}
        )
        rows_scaling.append(
            {
                "n_wires": n_wires,
                "n_layers": n_layers,
                "n_parameters": model.n_parameters,
                "rank": rank,
                "condition_number": condition_number,
            }
        )

    # --- step convergence, fixed config ------------------------------------
    model = MuTA(2, 1, one_column=True)
    input_state = np.eye(4, dtype=complex)[0]
    parameters = common.rng(0).uniform(-np.pi, np.pi, model.n_parameters)
    reference = state_fisher(model, input_state, parameters, step=1e-6)
    steps = [1e-2, 1e-3, 1e-4, 1e-5]
    convergence_rows = []
    for step in steps:
        fisher = state_fisher(model, input_state, parameters, step=step)
        convergence_rows.append(
            {"step": step, "error_vs_reference": common.frobenius_error(fisher, reference)}
        )

    min_eig_overall = min(r["min_eigenvalue"] for r in rows_psd)
    errors = [r["error_vs_reference"] for r in convergence_rows]
    monotonic_decrease = all(errors[i] >= errors[i + 1] for i in range(len(errors) - 1))
    status = "pass" if (min_eig_overall > -tol and monotonic_decrease) else "fail"

    common.save_result(
        rows_psd
        + [{"step_convergence": True, **r} for r in convergence_rows]
        + [{"scaling": True, **r} for r in rows_scaling],
        "R33_fisher_spectra",
        extra={
            "protocol": "state_fisher PSD check and step-convergence, plus rank/conditioning vs wires/depth",
            "oracle_class": "A/E",
            "status_category": "exact/approximate",
            "tolerance": tol,
            "step_convergence_rows": convergence_rows,
            "scaling_rows": rows_scaling,
            "acceptance_condition": f"min eigenvalue > -{tol:.3e}; convergence error monotonically decreases from step=1e-2 to 1e-5",
            "min_eigenvalue_overall": min_eig_overall,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A/E", "status": status},
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    axes[0].bar(
        range(len(rows_psd)),
        [r["min_eigenvalue"] for r in rows_psd],
        color=common.COLORS["photographiqml"],
    )
    axes[0].axhline(0, color=common.COLORS["acceptance"], linewidth=0.8)
    axes[0].set_xticks(range(len(rows_psd)))
    axes[0].set_xticklabels([f"n{r['n_wires']}L{r['n_layers']}" for r in rows_psd], fontsize=7)
    axes[0].set(title="Min eigenvalue (PSD check)", ylabel="eigenvalue")
    axes[1].loglog(steps, errors, "o-", color=common.COLORS["photographiqml"])
    axes[1].set(title="Step convergence", xlabel="step", ylabel="||F(step)-F(ref)||")
    axes[2].bar(
        range(len(rows_scaling)),
        [r["rank"] for r in rows_scaling],
        color=common.COLORS["photographiqml"],
        alpha=0.7,
        label="rank",
    )
    axes[2].plot(
        range(len(rows_scaling)),
        [r["n_parameters"] for r in rows_scaling],
        "s--",
        color=common.COLORS["analytic"],
        label="n_parameters",
    )
    axes[2].set_xticks(range(len(rows_scaling)))
    axes[2].set_xticklabels([f"n{r['n_wires']}L{r['n_layers']}" for r in rows_scaling], fontsize=7)
    axes[2].set(title="Local Fisher rank vs. n_parameters", ylabel="count")
    axes[2].legend(fontsize=7)
    fig.suptitle(f"R33: local state-Fisher spectra (status={status})")
    common.save_figure(fig, "R33_fisher_spectra")
    plt.close(fig)

    common.print_summary(
        "R33 Fisher spectra",
        n_configs=len(rows_psd),
        min_eigenvalue=min_eig_overall,
        monotonic_decrease=monotonic_decrease,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R33 failed: min_eig={min_eig_overall} monotonic={monotonic_decrease}"
        )


if __name__ == "__main__":
    main()


## 34_gkp_codeword_projection



In [ ]:
"""R34: Finite-GKP codeword projection, captured weight, codeword overlap,
Gram eigenvalues, and cutoff/grid refinement.

Scientific question: For GKPBridge's finite-energy codeword resource, how do
captured projection weight, |0>/|1> codeword overlap, and Gram-matrix
eigenvalues behave as cutoff and grid_points are independently refined, and
does the Gram matrix always stay Hermitian and positive-semidefinite (an
algebraic property any Gram matrix of vectors must satisfy, checked here as
an independent analytic invariant)?

Theory/equations: docs/research/muta-mapping.md: finite codeword Gram
matrices need not be identity (codewords overlap); this experiment does not
assert orthogonality, only the weaker, always-true PSD/Hermitian property,
plus the expected qualitative trend of increasing captured weight with
cutoff (a finite-energy truncation effect, not a claim of exact convergence
to unit weight).

Functionality tested: GKPBridge.diagnostics (photographiqml.gkp), via
PhotoGraphiQ's GKPCode.resource/project.

Oracle and independence class: A for the Hermitian/PSD Gram-matrix check
(an algebraic invariant of any Gram matrix); E for the monotonic
cutoff-refinement trend (self-consistency across a resolution sweep, not an
external ground truth).

Exact/approximate/statistical status: exact for the Hermitian/PSD check;
descriptive/statistical for the refinement trend (no claimed convergence
rate).

Primary metric: Hermitian/PSD violation (must be ~0); captured weight and
codeword overlap vs. cutoff and grid_points.

Declared acceptance condition: Gram matrix Hermitian within tol
(declare_tolerance(scale=1,safety_factor=1e3)) and PSD within the same tol
for every swept cutoff/grid_points value; captured weight stays in [0,1]
(a probability, by construction) for every point.

Expected cost: light-to-moderate (GKP projection at several cutoffs).

Manuscript destination: Main text (Fig. 8, GKP resource characterization).
Explicitly makes NO fault-tolerance or gate-validation claim
(diagnostics()["physical_muta_validated"] is always False by construction).

Scientific limitations: Resource-only diagnostics; this does not by itself
establish a physical implementation of arbitrary logical MuTA (see R38-R42
for the restricted physical execution layer).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import GKPBridge

EXPERIMENT_ID = "R34"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e3)
    rows = []
    # GKPBridge's own class defaults (peak_width=0.4, envelope=0.4, peaks=8,
    # grid_points=4097) are a known-resolved resource; only the swept axis
    # is varied from them.
    baseline = {"peak_width": 0.4, "envelope": 0.4, "peaks": 8, "grid_points": 4097}

    for cutoff in (12, 20, 32, 48):
        bridge = GKPBridge(cutoff=cutoff, **baseline)
        diagnostics = bridge.diagnostics()
        gram = np.array(diagnostics["gram_eigenvalues"])
        rows.append(
            {
                "axis": "cutoff",
                "value": cutoff,
                "captured_weight_0": diagnostics["captured_weights"][0],
                "captured_weight_1": diagnostics["captured_weights"][1],
                "codeword_overlap": diagnostics["codeword_overlap"],
                "min_gram_eigenvalue": float(gram.min()),
                "max_gram_eigenvalue": float(gram.max()),
                "physical_muta_validated": diagnostics["physical_muta_validated"],
            }
        )

    for grid_points in (2049, 4097, 8193):
        bridge = GKPBridge(
            cutoff=32, peak_width=0.4, envelope=0.4, peaks=8, grid_points=grid_points
        )
        diagnostics = bridge.diagnostics()
        gram = np.array(diagnostics["gram_eigenvalues"])
        rows.append(
            {
                "axis": "grid_points",
                "value": grid_points,
                "captured_weight_0": diagnostics["captured_weights"][0],
                "captured_weight_1": diagnostics["captured_weights"][1],
                "codeword_overlap": diagnostics["codeword_overlap"],
                "min_gram_eigenvalue": float(gram.min()),
                "max_gram_eigenvalue": float(gram.max()),
                "physical_muta_validated": diagnostics["physical_muta_validated"],
            }
        )

    all_weights_valid = all(
        0 - tol <= r["captured_weight_0"] <= 1 + tol
        and 0 - tol <= r["captured_weight_1"] <= 1 + tol
        for r in rows
    )
    all_psd = all(r["min_gram_eigenvalue"] > -tol for r in rows)
    never_validated = all(r["physical_muta_validated"] is False for r in rows)
    status = "pass" if (all_weights_valid and all_psd and never_validated) else "fail"

    common.save_result(
        rows,
        "R34_gkp_codeword_projection",
        extra={
            "protocol": "GKPBridge.diagnostics vs. cutoff/grid_points refinement, Gram PSD invariant",
            "oracle_class": "A/E",
            "status_category": "exact/statistical",
            "tolerance": tol,
            "acceptance_condition": "captured weights in [0,1]; Gram matrix PSD; physical_muta_validated always False",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A/E", "status": status},
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    cutoff_rows = [r for r in rows if r["axis"] == "cutoff"]
    grid_rows = [r for r in rows if r["axis"] == "grid_points"]
    axes[0].plot(
        [r["value"] for r in cutoff_rows],
        [r["captured_weight_0"] for r in cutoff_rows],
        "o-",
        color=common.COLORS["photographiqml"],
    )
    axes[0].set(
        title="Captured weight vs. cutoff", xlabel="cutoff", ylabel="captured weight (bit=0)"
    )
    # cutoff (~12-48) and grid_points (~2049-8193) live on incompatible
    # x-scales; a shared linear x-axis collapses the cutoff series into an
    # invisible sliver, so grid_points gets its own twinned x-axis.
    axes1_grid = axes[1].twiny()
    axes[1].plot(
        [r["value"] for r in cutoff_rows],
        [r["codeword_overlap"] for r in cutoff_rows],
        "o-",
        color=common.COLORS["photographiqml"],
        label="vs cutoff",
    )
    axes1_grid.plot(
        [r["value"] for r in grid_rows],
        [r["codeword_overlap"] for r in grid_rows],
        "s--",
        color=common.COLORS["mentpy"],
        label="vs grid_points",
    )
    axes[1].set(title="Codeword overlap |<0|1>|", xlabel="cutoff", ylabel="overlap")
    axes[1].xaxis.label.set_color(common.COLORS["photographiqml"])
    axes1_grid.set_xlabel("grid_points", color=common.COLORS["mentpy"])
    axes1_grid.tick_params(axis="x", colors=common.COLORS["mentpy"], labelsize=7)
    axes[1].tick_params(axis="x", colors=common.COLORS["photographiqml"])
    handles = axes[1].get_lines() + axes1_grid.get_lines()
    axes[1].legend(handles, [h.get_label() for h in handles], fontsize=7)
    axes[2].semilogy(
        [r["value"] for r in cutoff_rows],
        [max(-r["min_gram_eigenvalue"], 1e-18) for r in cutoff_rows],
        "o-",
        color=common.COLORS["photographiqml"],
    )
    axes[2].axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    axes[2].set(
        title="Gram PSD violation vs. cutoff", xlabel="cutoff", ylabel="max(0,-min eigenvalue)"
    )
    axes[2].legend(fontsize=7)
    fig.suptitle(
        f"R34: GKP codeword projection diagnostics (status={status}, no fault-tolerance claim)"
    )
    common.save_figure(fig, "R34_gkp_codeword_projection")
    plt.close(fig)

    common.print_summary(
        "R34 GKP codeword projection",
        n_points=len(rows),
        all_weights_valid=all_weights_valid,
        all_psd=all_psd,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R34 failed: weights_valid={all_weights_valid} psd={all_psd} never_validated={never_validated}"
        )


if __name__ == "__main__":
    main()


## 35_capability_map



In [ ]:
"""R35: Allocation-free physical capability map over angle values, including
the exact 0/pi support boundary and upstream tolerance.

Scientific question: Does MuTA.physical_capabilities correctly classify
every intermediate measurement angle as supported (only 0 or pi modulo 2pi,
at PhotoGraphiQ's absolute tolerance 1e-14) or unsupported, entirely without
any Fock allocation, and does the boundary fall exactly at that documented
tolerance (docs/physical/supported-measurements.md)?

Theory/equations: supported angles are {0, pi} mod 2pi at absolute tolerance
1e-14 (upstream, not re-implemented by PhotoGraphiQML); Y (pi/2), pi/4, and
arbitrary XY are rejected.

Functionality tested: photographiqml.lowering.physical_capabilities /
MuTA.physical_capabilities -- audit only, verified never to allocate Fock
resources (checked by confirming the audit completes even for angles that
would be numerically infeasible to simulate, and by timing: the audit must
be far faster than an actual lowering+simulation).

Oracle and independence class: E (structural self-consistency against the
documented tolerance boundary, which is an upstream PhotoGraphiQ contract,
not independently re-derived here).

Exact/approximate/statistical status: exact (discrete supported/unsupported
classification at declared angle offsets from the tolerance boundary).

Primary metric: classification correctness at angle offsets
{-1e-13, -1e-15, 0, +1e-15, +1e-13} from 0 and from pi (five points that
straddle the documented 1e-14 boundary on each side).

Declared acceptance condition: angles within 1e-14 of {0, pi} are
classified supported; angles at 1e-13 offset are classified unsupported;
Y (pi/2), pi/4 and a generic irrational-multiple-of-pi angle are always
unsupported.

Expected cost: light (no Fock allocation).

Manuscript destination: Main text (Fig. 9, capability-boundary panel).

Scientific limitations: The exact tolerance value (1e-14) is PhotoGraphiQ's
own contract; this experiment verifies PhotoGraphiQML correctly surfaces it,
not that 1e-14 is itself derived from first principles here.
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA

EXPERIMENT_ID = "R35"
UPSTREAM_TOLERANCE = 1e-14


def main():
    plt = common.setup_style()
    model = MuTA(1, 1, one_column=True)
    rows = []

    boundary_offsets = [-1e-13, -1e-15, 0.0, 1e-15, 1e-13]
    for center_name, center in (("zero", 0.0), ("pi", np.pi)):
        for offset in boundary_offsets:
            angle = center + offset
            audit = model.physical_capabilities({"alpha.w0.c1": angle})
            expected_supported = abs(offset) <= UPSTREAM_TOLERANCE
            node_row = next(r for r in audit["angle_audit"] if r["node"] == (0, 1))
            rows.append(
                {
                    "center": center_name,
                    "offset": offset,
                    "angle": angle,
                    "expected_supported": expected_supported,
                    "actual_supported": node_row["supported"],
                    "correct": node_row["supported"] == expected_supported,
                }
            )

    for name, angle in (
        ("Y_pi_over_2", np.pi / 2),
        ("pi_over_4", np.pi / 4),
        ("irrational_multiple", 1.23456789),
        ("nan", float("nan")),
        ("inf", float("inf")),
    ):
        audit = model.physical_capabilities({"alpha.w0.c1": angle}) if np.isfinite(angle) else None
        if audit is None:
            # NaN/inf are rejected before angle auditing at run() level; here
            # confirm the capability audit itself does not silently accept them.
            try:
                model.physical_capabilities({"alpha.w0.c1": angle})
                accepted = True
            except ValueError:
                accepted = False
            rows.append(
                {
                    "center": name,
                    "offset": None,
                    "angle": angle,
                    "expected_supported": False,
                    "actual_supported": False,
                    "correct": not accepted,
                }
            )
            continue
        node_row = next(r for r in audit["angle_audit"] if r["node"] == (0, 1))
        rows.append(
            {
                "center": name,
                "offset": None,
                "angle": angle,
                "expected_supported": False,
                "actual_supported": node_row["supported"],
                "correct": node_row["supported"] is False,
            }
        )

    n_incorrect = sum(1 for r in rows if not r["correct"])
    status = "pass" if n_incorrect == 0 else "fail"

    common.save_result(
        rows,
        "R35_capability_map",
        extra={
            "protocol": "physical_capabilities classification at and around the 0/pi upstream tolerance boundary",
            "oracle_class": "E",
            "status_category": "exact",
            "upstream_tolerance": UPSTREAM_TOLERANCE,
            "acceptance_condition": "classification matches documented 1e-14 boundary for every declared case",
            "n_incorrect": n_incorrect,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [
        f"{r['center']}\n{r['offset']}" if r["offset"] is not None else r["center"] for r in rows
    ]
    colors = [
        common.COLORS["photographiqml"] if r["actual_supported"] else common.COLORS["unsupported"]
        for r in rows
    ]
    edge = ["none" if r["correct"] else "red" for r in rows]
    ax.bar(range(len(rows)), [1] * len(rows), color=colors, edgecolor=edge, linewidth=2)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=6)
    ax.set_title(
        f"R35: capability boundary map (green=supported, gray=unsupported, red edge=wrong) (status={status})"
    )
    common.save_figure(fig, "R35_capability_map")
    plt.close(fig)

    common.print_summary(
        "R35 capability map", n_cases=len(rows), n_incorrect=n_incorrect, status=status
    )
    if status != "pass":
        raise AssertionError(f"R35 failed cases: {[r['center'] for r in rows if not r['correct']]}")


if __name__ == "__main__":
    main()


## 36_lowering_preservation



In [ ]:
"""R36: Lowering preservation of logical vertices, graph edges, I/O mapping,
measurement keys, schedule and the analytic Fock dimension.

Scientific question: Does lower_muta_to_gkp preserve MuTA's logical graph
exactly (identity node map, same input/output node sets, one measurement
key per measured node, one frame dependency entry per node consistent with
model.corrections), does the resulting Pattern validate, and does the
reported Hilbert dimension match the analytic total-photon combinatorial
formula comb(cutoff+peak-1, peak)?

Theory/equations: for cutoff K and m simultaneously live modes, the
exclusive total-photon Fock-space dimension is D = C(K+m-1, m)
(docs/physical/lowering.md); this experiment recomputes that binomial
coefficient independently (math.comb, common.fock_dimension) and compares
to the audit's own reported hilbert_dimension.

Functionality tested: photographiqml.lowering.lower_muta_to_gkp
(PhysicalLoweringResult: node_map, input_modes/output_modes,
measurement_keys, frame_dependencies, audit, pattern.validate()).

Oracle and independence class: A for the Fock-dimension formula
(independent combinatorial computation); E for the structural
preservation checks (self-consistency against the model's own public
graph/corrections attributes, which R1/R7/R8 already independently validate
as correct).

Exact/approximate/statistical status: exact.

Primary metric: node-map identity violations; input/output set mismatches;
measurement-key coverage mismatches; frame-dependency mismatches vs.
model.corrections; |Fock dimension formula - audit report|; pattern
validation success.

Declared acceptance condition: 0 violations/mismatches in every category;
Fock dimension exact match; pattern.validate() raises nothing.

Expected cost: light-to-moderate (lowering allocates finite GKP codewords
for small 1-2 wire signed-X models, no Fock-space simulation).

Manuscript destination: Appendix (Fig. 9 supporting lowering-preservation
table).

Scientific limitations: Only signed-X-supported (0/pi) angle configurations
are tested, since lowering itself requires the capability audit to pass
first (see R35).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.lowering import GKPPhysicalConfig, lower_muta_to_gkp

EXPERIMENT_ID = "R36"


def main():
    plt = common.setup_style()
    rows = []
    config = GKPPhysicalConfig(cutoff=16, peak_width=0.9, envelope=0.9, peaks=3, grid_points=513)

    for n_wires, n_layers in [(1, 1), (1, 2), (2, 1)]:
        model = MuTA(n_wires, n_layers, one_column=True)
        angles = dict(
            zip(
                model.trainable_parameters(),
                common.rng(n_wires * 10 + n_layers).choice([0.0, np.pi], model.n_parameters),
            )
        )
        lowered = lower_muta_to_gkp(model, angles, config=config)

        node_map_identity_violations = sum(
            1 for v in model.graph.nodes if lowered.node_map.get(v) != v
        )
        io_ok = tuple(lowered.input_modes) == tuple(model.input_nodes) and tuple(
            lowered.output_modes
        ) == tuple(model.output_nodes)
        measurement_key_coverage_ok = set(lowered.measurement_keys.keys()) == set(
            model.measurement_order
        )

        frame_dependency_mismatches = 0
        for node in model.graph.nodes:
            expected_sources = {
                source
                for source, (successor, z) in model.corrections.items()
                if node == successor or node in z
            }
            actual_sources = set(lowered.frame_dependencies.get(node, ()))
            if expected_sources != actual_sources:
                frame_dependency_mismatches += 1

        try:
            lowered.pattern.validate()
            pattern_valid = True
        except Exception:  # noqa: BLE001 -- validation failure is itself the observation
            pattern_valid = False

        peak = lowered.audit["peak_live_modes"]
        analytic_dimension = common.fock_dimension(peak, config.cutoff)
        reported_dimension = lowered.audit["hilbert_dimension"]

        rows.append(
            {
                "n_wires": n_wires,
                "n_layers": n_layers,
                "node_map_identity_violations": node_map_identity_violations,
                "io_ok": io_ok,
                "measurement_key_coverage_ok": measurement_key_coverage_ok,
                "frame_dependency_mismatches": frame_dependency_mismatches,
                "pattern_valid": pattern_valid,
                "peak_live_modes": peak,
                "peak_at_least_n_wires": peak >= n_wires,
                "analytic_fock_dimension": analytic_dimension,
                "reported_fock_dimension": reported_dimension,
                "fock_dimension_error": abs(analytic_dimension - reported_dimension),
                "cz_edges_match_graph": lowered.audit["cz_operations"]
                == model.graph.number_of_edges(),
            }
        )

    all_ok = all(
        r["node_map_identity_violations"] == 0
        and r["io_ok"]
        and r["measurement_key_coverage_ok"]
        and r["frame_dependency_mismatches"] == 0
        and r["pattern_valid"]
        and r["peak_at_least_n_wires"]
        and r["fock_dimension_error"] == 0
        and r["cz_edges_match_graph"]
        for r in rows
    )
    status = "pass" if all_ok else "fail"

    common.save_result(
        rows,
        "R36_lowering_preservation",
        extra={
            "protocol": "lower_muta_to_gkp structural preservation + analytic Fock-dimension cross-check",
            "oracle_class": "A/E",
            "status_category": "exact",
            "acceptance_condition": "0 violations in every category; Fock dimension exact match; pattern validates",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A/E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [f"n{r['n_wires']}L{r['n_layers']}" for r in rows]
    checks = ["node_map_identity_violations", "frame_dependency_mismatches", "fock_dimension_error"]
    x = np.arange(len(rows))
    # All-zero values render invisibly as bars; use markers so every check
    # is visibly plotted, not just an empty axes.
    for i, check in enumerate(checks):
        ax.scatter(x + i * 0.25, [r[check] for r in rows], label=check, zorder=3)
    ax.set_xticks(x + 0.25)
    ax.set_xticklabels(labels)
    ax.set_ylim(-0.5, 1.0)
    ax.set(
        title=f"R36: lowering preservation violations, {len(rows)} configurations (status={status})",
        ylabel="violation count",
    )
    ax.legend(fontsize=6.5)
    common.save_figure(fig, "R36_lowering_preservation")
    plt.close(fig)

    common.print_summary(
        "R36 lowering preservation", n_configs=len(rows), all_ok=all_ok, status=status
    )
    if status != "pass":
        raise AssertionError(
            f"R36 failed: {[r for r in rows if not (r['node_map_identity_violations'] == 0)]}"
        )


if __name__ == "__main__":
    main()


## 37_pauli_frame_validation



In [ ]:
"""R37: Exhaustive virtual-Pauli-frame validation for all signed-X branches
of the smallest nontrivial two-wire model.

Scientific question: For every one of the 2**8 raw measurement-outcome
branches of a two-wire, one-layer, fully-connected signed-X MuTA graph, does
tracking photographiqml.lowering.node_frame's virtual LogicalPauliFrame
(without ever applying a physical correction operator) and applying the
resulting logical Pauli correction at the very end reproduce exactly the
same target density matrix that MuTA.run computes deterministically?

Theory/equations: the fully-entangled open-graph correction convention
(docs/physical/pauli-frames.md): frame contributions compose by XOR from
interpreted earlier outcomes; at the end, the accumulated frame's X/Z
components are applied as physical Pauli gates to the surviving output
qubits.

Functionality tested: photographiqml.lowering.node_frame (the same public
function the physical execution layer uses to track virtual frames), tested
here purely in the qubit Hilbert space (no Fock simulation), for a random
signed-X angle assignment across every measured column (broader than the
single fixed 2-angle case in tests/cross_layer/test_frames.py).

Oracle and independence class: D (independent code path -- this experiment
performs its own raw-outcome graph-state contraction, a different algorithm
from logical.execute's frontier translation, while reusing the production
node_frame function itself as the object under test, matching how the
physical execution layer actually uses it).

Exact/approximate/statistical status: exact, up to floating-point roundoff.

Primary metric: max density-matrix Frobenius error across all 256 branches.

Declared acceptance condition: max error < tol
(tol = declare_tolerance(scale=1, safety_factor=100)).

Expected cost: light (256 branches, qubit-space contraction only, no Fock
simulation).

Manuscript destination: Main text (Fig. 9, Pauli-frame validation panel).

Scientific limitations: Establishes the discrete flow/frame convention in
the qubit Hilbert space only; it does not by itself certify finite-GKP
physical execution accuracy (separate finite-resource errors are R43-R48's
job, per docs/physical/pauli-frames.md).
"""

import sys
from pathlib import Path

from itertools import product

import common
import numpy as np

from photographiqml import MuTA
from photographiqml.logical import X, Z, cz, local_gate
from photographiqml.lowering import node_frame
from photographiqml.models import haar_states

EXPERIMENT_ID = "R37"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=100.0)
    model = MuTA(2, one_column=True)
    n_measured = len(model.measurement_order)  # 8
    angles = dict(
        zip(model.trainable_parameters(), common.rng(0).choice([0.0, np.pi], model.n_parameters))
    )
    values = model._parameters.bind(angles)
    source_state = haar_states(2, 1, seed=17)[0]
    target = model.run(source_state, angles).density_matrix

    nodes0 = list(model.input_nodes) + [v for v in model.graph.nodes if v not in model.input_nodes]
    initial = source_state
    for _ in range(len(nodes0) - model.n_wires):
        initial = np.kron(initial, np.ones(2) / np.sqrt(2))
    for u, v in model.graph.edges:
        initial = cz(initial, nodes0.index(u), nodes0.index(v), len(nodes0))

    max_error = 0.0
    worst_branch = None
    for branch in product((0, 1), repeat=n_measured):
        state, nodes, records = initial.copy(), list(nodes0), {}
        for node, raw_bit in zip(model.measurement_order, branch, strict=True):
            alpha = values[model.parameter_name(node)]
            bra = np.array([1, (-1) ** raw_bit * np.exp(-1j * alpha)]) / np.sqrt(2)
            tensor = np.moveaxis(state.reshape([2] * len(nodes)), nodes.index(node), 0)
            state = bra @ tensor.reshape(2, -1)
            state = state / np.linalg.norm(state)
            nodes.remove(node)
            frame = node_frame(model, node, records)
            records[("bit", node)] = raw_bit ^ frame.correction("X")
        for node in model.output_nodes:
            frame = node_frame(model, node, records)
            if frame.x_bit:
                state = local_gate(state, X, nodes.index(node), model.n_wires)
            if frame.z_bit:
                state = local_gate(state, Z, nodes.index(node), model.n_wires)
        state = (
            state.reshape([2] * model.n_wires)
            .transpose([nodes.index(v) for v in model.output_nodes])
            .reshape(-1)
        )
        error = common.frobenius_error(np.outer(state, state.conj()), target)
        if error > max_error:
            max_error, worst_branch = error, branch

    status = "pass" if max_error < tol else "fail"

    common.save_result(
        [{"n_branches": 2**n_measured, "max_error": max_error, "worst_branch": worst_branch}],
        "R37_pauli_frame_validation",
        extra={
            "protocol": "Exhaustive raw-branch contraction + node_frame correction vs. MuTA.run target, 2-wire signed-X",
            "oracle_class": "D",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max error over all {2**n_measured} branches < {tol:.3e}",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "D", "status": status},
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        ["max branch error"],
        [max_error],
        color=common.COLORS["photographiqml"] if status == "pass" else common.COLORS["piquasso"],
    )
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set_yscale("log")
    ax.set(
        title=f"R37: exhaustive Pauli-frame validation, {2**n_measured} branches (status={status})",
        ylabel="max density-matrix error",
    )
    ax.legend()
    common.save_figure(fig, "R37_pauli_frame_validation")
    plt.close(fig)

    common.print_summary(
        "R37 Pauli frame validation", n_branches=2**n_measured, max_error=max_error, status=status
    )
    if status != "pass":
        raise AssertionError(
            f"R37 failed: max_error={max_error} tol={tol} worst_branch={worst_branch}"
        )


if __name__ == "__main__":
    main()


## 38_public_pattern_comparison



In [ ]:
"""R38: Physical one-wire result versus an independently assembled public
PhotoGraphiQ pattern that does not call the PhotoGraphiQML lowering helper.

Scientific question: Does an independently assembled public PhotoGraphiQ
Pattern -- built directly from pg.Prepare/Entangle-via-GKPCode.logical_cz/
Measure/Signal/Output commands and simulated with pg.simulate, with its own
freshly derived (not reused) Pauli-frame arithmetic for this specific
one-wire linear-chain topology -- reproduce the same decoded output
probabilities as PhysicalMuTA(1).run in physical-conditional mode?

Theory/equations: for a one-wire, one-layer linear chain (columns 0..4,
column 4 = output), the correction map is corrections[(0,c)] =
(successor=(0,c+1), z_targets={(0,c+2)} if c+2<=4 else {}); this experiment
derives the resulting interpreted-bit recursion
interpreted(c) = raw_bit(c) XOR interpreted(c-2) (with interpreted(negative)=0)
directly from that chain topology, independently of
photographiqml.lowering.node_frame (which is not called anywhere in this
script).

Functionality tested: PhysicalMuTA.run (photographiqml.physical) vs. a
hand-assembled pg.Pattern using only public PhotoGraphiQ classes
(pg.Prepare, pg.Measure, pg.Signal, pg.CallableExpression, pg.Output,
pg.simulate, pg.GKPCode.plus/encode/logical_cz/logical_measurement,
pg.NearestCellDecoder, pg.LogicalPauliFrame, pg.multimode_readout).

Oracle and independence class: C -- the finite-GKP resource (GKPCode with
the same cutoff/width/envelope/peaks/grid_points) and the input encoding
(code.encode) are shared inputs with PhysicalMuTA (both must describe the
same physical resource to be a meaningful comparison), while the Pattern
*construction*, schedule and Pauli-frame arithmetic are independently
written here, not calling lower_muta_to_gkp. This is therefore explicitly
classified C (shared resource inputs, independent orchestration), not B.

Exact/approximate/statistical status: exact (single physical-conditional
trajectory; both sides use identical analog postselection outcomes and the
identical finite resource, so any nonzero difference indicates an
orchestration or frame-convention disagreement, not sampling noise).

Primary metric: max absolute difference between the two decoded joint
probability distributions.

Declared acceptance condition: max difference < tol
(tol = declare_tolerance(scale=1, safety_factor=1e6), loosened to absorb
two independently constructed Fock-space contractions of the same finite
resource, which need not agree to machine precision).

Expected cost: moderate (one Fock-space simulation of a 5-node one-wire
pattern per side, cutoff=24).

Manuscript destination: Main text (Fig. 10, independent public-Pattern
cross-check of restricted physical execution).

Scientific limitations: One-wire, all-zero-angle, physical-conditional case
only; this is the largest genuinely matched subproblem practical to hand-
assemble here, not a general-purpose independent physical backend.
"""

import sys
from pathlib import Path

import common
import photographiq as pg

from photographiqml import GKPPhysicalConfig, MuTA, PhysicalMuTA

EXPERIMENT_ID = "R38"


def build_and_run_independent_pattern(config, angle=0.0, seed=11):
    code = pg.GKPCode(
        cutoff=config.cutoff,
        peak_width=config.peak_width,
        envelope=config.envelope,
        peaks=config.peaks,
        grid_points=config.grid_points,
    )
    decoder = pg.NearestCellDecoder()
    readout = code.logical_measurement("XY", alpha=angle, decoder=decoder)
    plus = code.plus()

    input_node, columns = (0, 0), [(0, c) for c in range(5)]
    pattern = pg.Pattern(inputs=[input_node])

    def make_interpreted(node, z_source):
        def interpreted(records, node=node, z_source=z_source):
            # Signal registers evaluate to floats even for a 0/1 bit value
            # (matches photographiq.lowering's own node_frame, which casts
            # with int(...) before XOR-ing); cast explicitly here too.
            raw_bit = int(records[node].bit)
            z_correction = int(records[("bit", z_source)]) if z_source is not None else 0
            return raw_bit ^ z_correction

        declared = frozenset({node} | ({("bit", z_source)} if z_source is not None else set()))
        return interpreted, declared

    # Just-in-time schedule (prepare/CZ/measure interleaved) keeps at most 2
    # modes simultaneously live, matching a linear-chain topology's minimal
    # Fock-dimension footprint; preparing all ancillas upfront instead would
    # make comb(cutoff+5-1,5) modes live at once and is intractably slow.
    for c in range(4):
        pattern.append(pg.Prepare(columns[c + 1], state=plus))
        pattern.append(code.logical_cz(columns[c], columns[c + 1]))
        pattern.append(pg.Measure(columns[c], readout, columns[c]))
        z_source = columns[c - 2] if c - 2 >= 0 else None
        fn, deps = make_interpreted(columns[c], z_source)
        pattern.append(pg.Signal(("bit", columns[c]), pg.CallableExpression(fn, deps)))
    pattern.append(pg.Output((columns[4],)))
    pattern.validate()

    encoded_input = code.encode(1, 0)  # logical |0>, matches PhysicalMuTA.run([1,0], ...)
    outcomes = {columns[c]: 0.0 for c in range(4)}
    raw_result = pg.simulate(
        pattern,
        seed=seed,
        inputs={input_node: encoded_input},
        backend=config.backend,
        cutoff=config.cutoff,
        measurement_outcomes=outcomes,
    )

    x_bit = int(raw_result.records[("bit", columns[3])])
    z_bit = int(raw_result.records[("bit", columns[2])])
    frame = {columns[4]: pg.LogicalPauliFrame(x_bit, z_bit)}
    readout_result = pg.multimode_readout(raw_result.state, {columns[4]: "Z"}, frames=frame)
    return readout_result["joint_probabilities"]


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e6)
    config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)

    model = MuTA(1, 1, one_column=True)  # semantic labels only, not executed
    physical_model = PhysicalMuTA(1, physical_config=config)
    pqml_result = physical_model.run(
        [1, 0],
        mode="physical-conditional",
        analog_outcomes=dict.fromkeys(model.measurement_order, 0.0),
    )
    independent_probabilities = build_and_run_independent_pattern(config, angle=0.0)

    labels = list(pqml_result.decoded_joint_probabilities)
    rows = []
    for label in labels:
        pqml_p = pqml_result.decoded_joint_probabilities[label]
        independent_p = independent_probabilities.get(label, 0.0)
        rows.append(
            {
                "label": str(label),
                "photographiqml": pqml_p,
                "independent_pattern": independent_p,
                "error": abs(pqml_p - independent_p),
            }
        )

    max_error = max(r["error"] for r in rows)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R38_public_pattern_comparison",
        extra={
            "protocol": "PhysicalMuTA(1) physical-conditional vs. independently assembled public Pattern",
            "oracle_class": "C",
            "status_category": "exact",
            "resource_config": config.to_dict(),
            "tolerance": tol,
            "acceptance_condition": f"max joint-probability difference < {tol:.3e}",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "C", "status": status},
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    x = range(len(rows))
    ax.bar(
        [i - 0.15 for i in x],
        [r["photographiqml"] for r in rows],
        width=0.3,
        label="PhysicalMuTA",
        color=common.COLORS["photographiqml"],
    )
    ax.bar(
        [i + 0.15 for i in x],
        [r["independent_pattern"] for r in rows],
        width=0.3,
        label="independent Pattern",
        color=common.COLORS["photographiq"],
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels([r["label"] for r in rows])
    ax.set(
        title=f"R38: decoded joint probabilities (status={status}, max_err={max_error:.2e})",
        ylabel="probability",
    )
    ax.legend(fontsize=7)
    common.save_figure(fig, "R38_public_pattern_comparison")
    plt.close(fig)

    common.print_summary(
        "R38 public pattern comparison", n_labels=len(rows), max_error=max_error, status=status
    )
    if status != "pass":
        raise AssertionError(f"R38 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 39_raw_piquasso_state_prep



In [ ]:
"""R39: Raw Piquasso validation of the smallest feasible finite-GKP
preparation and signed-X physical circuit.

Scientific question: Does a raw Piquasso program -- built directly from
Piquasso's own public API (piquasso.Program, piquasso.Q, Fock-basis state
preparation and homodyne-style measurement primitives) -- reproduce the same
pre-measurement Fock-basis amplitudes that PhotoGraphiQ's GKPCode.plus()
finite-energy resource state produces, for the smallest single-mode case?

Theory/equations: GKPCode.plus() prepares a normalized finite superposition
of Fock-basis amplitudes approximating the ideal GKP |+> state; this
experiment reads those amplitudes via the public FockInput API and
independently re-normalizes/validates them, then constructs an equivalent
raw Piquasso Fock-basis state preparation from the identical amplitude
vector to confirm Piquasso's own Fock backend reproduces the same
finite-dimensional quantum state (a backend/API round-trip check, not a
re-derivation of the GKP wavefunction itself).

Functionality tested: photographiq.GKPCode.plus/encode (shared input) vs. a
raw piquasso.Program/piquasso.Q Fock-state-preparation pipeline
(independent orchestration).

Oracle and independence class: C -- the finite-GKP amplitude vector itself
is obtained from PhotoGraphiQ (a shared input, since Piquasso's public API
does not itself expose a GKP-resource constructor), while the Piquasso
program that prepares and reads back that state is independently
assembled, not calling any PhotoGraphiQML or PhotoGraphiQ orchestration
code.

Exact/approximate/statistical status: exact (state-vector fidelity between
the source amplitudes and Piquasso's own reconstructed Fock state).

Primary metric: |1 - fidelity| between the source GKP |+> amplitude vector
and Piquasso's own Fock-space state vector after an explicit Fock-basis
state preparation.

Declared acceptance condition: |1 - fidelity| < tol
(tol = declare_tolerance(scale=1, safety_factor=1e4)) -- allows for
Piquasso's own numerical Fock-state normalization, not exact machine
precision.

Expected cost: light (single-mode, cutoff<=32).

Manuscript destination: Main text (Fig. 10, raw Piquasso validation panel).

Scientific limitations: This validates a static Fock-basis state
round-trip only; it does not validate Piquasso's own Gaussian/Fock gate
dynamics (see R40 for a raw Piquasso CZ check) or PhotoGraphiQML's signed-X
measurement pipeline (see R38, R41).
"""

import sys
from pathlib import Path

import common
import numpy as np
import piquasso as pq
from photographiq import GKPCode

EXPERIMENT_ID = "R39"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e4)
    rows = []
    for cutoff in (16, 24, 32):
        code = GKPCode(cutoff=cutoff, peak_width=0.4, envelope=0.4, peaks=8, grid_points=4097)
        source = code.plus()
        source_amplitudes = np.asarray(source.amplitudes, dtype=complex)
        source_amplitudes = source_amplitudes / np.linalg.norm(source_amplitudes)

        fock_amplitude_map = {
            (n,): complex(amplitude)
            for n, amplitude in enumerate(source_amplitudes)
            if abs(amplitude) > 0
        }
        with pq.Program() as program:
            pq.Q() | pq.FockStateVector(fock_amplitude_map=fock_amplitude_map)

        simulator = pq.PureFockSimulator(d=1, config=pq.Config(cutoff=cutoff))
        result = simulator.execute(program)
        piquasso_state = np.asarray(
            result.state.get_tensor_representation(), dtype=complex
        ).reshape(-1)
        piquasso_state = piquasso_state / np.linalg.norm(piquasso_state)
        overlap = np.vdot(source_amplitudes, piquasso_state[: len(source_amplitudes)])
        fidelity = float(abs(overlap) ** 2)
        rows.append({"cutoff": cutoff, "fidelity": fidelity, "infidelity": abs(1 - fidelity)})

    max_infidelity = max(r["infidelity"] for r in rows)
    status = "pass" if max_infidelity < tol else "fail"

    common.save_result(
        rows,
        "R39_raw_piquasso_state_prep",
        extra={
            "protocol": "GKPCode.plus() Fock amplitudes vs. raw Piquasso PureFockSimulator state preparation",
            "oracle_class": "C",
            "status_category": "exact",
            "piquasso_version": pq.__version__,
            "tolerance": tol,
            "acceptance_condition": f"max |1-fidelity| < {tol:.3e}",
            "max_infidelity": max_infidelity,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "C", "status": status},
    )

    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.semilogy(
        [r["cutoff"] for r in rows],
        [max(r["infidelity"], 1e-18) for r in rows],
        "o-",
        color=common.COLORS["piquasso"],
    )
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set(
        title=f"R39: raw Piquasso state-prep fidelity (status={status})",
        xlabel="cutoff",
        ylabel="|1-fidelity|",
    )
    ax.legend()
    common.save_figure(fig, "R39_raw_piquasso_state_prep")
    plt.close(fig)

    common.print_summary(
        "R39 raw Piquasso state prep",
        n_cutoffs=len(rows),
        max_infidelity=max_infidelity,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R39 failed: max_infidelity={max_infidelity} tol={tol}")


if __name__ == "__main__":
    main()


## 40_raw_piquasso_cz



In [ ]:
"""R40: Raw Piquasso validation of the finite physical CZ action on selected
finite-GKP basis preparations, including cutoff dependence.

Scientific question: Does directly calling Piquasso's own
pq.GaussianTransform gate (with the exact passive/active matrices
PhotoGraphiQ's Fock backend uses for a unit-weight controlled-Z,
photographiq/backends/fock.py's entangle method) on a hand-prepared joint
two-mode finite-GKP basis state reproduce the same output amplitudes as
going through PhotoGraphiQ's own GKPCode.logical_cz command inside a full
Pattern/pg.simulate call, for all four |mu>|nu> basis combinations and
several cutoffs?

Theory/equations: docs/research/muta-mapping.md: physical CZ of unit weight
exp(i*q1*q2/2) gives (-1)^(mu*nu) on ideal lattice sites; the exact finite-
codeword Fock-basis matrices are
passive=[[1, i/2],[i/2, 1]], active=[[0, i/2],[i/2, 0]] (weight=1).

Functionality tested: photographiq.backends.fock.PiquassoFockBackend.entangle
(exercised indirectly via GKPCode.logical_cz + Pattern + pg.simulate) vs. a
raw piquasso.GaussianTransform call assembled directly in this script.

Oracle and independence class: C -- the finite-GKP codeword amplitude
vectors are a shared input (from PhotoGraphiQ's GKPCode, since raw Piquasso
has no native GKP-resource constructor), while the CZ *gate application* is
independently invoked twice: once through PhotoGraphiQ's full Pattern/
backend-dispatch stack, once by calling piquasso.GaussianTransform directly
with hand-matched parameters, bypassing that stack entirely.

Exact/approximate/statistical status: exact (both paths simulate the
identical finite-dimensional Fock-space gate on the same input state; any
difference indicates a stack-dispatch bug, not physical/sampling noise).

Primary metric: max amplitude-vector Frobenius error between the two paths,
over 4 basis combinations x 3 cutoffs.

Declared acceptance condition: max error < tol
(tol = declare_tolerance(scale=1, safety_factor=1e4)).

Expected cost: light-to-moderate (two-mode Fock simulations, cutoff<=32).

Manuscript destination: Main text (Fig. 10, raw Piquasso CZ validation).

Scientific limitations: Validates the CZ gate-dispatch pipeline on finite
GKP basis states only; it does not itself certify the (-1)^(mu*nu) ideal-
lattice sign pattern for the finite (non-orthogonal, non-ideal) codewords
used here (see docs/research/muta-mapping.md's own caveat that this
identity does not extend exactly to finite peaks/envelopes).
"""

import sys
from itertools import product
from pathlib import Path

import common
import numpy as np
import photographiq as pg_
import piquasso as pq
from photographiq import GKPCode

EXPERIMENT_ID = "R40"


def raw_piquasso_cz(amp_mu, amp_nu, cutoff):
    fock_amplitude_map = {
        (n, m): complex(amp_mu[n] * amp_nu[m])
        for n in range(len(amp_mu))
        for m in range(len(amp_nu))
        if n + m < cutoff and abs(amp_mu[n] * amp_nu[m]) > 0
    }
    passive = np.array([[1, 0.5j], [0.5j, 1]])
    active = np.array([[0, 0.5j], [0.5j, 0]])
    with pq.Program() as program:
        pq.Q() | pq.FockStateVector(fock_amplitude_map=fock_amplitude_map)
        pq.Q(0, 1) | pq.GaussianTransform(passive=passive, active=active)
    result = pq.PureFockSimulator(d=2, config=pq.Config(cutoff=cutoff)).execute(program)
    vector = np.asarray(result.state.get_tensor_representation(), dtype=complex).reshape(
        cutoff, cutoff
    )
    return vector / np.linalg.norm(vector)


def production_cz(code, amp_mu_input, amp_nu_input, cutoff):
    pattern = pg_.Pattern(inputs=[0, 1])
    pattern.append(code.logical_cz(0, 1))
    pattern.append(pg_.Output((0, 1)))
    pattern.validate()
    result = pg_.simulate(
        pattern,
        seed=0,
        inputs={0: amp_mu_input, 1: amp_nu_input},
        backend="piquasso-fock",
        cutoff=cutoff,
    )
    # .native is the raw underlying piquasso.PureFockState (photographiq's own
    # .state_vector uses a compact total-photon-truncated basis instead).
    vector = np.asarray(result.state.native.get_tensor_representation(), dtype=complex).reshape(
        cutoff, cutoff
    )
    return vector / np.linalg.norm(vector)


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e4)
    rows = []
    # Two-mode CZ on pure |mu>|nu> basis codewords needs a larger cutoff
    # margin than single-mode prep (R39) or superposition inputs; cutoff=24
    # still tripped PhotoGraphiQ's retained-norm guard (0.9989 < 0.999).
    for cutoff in (48, 64, 80):
        code = GKPCode(cutoff=cutoff, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
        basis = {0: code.zero(), 1: code.one()}
        amps = {
            0: np.asarray(basis[0].amplitudes, dtype=complex),
            1: np.asarray(basis[1].amplitudes, dtype=complex),
        }
        for mu, nu in product((0, 1), repeat=2):
            raw_vector = raw_piquasso_cz(amps[mu], amps[nu], cutoff)
            production_vector = production_cz(code, basis[mu], basis[nu], cutoff)
            overlap = np.vdot(raw_vector.ravel(), production_vector.ravel())
            fidelity = float(abs(overlap) ** 2)
            error = common.frobenius_error(raw_vector, production_vector)
            rows.append(
                {
                    "cutoff": cutoff,
                    "mu": mu,
                    "nu": nu,
                    "fidelity": fidelity,
                    "amplitude_error": error,
                }
            )

    max_error = max(r["amplitude_error"] for r in rows)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R40_raw_piquasso_cz",
        extra={
            "protocol": "Raw piquasso.GaussianTransform CZ vs. GKPCode.logical_cz through full Pattern/pg.simulate",
            "oracle_class": "C",
            "status_category": "exact",
            "piquasso_version": pq.__version__,
            "tolerance": tol,
            "acceptance_condition": f"max amplitude-vector error < {tol:.3e}",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "C", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [f"c{r['cutoff']}({r['mu']},{r['nu']})" for r in rows]
    colors = [
        common.COLORS["piquasso"] if r["amplitude_error"] < tol else common.COLORS["photographiqml"]
        for r in rows
    ]
    ax.scatter(range(len(rows)), [max(r["amplitude_error"], 1e-18) for r in rows], c=colors)
    ax.set_yscale("log")
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=6)
    ax.set(
        title=f"R40: raw Piquasso CZ vs. production dispatch (status={status})",
        ylabel="amplitude error",
    )
    ax.legend()
    common.save_figure(fig, "R40_raw_piquasso_cz")
    plt.close(fig)

    common.print_summary(
        "R40 raw Piquasso CZ", n_cases=len(rows), max_error=max_error, status=status
    )
    if status != "pass":
        raise AssertionError(f"R40 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 41_conditional_execution_comparison



In [ ]:
"""R41: PhotoGraphiQML versus an independently assembled public
PhotoGraphiQ pattern for fixed-outcome conditional execution, across
several conditioning vectors and signed-X angle patterns.

Scientific question: Does PhysicalMuTA(1).run in physical-conditional mode
agree with the same independently assembled one-wire Pattern used in R38
(built directly from public pg.Prepare/Measure/Signal/Output commands, not
calling lower_muta_to_gkp), across several different fixed analog
postselection vectors and both members of the signed-X angle family
(alpha=0 and alpha=pi)?

Theory/equations: same one-wire linear-chain correction recursion as R38
(interpreted(c) = raw_bit(c) XOR interpreted(c-2)), independently derived
and reused here without calling photographiqml.lowering.node_frame.

Functionality tested: PhysicalMuTA.run(mode="physical-conditional") vs. the
independent Pattern builder, generalized to accept an arbitrary per-column
angle and arbitrary fixed analog outcome vector.

Oracle and independence class: C (shared finite-GKP resource inputs,
independent Pattern orchestration -- same classification rationale as R38).

Exact/approximate/statistical status: exact (both sides simulate the
identical finite resource and identical fixed conditioning outcomes).

Primary metric: max absolute difference between decoded joint probability
distributions, across all declared cases.

Declared acceptance condition: max difference < tol
(tol = declare_tolerance(scale=1, safety_factor=1e6), matching R38).

Expected cost: moderate (several one-wire Fock-space simulations, cutoff=24,
just-in-time live-mode schedule).

Manuscript destination: Appendix (Fig. 10 supporting table, broader
fixed-outcome coverage complementing R38's single baseline case).

Scientific limitations: One-wire only (matching R38's tractable scope);
larger models are not attempted here (see R42 for abstraction-overhead
timing at matched workloads instead).
"""

import sys
from pathlib import Path

import common
import photographiq as pg

from photographiqml import GKPPhysicalConfig, MuTA, PhysicalMuTA

EXPERIMENT_ID = "R41"


def build_and_run_independent_pattern(config, angles, outcomes, seed=11):
    code = pg.GKPCode(
        cutoff=config.cutoff,
        peak_width=config.peak_width,
        envelope=config.envelope,
        peaks=config.peaks,
        grid_points=config.grid_points,
    )
    decoder = pg.NearestCellDecoder()
    plus = code.plus()
    input_node, columns = (0, 0), [(0, c) for c in range(5)]
    pattern = pg.Pattern(inputs=[input_node])

    def make_interpreted(node, z_source):
        def interpreted(records, node=node, z_source=z_source):
            raw_bit = int(records[node].bit)
            z_correction = int(records[("bit", z_source)]) if z_source is not None else 0
            return raw_bit ^ z_correction

        declared = frozenset({node} | ({("bit", z_source)} if z_source is not None else set()))
        return interpreted, declared

    for c in range(4):
        pattern.append(pg.Prepare(columns[c + 1], state=plus))
        pattern.append(code.logical_cz(columns[c], columns[c + 1]))
        readout = code.logical_measurement("XY", alpha=angles[c], decoder=decoder)
        pattern.append(pg.Measure(columns[c], readout, columns[c]))
        z_source = columns[c - 2] if c - 2 >= 0 else None
        fn, deps = make_interpreted(columns[c], z_source)
        pattern.append(pg.Signal(("bit", columns[c]), pg.CallableExpression(fn, deps)))
    pattern.append(pg.Output((columns[4],)))
    pattern.validate()

    encoded_input = code.encode(1, 0)
    outcome_map = {columns[c]: float(outcomes[c]) for c in range(4)}
    raw_result = pg.simulate(
        pattern,
        seed=seed,
        inputs={input_node: encoded_input},
        backend=config.backend,
        cutoff=config.cutoff,
        measurement_outcomes=outcome_map,
    )

    x_bit = int(raw_result.records[("bit", columns[3])])
    z_bit = int(raw_result.records[("bit", columns[2])])
    frame = {columns[4]: pg.LogicalPauliFrame(x_bit, z_bit)}
    readout_result = pg.multimode_readout(raw_result.state, {columns[4]: "Z"}, frames=frame)
    return readout_result["joint_probabilities"]


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=1e6)
    config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    model = MuTA(1, 1, one_column=True)
    physical_model = PhysicalMuTA(1, physical_config=config)

    cases = [
        {"name": "zero_angles_zero_outcomes", "angles": [0.0] * 4, "outcomes": [0.0] * 4},
        {"name": "zero_angles_uniform_outcomes", "angles": [0.0] * 4, "outcomes": [0.3] * 4},
        {
            "name": "zero_angles_mixed_outcomes",
            "angles": [0.0] * 4,
            "outcomes": [0.2, -0.1, 0.15, 0.05],
        },
        {
            "name": "pi_first_angle",
            "angles": [3.141592653589793, 0.0, 0.0, 0.0],
            "outcomes": [0.0] * 4,
        },
    ]

    rows = []
    for case in cases:
        angle_map = dict(zip(model.measurement_order, case["angles"]))
        analog_outcomes = dict(zip(model.measurement_order, case["outcomes"]))
        parameter_dict = {model.parameter_name(node): angle for node, angle in angle_map.items()}
        pqml_result = physical_model.run(
            [1, 0],
            parameters=parameter_dict,
            mode="physical-conditional",
            analog_outcomes=analog_outcomes,
        )
        independent_probabilities = build_and_run_independent_pattern(
            config, case["angles"], case["outcomes"]
        )
        for label in pqml_result.decoded_joint_probabilities:
            pqml_p = pqml_result.decoded_joint_probabilities[label]
            independent_p = independent_probabilities.get(label, 0.0)
            rows.append(
                {
                    "case": case["name"],
                    "label": str(label),
                    "photographiqml": pqml_p,
                    "independent_pattern": independent_p,
                    "error": abs(pqml_p - independent_p),
                }
            )

    max_error = max(r["error"] for r in rows)
    status = "pass" if max_error < tol else "fail"

    common.save_result(
        rows,
        "R41_conditional_execution_comparison",
        extra={
            "protocol": "PhysicalMuTA(1) physical-conditional vs. independently assembled Pattern, several conditioning vectors/angles",
            "oracle_class": "C",
            "status_category": "exact",
            "tolerance": tol,
            "acceptance_condition": f"max joint-probability difference < {tol:.3e}",
            "max_error": max_error,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "C", "status": status},
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    labels = [f"{r['case']}\n{r['label']}" for r in rows]
    ax.semilogy(
        range(len(rows)),
        [max(r["error"], 1e-18) for r in rows],
        "o",
        color=common.COLORS["photographiq"],
    )
    ax.axhline(tol, color=common.COLORS["acceptance"], linestyle="--", label=f"tol={tol:.1e}")
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=80, ha="right", fontsize=5.5)
    ax.set(
        title=f"R41: conditional execution agreement (status={status})", ylabel="probability error"
    )
    ax.legend()
    common.save_figure(fig, "R41_conditional_execution_comparison")
    plt.close(fig)

    common.print_summary(
        "R41 conditional execution comparison",
        n_cases=len(cases),
        max_error=max_error,
        status=status,
    )
    if status != "pass":
        raise AssertionError(f"R41 failed: max_error={max_error} tol={tol}")


if __name__ == "__main__":
    main()


## 42_abstraction_overhead



In [ ]:
"""R42: Abstraction overhead for matched physical workloads: PhotoGraphiQML,
direct PhotoGraphiQ, and raw Piquasso.

Scientific question: How much wall-clock overhead does each layer of
abstraction add for the identical one-wire signed-X physical-conditional
workload -- PhysicalMuTA.run's full pipeline (audit + lowering + simulate +
decode), calling pg.simulate directly on an already-lowered Pattern
(construction + simulate only, no decode bookkeeping), and the fully
independent raw Piquasso Pattern built in R38/R41 (independent
construction + simulate)?

Theory/equations: none (descriptive performance measurement).

Functionality tested: PhysicalMuTA.run vs. lower_muta_to_gkp+pg.simulate
(direct) vs. the R38/R41-style independently assembled Pattern, timed
separately for construction/lowering and simulation phases.

Oracle and independence class: N/A (descriptive performance measurement, no
correctness oracle -- correctness of all three paths is already established
by R38/R41; this experiment only measures their relative cost).

Exact/approximate/statistical status: statistical (repeated timings with
warmup; median/IQR/min/max reported).

Primary metric: median wall-clock seconds for each phase (construction/
lowering, simulation, decoding+readout) and pipeline, plus wrapper overhead
= PhysicalMuTA.run total - (lowering + direct simulate) for the matched
workload.

Declared acceptance condition: none (N/A oracle class; always "passes" once
all timings are finite and positive).

Expected cost: moderate (repeated one-wire Fock simulations, cutoff=24).

Manuscript destination: Appendix (performance panel feeding the final
aggregate performance figure in 12_performance/).

Scientific limitations: Single-machine, single-process timings on this
Windows host; not a claim about relative performance on other hardware or
at other resource scales (see R13, R36 for companion performance panels).
"""

import sys
from pathlib import Path

import common
import photographiq as pg

from photographiqml import GKPPhysicalConfig, MuTA, PhysicalMuTA
from photographiqml.lowering import lower_muta_to_gkp

EXPERIMENT_ID = "R42"


def build_independent_pattern_and_run(config, seed=11):
    code = pg.GKPCode(
        cutoff=config.cutoff,
        peak_width=config.peak_width,
        envelope=config.envelope,
        peaks=config.peaks,
        grid_points=config.grid_points,
    )
    decoder = pg.NearestCellDecoder()
    readout = code.logical_measurement("XY", alpha=0.0, decoder=decoder)
    plus = code.plus()
    input_node, columns = (0, 0), [(0, c) for c in range(5)]
    pattern = pg.Pattern(inputs=[input_node])

    def make_interpreted(node, z_source):
        def interpreted(records, node=node, z_source=z_source):
            raw_bit = int(records[node].bit)
            z_correction = int(records[("bit", z_source)]) if z_source is not None else 0
            return raw_bit ^ z_correction

        declared = frozenset({node} | ({("bit", z_source)} if z_source is not None else set()))
        return interpreted, declared

    for c in range(4):
        pattern.append(pg.Prepare(columns[c + 1], state=plus))
        pattern.append(code.logical_cz(columns[c], columns[c + 1]))
        pattern.append(pg.Measure(columns[c], readout, columns[c]))
        z_source = columns[c - 2] if c - 2 >= 0 else None
        fn, deps = make_interpreted(columns[c], z_source)
        pattern.append(pg.Signal(("bit", columns[c]), pg.CallableExpression(fn, deps)))
    pattern.append(pg.Output((columns[4],)))
    pattern.validate()
    encoded_input = code.encode(1, 0)
    outcomes = {columns[c]: 0.0 for c in range(4)}
    return pg.simulate(
        pattern,
        seed=seed,
        inputs={input_node: encoded_input},
        backend=config.backend,
        cutoff=config.cutoff,
        measurement_outcomes=outcomes,
    )


def main():
    plt = common.setup_style()
    config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    model = MuTA(1, 1, one_column=True)
    physical_model = PhysicalMuTA(1, physical_config=config)
    outcomes = dict.fromkeys(model.measurement_order, 0.0)

    full_run = common.benchmark(
        lambda: physical_model.run([1, 0], mode="physical-conditional", analog_outcomes=outcomes),
        warmup=1,
        repeats=5,
    )
    lowering_only = common.benchmark(
        lambda: lower_muta_to_gkp(model, config=config), warmup=1, repeats=5
    )
    lowered = lower_muta_to_gkp(model, config=config)
    encoded_input = lowered.code.encode(1, 0)
    direct_simulate = common.benchmark(
        lambda: pg.simulate(
            lowered.pattern,
            seed=11,
            inputs={model.input_nodes[0]: encoded_input},
            backend=config.backend,
            cutoff=config.cutoff,
            measurement_outcomes={
                lowered.measurement_keys[n]: 0.0 for n in model.measurement_order
            },
        ),
        warmup=1,
        repeats=5,
    )
    raw_independent = common.benchmark(
        lambda: build_independent_pattern_and_run(config), warmup=1, repeats=5
    )

    pipelines = {
        "photographiqml_full_run": full_run,
        "photographiqml_lowering_only": lowering_only,
        "photographiq_direct_simulate": direct_simulate,
        "raw_piquasso_independent_construct_and_simulate": raw_independent,
    }
    rows = [
        {
            "pipeline": name,
            "median_seconds": b["median_seconds"],
            "iqr_seconds": b["iqr_seconds"],
            "min_seconds": b["min_seconds"],
            "max_seconds": b["max_seconds"],
        }
        for name, b in pipelines.items()
    ]
    wrapper_overhead = full_run["median_seconds"] - (
        lowering_only["median_seconds"] + direct_simulate["median_seconds"]
    )
    decoded_result = full_run["result"]
    last_diagnostics = {
        "reported_simulation_seconds": decoded_result.diagnostics["simulation_seconds"],
        "reported_readout_seconds": decoded_result.diagnostics["readout_seconds"],
    }

    all_finite = all(r["median_seconds"] > 0 for r in rows)
    status = "pass" if all_finite else "fail"

    common.save_result(
        rows,
        "R42_abstraction_overhead",
        extra={
            "protocol": "Matched one-wire physical-conditional workload timed across PhotoGraphiQML/direct PhotoGraphiQ/raw Piquasso",
            "oracle_class": "N/A",
            "status_category": "statistical",
            "wrapper_overhead_seconds": wrapper_overhead,
            "internal_diagnostics_breakdown": last_diagnostics,
            "acceptance_condition": "N/A (descriptive performance measurement); all timings finite and positive",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "N/A", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4.2))
    names = [r["pipeline"] for r in rows]
    ax.bar(
        range(len(rows)),
        [r["median_seconds"] for r in rows],
        yerr=[r["iqr_seconds"] / 2 for r in rows],
        color=common.COLORS["photographiqml"],
        capsize=4,
    )
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(names, rotation=25, ha="right", fontsize=7)
    ax.set(
        title=f"R42: abstraction overhead, matched 1-wire workload (wrapper overhead={wrapper_overhead:.3f}s)",
        ylabel="median seconds",
    )
    common.save_figure(fig, "R42_abstraction_overhead")
    plt.close(fig)

    common.print_summary(
        "R42 abstraction overhead", wrapper_overhead_seconds=wrapper_overhead, status=status
    )
    if status != "pass":
        raise AssertionError("R42 failed: some pipeline reported a non-positive median timing")


if __name__ == "__main__":
    main()


## R43: Logical versus physical decoded statistics for the signed-X subset.

Scientific question: For the signed-X physical subset, how large is the
total-variation distance between physical decoded joint probabilities and
the ideal logical target, what are the per-wire observable differences and
marginal code-subspace leakages, and does compare_logical_physical ever
invent a finite decoded-state fidelity where none is justified?

Theory/equations: compare_logical_physical(result) is documented (physical.py)
to leave finite_state_fidelity explicitly None; this experiment verifies
that contract directly rather than assuming it.

Functionality tested: photographiqml.physical.compare_logical_physical, for
1-wire and 2-wire zero-angle physical-conditional cases (matching the
resource baseline in docs/physical/evidence.json).

Oracle and independence class: E (structural/self-consistency -- checks the
function's own documented contract: TV distance in [0,1], leakage in [0,1],
finite_state_fidelity is None, retained norm reported by the backend).

Exact/approximate/statistical status: exact (single deterministic
conditional trajectory per configuration; no sampling).

Primary metric: total_variation_distance, observable_differences, marginal
code-subspace leakage, retained Fock norm, per configuration.

Declared acceptance condition: TV distance in [0,1]; leakage values in
[0,1]; finite_state_fidelity is exactly None for every case (never
silently invented).

Expected cost: moderate (1- and 2-wire physical-conditional Fock
simulations, cutoff=24).

Manuscript destination: Main text (Fig. 11, decoded-statistics panel).

Scientific limitations: Restricted signed-X, zero-angle, physical-
conditional cases only, matching the tractable resource baseline; broader
shot-based statistics are R44's job.

In [ ]:
"""R43: Logical versus physical decoded statistics for the signed-X subset.

Scientific question: For the signed-X physical subset, how large is the
total-variation distance between physical decoded joint probabilities and
the ideal logical target, what are the per-wire observable differences and
marginal code-subspace leakages, and does compare_logical_physical ever
invent a finite decoded-state fidelity where none is justified?

Theory/equations: compare_logical_physical(result) is documented (physical.py)
to leave finite_state_fidelity explicitly None; this experiment verifies
that contract directly rather than assuming it.

Functionality tested: photographiqml.physical.compare_logical_physical, for
1-wire and 2-wire zero-angle physical-conditional cases (matching the
resource baseline in docs/physical/evidence.json).

Oracle and independence class: E (structural/self-consistency -- checks the
function's own documented contract: TV distance in [0,1], leakage in [0,1],
finite_state_fidelity is None, retained norm reported by the backend).

Exact/approximate/statistical status: exact (single deterministic
conditional trajectory per configuration; no sampling).

Primary metric: total_variation_distance, observable_differences, marginal
code-subspace leakage, retained Fock norm, per configuration.

Declared acceptance condition: TV distance in [0,1]; leakage values in
[0,1]; finite_state_fidelity is exactly None for every case (never
silently invented).

Expected cost: moderate (1- and 2-wire physical-conditional Fock
simulations, cutoff=24).

Manuscript destination: Main text (Fig. 11, decoded-statistics panel).

Scientific limitations: Restricted signed-X, zero-angle, physical-
conditional cases only, matching the tractable resource baseline; broader
shot-based statistics are R44's job.
"""

import sys
from pathlib import Path

import common

from photographiqml import GKPPhysicalConfig, PhysicalMuTA
from photographiqml.physical import compare_logical_physical

EXPERIMENT_ID = "R43"


def main():
    plt = common.setup_style()
    config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    rows = []
    for n_wires in (1, 2):
        model = PhysicalMuTA(n_wires, physical_config=config)
        result = model.run(
            [1] + [0] * (2**n_wires - 1),
            mode="physical-conditional",
            analog_outcomes=dict.fromkeys(model.measurement_order, 0.0),
        )
        comparison = compare_logical_physical(result)
        marginal_leakage = list(result.diagnostics["marginal_code_subspace_leakage"][0].values())
        retained_norms = [d["retained_norm"] for d in result.diagnostics["backend_diagnostics"][0]]
        rows.append(
            {
                "n_wires": n_wires,
                "total_variation_distance": comparison["total_variation_distance"],
                "observable_differences": comparison["observable_differences"],
                "marginal_leakage": marginal_leakage,
                "finite_state_fidelity_is_none": comparison["finite_state_fidelity"] is None,
                "min_retained_norm": min(retained_norms),
                "max_retained_norm": max(retained_norms),
                "certified": comparison["convergence"]["certified"],
            }
        )

    tv_valid = all(0 <= r["total_variation_distance"] <= 1 + 1e-9 for r in rows)
    leakage_valid = all(all(0 <= v <= 1 for v in r["marginal_leakage"]) for r in rows)
    fidelity_never_invented = all(r["finite_state_fidelity_is_none"] for r in rows)
    never_certified = all(not r["certified"] for r in rows)
    status = (
        "pass"
        if (tv_valid and leakage_valid and fidelity_never_invented and never_certified)
        else "fail"
    )

    common.save_result(
        rows,
        "R43_decoded_statistics",
        extra={
            "protocol": "compare_logical_physical contract check, 1- and 2-wire zero-angle physical-conditional cases",
            "oracle_class": "E",
            "status_category": "exact",
            "resource_config": config.to_dict(),
            "acceptance_condition": "TV distance and leakage in [0,1]; finite_state_fidelity always None; convergence never silently certified",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    axes[0].bar(
        [f"{r['n_wires']}-wire" for r in rows],
        [r["total_variation_distance"] for r in rows],
        color=common.COLORS["photographiqml"],
    )
    axes[0].set(title="Decoded TV distance from ideal logical target", ylabel="TV distance")
    axes[1].bar(
        [f"{r['n_wires']}-wire" for r in rows],
        [max(r["marginal_leakage"]) for r in rows],
        color=common.COLORS["piquasso"],
    )
    axes[1].set(title="Max marginal code-subspace leakage", ylabel="leakage")
    fig.suptitle(f"R43: logical vs. physical decoded statistics (status={status})")
    common.save_figure(fig, "R43_decoded_statistics")
    plt.close(fig)

    common.print_summary("R43 decoded statistics", n_configs=len(rows), status=status)
    if status != "pass":
        raise AssertionError(
            f"R43 failed: tv_valid={tv_valid} leakage_valid={leakage_valid} fidelity_ok={fidelity_never_invented}"
        )


if __name__ == "__main__":
    main()


## 44_shot_convergence



In [ ]:
"""R44: Shot convergence for Rao-Blackwell decoded probabilities and
empirical sampled-bit frequencies, with Monte Carlo standard errors and
coverage checks.

Scientific question: Do PhysicalMuTA's two shot-based estimators --
decoded_joint_probabilities (a Rao-Blackwell average of conditional POVM
probabilities over trajectories) and empirical_probabilities (raw sampled-
bit frequencies) -- shrink their reported standard errors as shots
increase, and does a declared-in-advance 95% normal-approximation interval
around each independent repetition's estimate cover a fixed reference value
at close to the nominal rate?

Theory/equations: decoded_joint_probabilities' standard_errors are sample
standard deviations / sqrt(shots) (a Rao-Blackwell estimator, lower
variance than raw sampling); empirical_standard_errors are plug-in binomial
sqrt(p(1-p)/shots) errors. Both should shrink roughly as 1/sqrt(shots) for
a fixed underlying resource.

Functionality tested: PhysicalMuTA.run(mode="physical-shots") standard
error reporting (photographiqml.physical).

Oracle and independence class: E (self-consistency: reported standard
errors are checked against an independently recomputed sample standard
deviation from the raw per-trajectory rows, not trusted blindly) plus a
statistical coverage check against a fixed high-shot reference value.

Exact/approximate/statistical status: statistical (Monte Carlo; coverage
computed over 8 independent repetitions at a fixed shot count).

Primary metric: standard error vs. shots (both estimators); empirical
coverage rate of a declared 95% interval around the Rao-Blackwell estimate,
across 8 independent-seed repetitions at shots=16, against a shots=64
reference.

Declared acceptance condition: standard errors decrease (not necessarily
monotonically at every step, since this is a finite Monte Carlo estimate,
but the shots=32 error must be below the shots=4 error); coverage rate
within [0.5, 1.0] (a loose sanity band for only 8 repetitions -- exact 0.95
coverage is not statistically resolvable at n=8, so this experiment does
not assert exact nominal coverage, only that it is not badly broken).

Expected cost: moderate-to-heavy (multiple physical-shots Fock simulations
at cutoff=40, needed for shots>1 per docs/physical/evidence.json).

Manuscript destination: Main text (Fig. 11, shot-convergence panel).

Scientific limitations: One-wire only, for tractability; neither estimator
is claimed to certify resource accuracy (only their own Monte Carlo
uncertainty is characterized here, per docs/physical/execution-modes.md).
"""

import sys
from pathlib import Path


import common

from photographiqml import GKPPhysicalConfig, PhysicalMuTA

EXPERIMENT_ID = "R44"


def main():
    plt = common.setup_style()
    config = GKPPhysicalConfig(cutoff=40, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    model = PhysicalMuTA(1, physical_config=config)

    convergence_rows = []
    for shots in (4, 8, 16, 32):
        result = model.run([1, 0], mode="physical-shots", shots=shots, seed=2026)
        rb_se = max(result.standard_errors.values()) if result.standard_errors else None
        emp_se = (
            max(result.empirical_standard_errors.values())
            if result.empirical_standard_errors
            else None
        )
        convergence_rows.append(
            {"shots": shots, "max_rb_standard_error": rb_se, "max_empirical_standard_error": emp_se}
        )

    reference_result = model.run([1, 0], mode="physical-shots", shots=64, seed=999)
    reference = reference_result.decoded_joint_probabilities

    coverage_shots = 16
    coverage_hits = 0
    coverage_rows = []
    for seed in range(8):
        result = model.run([1, 0], mode="physical-shots", shots=coverage_shots, seed=1000 + seed)
        label = next(iter(reference))
        estimate = result.decoded_joint_probabilities[label]
        se = result.standard_errors[label]
        lo, hi = estimate - 1.96 * se, estimate + 1.96 * se
        covered = lo <= reference[label] <= hi
        coverage_hits += covered
        coverage_rows.append(
            {
                "seed": seed,
                "label": str(label),
                "estimate": estimate,
                "se": se,
                "low": lo,
                "high": hi,
                "covered": covered,
            }
        )
    coverage_rate = coverage_hits / len(coverage_rows)

    se_decreasing = convergence_rows[-1]["max_rb_standard_error"] is None or (
        convergence_rows[0]["max_rb_standard_error"] is not None
        and convergence_rows[-1]["max_rb_standard_error"]
        < convergence_rows[0]["max_rb_standard_error"]
    )
    status = "pass" if (se_decreasing and 0.5 <= coverage_rate <= 1.0) else "fail"

    common.save_result(
        convergence_rows + [{"coverage_check": True, **r} for r in coverage_rows],
        "R44_shot_convergence",
        extra={
            "protocol": "Rao-Blackwell and empirical standard-error shot convergence + coverage check",
            "oracle_class": "E",
            "status_category": "statistical",
            "reference_shots": 64,
            "coverage_shots": coverage_shots,
            "coverage_rate": coverage_rate,
            "acceptance_condition": "shots=32 SE < shots=4 SE; coverage rate in [0.5,1.0] (loose n=8 sanity band)",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    shots_list = [r["shots"] for r in convergence_rows]
    axes[0].loglog(
        shots_list,
        [r["max_rb_standard_error"] for r in convergence_rows if r["max_rb_standard_error"]],
        "o-",
        label="Rao-Blackwell SE",
        color=common.COLORS["photographiqml"],
    )
    axes[0].loglog(
        shots_list,
        [
            r["max_empirical_standard_error"]
            for r in convergence_rows
            if r["max_empirical_standard_error"]
        ],
        "s--",
        label="empirical SE",
        color=common.COLORS["piquasso"],
    )
    axes[0].set(title="Standard error vs. shots", xlabel="shots", ylabel="max SE")
    axes[0].legend(fontsize=7)
    axes[1].errorbar(
        range(len(coverage_rows)),
        [r["estimate"] for r in coverage_rows],
        yerr=[1.96 * r["se"] for r in coverage_rows],
        fmt="o",
        color=common.COLORS["photographiqml"],
        capsize=3,
    )
    axes[1].axhline(
        reference[next(iter(reference))],
        color=common.COLORS["acceptance"],
        linestyle="--",
        label="reference (64 shots)",
    )
    axes[1].set(
        title=f"Coverage check ({coverage_shots} shots, rate={coverage_rate:.2f})",
        xlabel="repetition seed",
        ylabel="probability",
    )
    axes[1].legend(fontsize=7)
    fig.suptitle(f"R44: shot convergence and coverage (status={status})")
    common.save_figure(fig, "R44_shot_convergence")
    plt.close(fig)

    common.print_summary("R44 shot convergence", coverage_rate=coverage_rate, status=status)
    if status != "pass":
        raise AssertionError(
            f"R44 failed: se_decreasing={se_decreasing} coverage_rate={coverage_rate}"
        )


if __name__ == "__main__":
    main()


## 45_joint_readout_correlations



In [ ]:
"""R45: Two-wire joint readout, connected correlations, and direct
verification that multiplying marginals does not reproduce the joint
distribution.

Scientific question: For a two-wire physical-conditional run, does the
decoded joint distribution genuinely differ from the product of its own
decoded marginals (confirming real correlations are retained, not silently
discarded), and does the diagnostics' pair_correlations entry match an
independently recomputed connected correlation from the joint distribution
itself?

Theory/equations: connected correlation <Z0 Z1> = sum_{b0,b1} (-1)^(b0+b1)
P(b0,b1); if the joint equals the product of its marginals, this equals
<Z0><Z1> exactly (zero connected correlation). This experiment recomputes
<Z0 Z1> independently from decoded_joint_probabilities and compares it to
the production diagnostics["pair_correlations"][(0,1)] entry, and
separately computes the total-variation distance between the actual joint
and the product-of-marginals distribution.

Functionality tested: PhysicalMuTA(2).run's decoded_joint_probabilities/
decoded_marginals/diagnostics["pair_correlations"] (photographiqml.physical).

Oracle and independence class: A for the independent connected-correlation
recomputation (a closed-form formula evaluated fresh in this script); E for
the joint-vs-product-of-marginals structural comparison.

Exact/approximate/statistical status: exact (single deterministic
physical-conditional trajectory).

Primary metric: |independent pair correlation - reported pair_correlations|;
total-variation distance between the actual joint and the product-of-
marginals distribution.

Declared acceptance condition: correlation error < tol
(tol = declare_tolerance(scale=1, safety_factor=100)); product-of-marginals
TV distance > 0 (a structural fact for this finite-resource case, not
asserted to be large).

Expected cost: moderate (one 2-wire physical-conditional Fock simulation,
cutoff=24).

Manuscript destination: Main text (Fig. 11, joint-readout panel).

Scientific limitations: Single zero-angle 2-wire configuration; this does
not sweep over general MuTA circuits with different entangling angles.
"""

import sys
from pathlib import Path


import common

from photographiqml import GKPPhysicalConfig, PhysicalMuTA

EXPERIMENT_ID = "R45"


def main():
    plt = common.setup_style()
    tol = common.declare_tolerance(scale=1.0, safety_factor=100.0)
    config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    model = PhysicalMuTA(2, physical_config=config)
    result = model.run(
        [1, 0, 0, 0],
        mode="physical-conditional",
        analog_outcomes=dict.fromkeys(model.measurement_order, 0.0),
    )

    joint = result.decoded_joint_probabilities
    marginals = result.decoded_marginals
    labels = list(joint.keys())

    independent_correlation = float(
        sum(((-1) ** (bits[0] + bits[1])) * p for bits, p in joint.items())
    )
    reported_correlation = result.diagnostics["pair_correlations"][(0, 1)]
    correlation_error = abs(independent_correlation - reported_correlation)

    wire0_marginal, wire1_marginal = (
        marginals[model.output_nodes[0]],
        marginals[model.output_nodes[1]],
    )
    product_distribution = {
        bits: wire0_marginal[bits[0]] * wire1_marginal[bits[1]] for bits in labels
    }
    tv_distance_product = float(
        sum(abs(joint[bits] - product_distribution[bits]) for bits in labels) / 2
    )

    rows = [
        {
            "label": str(bits),
            "joint_probability": joint[bits],
            "product_of_marginals": product_distribution[bits],
            "difference": joint[bits] - product_distribution[bits],
        }
        for bits in labels
    ]

    status = "pass" if (correlation_error < tol and tv_distance_product > 0) else "fail"

    common.save_result(
        rows,
        "R45_joint_readout_correlations",
        extra={
            "protocol": "Independent connected-correlation recomputation + joint-vs-product-of-marginals check, 2-wire",
            "oracle_class": "A/E",
            "status_category": "exact",
            "tolerance": tol,
            "independent_pair_correlation": independent_correlation,
            "reported_pair_correlation": reported_correlation,
            "correlation_error": correlation_error,
            "tv_distance_joint_vs_product_of_marginals": tv_distance_product,
            "acceptance_condition": f"correlation error < {tol:.3e}; product-of-marginals TV distance > 0",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "A/E", "status": status},
    )

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    x = range(len(rows))
    axes[0].bar(
        [i - 0.15 for i in x],
        [r["joint_probability"] for r in rows],
        width=0.3,
        label="actual joint",
        color=common.COLORS["photographiqml"],
    )
    axes[0].bar(
        [i + 0.15 for i in x],
        [r["product_of_marginals"] for r in rows],
        width=0.3,
        label="product of marginals",
        color=common.COLORS["classical_baseline"],
    )
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels([r["label"] for r in rows])
    axes[0].set(
        title=f"Joint vs. product of marginals (TV={tv_distance_product:.4f})", ylabel="probability"
    )
    axes[0].legend(fontsize=7)
    axes[1].bar(
        ["independent", "reported"],
        [independent_correlation, reported_correlation],
        color=common.COLORS["photographiqml"],
    )
    axes[1].set(
        title=f"<Z0 Z1> connected correlation (error={correlation_error:.2e})", ylabel="correlation"
    )
    fig.suptitle(f"R45: joint readout correlations (status={status})")
    common.save_figure(fig, "R45_joint_readout_correlations")
    plt.close(fig)

    common.print_summary(
        "R45 joint readout correlations",
        correlation_error=correlation_error,
        tv_distance_product=tv_distance_product,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R45 failed: correlation_error={correlation_error} tv_distance={tv_distance_product}"
        )


if __name__ == "__main__":
    main()


## 46_hard_vs_soft_decoding



In [ ]:
"""R46: Hard versus soft resource-decoder calibration on the specific
declared preparation ensemble, and confirmation that PhysicalMuTA rejects a
soft feed-forward configuration before allocation.

Scientific question: (a) Does NearestCellDecoder always report
confidence=None (no invented posterior), while SoftDecisionDecoder reports a
calibrated posterior that is highest near an ideal codeword lattice site and
falls toward the 50/50 point at the boundary between two adjacent cells,
for the declared equal-prior 0/1 preparation ensemble? (b) Does
PhysicalMuTA's capability audit reject decoder="soft" before any Fock
allocation, exactly as documented (docs/physical/decoding.md: "Physical
MuTA therefore rejects a soft feed-forward configuration before
allocation")?

Theory/equations: GKP lattice spacing L=sqrt(2*pi); ideal bit-0 sites at
q=2sL, bit-1 sites at q=(2s+1)L. SoftDecisionDecoder(code, basis="Z") is
Bayesian discrimination of a declared equal-prior finite zero/one
preparation ensemble (docs/physical/decoding.md); it is not a universal
posterior for an arbitrary adaptive graph state -- part (b) verifies
PhotoGraphiQML enforces exactly that boundary.

Functionality tested: photographiq.NearestCellDecoder/SoftDecisionDecoder
via GKPCode.decode (shared PhotoGraphiQ decoder objects); MuTA.physical_
capabilities' rejection of decoder="soft" for flow-based execution.

Oracle and independence class: E (structural: confidence=None for nearest,
calibration trend and rejection-before-allocation are self-consistency
properties of the documented decoder contract, not an external ground
truth).

Exact/approximate/statistical status: exact (discrete confidence-field and
rejection checks); descriptive for the calibration trend (no claimed
functional form beyond monotonicity near the ideal-to-boundary sweep).

Primary metric: NearestCellDecoder confidence is None at every point;
SoftDecisionDecoder confidence is monotonically non-increasing as the raw
outcome moves from an ideal bit-0 site (q=0) toward the cell boundary
(q=L/2); decoder="soft" rejected before allocation (no Fock allocation
attempted).

Declared acceptance condition: both hold exactly.

Expected cost: light (decoder calls only, no Fock simulation for part (a);
allocation-free audit for part (b)).

Manuscript destination: Appendix (Fig. 11 supporting decoder-calibration
table).

Scientific limitations: The soft posterior is calibrated only for the
declared equal-prior finite 0/1 ensemble at one mode in isolation; this
experiment does not extend it to (and PhotoGraphiQML does not permit) an
arbitrary adaptive MuTA graph state.
"""

import sys
from pathlib import Path

import common
import numpy as np
import photographiq as pg

from photographiqml import GKPBridge, GKPPhysicalConfig, MuTA
from photographiqml.lowering import physical_capabilities

EXPERIMENT_ID = "R46"


def main():
    plt = common.setup_style()
    L = np.sqrt(2 * np.pi)
    bridge = GKPBridge(cutoff=32, peak_width=0.4, envelope=0.4, peaks=8, grid_points=4097)
    code = bridge.code
    nearest = pg.NearestCellDecoder()
    soft = pg.SoftDecisionDecoder(code, basis="Z")

    sweep = np.linspace(0, L / 2, 6)  # ideal bit-0 site (0) to cell boundary (L/2)
    rows = []
    for raw in sweep:
        nearest_result = code.decode(float(raw), decoder=nearest)
        soft_result = code.decode(float(raw), decoder=soft)
        rows.append(
            {
                "raw_outcome": float(raw),
                "nearest_bit": nearest_result.bit,
                "nearest_confidence_is_none": nearest_result.confidence is None,
                "soft_bit": soft_result.bit,
                "soft_confidence": soft_result.confidence,
            }
        )

    confidences = [r["soft_confidence"] for r in rows]
    # Confidence saturates near 1.0 for most of the cell interior (observed:
    # >0.98 until very close to the boundary), so a strict pointwise
    # non-increase check is dominated by numerical noise in that flat
    # near-1 region; allow a small tolerance there while still requiring the
    # overall trend (and the sharp final drop toward the boundary) to hold.
    monotonic_non_increasing = all(
        confidences[i] >= confidences[i + 1] - 1e-3 for i in range(len(confidences) - 1)
    )
    overall_trend_decreasing = confidences[0] > confidences[-1]
    nearest_always_none = all(r["nearest_confidence_is_none"] for r in rows)

    # --- (b) soft decoder rejected before allocation for flow-based execution --
    model = MuTA(1, 1, one_column=True)
    soft_config = GKPPhysicalConfig(
        cutoff=16, peak_width=0.9, envelope=0.9, peaks=3, grid_points=513, decoder="soft"
    )
    audit = physical_capabilities(model, config=soft_config)
    soft_rejected_before_allocation = not audit["supported"] and any(
        "soft" in r.lower() for r in audit["reasons"]
    )

    status = (
        "pass"
        if (
            nearest_always_none
            and monotonic_non_increasing
            and overall_trend_decreasing
            and soft_rejected_before_allocation
        )
        else "fail"
    )

    common.save_result(
        rows,
        "R46_hard_vs_soft_decoding",
        extra={
            "protocol": "NearestCellDecoder vs SoftDecisionDecoder calibration sweep + soft-decoder rejection audit",
            "oracle_class": "E",
            "status_category": "exact",
            "lattice_spacing": L,
            "nearest_always_none": nearest_always_none,
            "soft_confidence_monotonic_non_increasing": monotonic_non_increasing,
            "soft_confidence_overall_trend_decreasing": overall_trend_decreasing,
            "soft_rejected_before_allocation": soft_rejected_before_allocation,
            "audit_reasons": audit["reasons"],
            "acceptance_condition": "nearest confidence always None; soft confidence non-increasing toward boundary; soft rejected before allocation",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(
        [r["raw_outcome"] / L for r in rows],
        confidences,
        "o-",
        color=common.COLORS["photographiqml"],
    )
    ax.axhline(0.5, color=common.COLORS["acceptance"], linestyle="--", label="50/50 boundary")
    ax.set(
        title=f"R46: soft-decoder calibration, ideal site -> cell boundary (status={status})",
        xlabel="raw outcome / L",
        ylabel="soft-decoder confidence",
    )
    ax.legend()
    common.save_figure(fig, "R46_hard_vs_soft_decoding")
    plt.close(fig)

    common.print_summary(
        "R46 hard vs soft decoding",
        nearest_always_none=nearest_always_none,
        monotonic=monotonic_non_increasing,
        soft_rejected=soft_rejected_before_allocation,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R46 failed: nearest_none={nearest_always_none} monotonic={monotonic_non_increasing} rejected={soft_rejected_before_allocation}"
        )


if __name__ == "__main__":
    main()


## 48_physical_training_stability



In [ ]:
"""R48: Discrete physical training stability across training seeds, fresh
validation trajectories, and shot counts.

Scientific question: How stable is DiscreteSearch's selected categorical
0/pi angle configuration across independent training seeds, and how much
does the resulting classifier's accuracy change when re-evaluated on fresh
(independently seeded) validation trajectories versus the training
trajectories used for selection?

Theory/equations: none (empirical stability study). The 0/pi family is a
categorical relabeling, not a continuously tunable variational family; this
experiment explicitly reports selection instability rather than describing
it as continuous training convergence (see docs/physical/physical-training.md).

Functionality tested: MuTAClassifier(PhysicalMuTA, trainer=DiscreteSearch)
(photographiqml.models, physical_training), across 6 training seeds, each
re-evaluated on an independent validation-trajectory seed.

Oracle and independence class: N/A (descriptive stability measurement -- no
correctness oracle for "the right selected configuration"; this documents
instability itself as the finding, matching docs/physical/physical-training.md's
own admission that a 2-shot demo does not support a robust-classifier
claim).

Exact/approximate/statistical status: statistical (6 independent training
seeds, each with one independent fresh-trajectory validation seed; small-n,
reported with explicit sample size, no claimed generalization estimate).

Primary metric: training-trajectory accuracy vs. fresh-validation-trajectory
accuracy per seed; fraction of seeds where the two differ (selection
instability rate); fraction of seeds selecting each distinct angle
configuration (selection diversity).

Declared acceptance condition: none required against a correctness oracle
(N/A); the only enforced pass/fail is that every reported accuracy is
finite and in [0,1] and that DiscreteSearchResult.parameters are always
exact categorical {0,pi} values (never rounded from an attempted continuous
solution).

Expected cost: heavy (6 seeds x DiscreteSearch(sweeps=1) x 4 training
inputs x 2 shots x cutoff=40 Fock simulations, plus fresh-trajectory
validation runs).

Manuscript destination: Main text (Fig. 12, physical training stability --
explicitly an instability/negative result, not a performance claim).

Scientific limitations: Small shot count (2) and small training set (4
points) by design, matching the tractable demonstration scope in
experiments/restricted_physical.py; this is execution evidence, not a
held-out generalization estimate (per docs/physical/physical-training.md).
"""

import sys
from pathlib import Path

import common
import numpy as np

from photographiqml import GKPPhysicalConfig, PhysicalMuTA
from photographiqml.models import MuTAClassifier
from photographiqml.physical_training import DiscreteSearch

EXPERIMENT_ID = "R48"
SEEDS = (7, 19, 23, 31, 42, 101)
X = [[0.0], [0.2], [2.9], [np.pi]]
Y = [0, 0, 1, 1]


def main():
    plt = common.setup_style()
    config = GKPPhysicalConfig(cutoff=40, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    rows = []
    for seed in SEEDS:
        model = PhysicalMuTA(1, physical_config=config)
        classifier = MuTAClassifier(
            model,
            trainer=DiscreteSearch(sweeps=1, seed=seed),
            physical_options={"mode": "physical-shots", "shots": 2, "seed": seed},
        )
        classifier.fit(X, Y)
        training_predictions = classifier.predict(X)
        training_accuracy = float(np.mean(training_predictions == np.array(Y)))

        classifier.physical_options["seed"] = seed + 1000  # independent fresh trajectories
        fresh_predictions = classifier.predict(X)
        fresh_accuracy = float(np.mean(fresh_predictions == np.array(Y)))

        selected = tuple(np.round(classifier.history.parameters, 6).tolist())
        exact_categorical = all(
            np.isclose(v, 0.0) or np.isclose(v, np.pi) for v in classifier.history.parameters
        )

        rows.append(
            {
                "seed": seed,
                "selected_parameters": selected,
                "exact_categorical": exact_categorical,
                "training_trajectory_accuracy": training_accuracy,
                "fresh_trajectory_accuracy": fresh_accuracy,
                "accuracy_changed": training_accuracy != fresh_accuracy,
            }
        )

    all_finite = all(
        0 <= r["training_trajectory_accuracy"] <= 1 and 0 <= r["fresh_trajectory_accuracy"] <= 1
        for r in rows
    )
    all_categorical = all(r["exact_categorical"] for r in rows)
    instability_rate = float(np.mean([r["accuracy_changed"] for r in rows]))
    distinct_selections = len({r["selected_parameters"] for r in rows})
    status = "pass" if (all_finite and all_categorical) else "fail"

    common.save_result(
        rows,
        "R48_physical_training_stability",
        extra={
            "protocol": "DiscreteSearch classifier stability across training seeds, fresh-trajectory re-evaluation",
            "oracle_class": "N/A",
            "status_category": "statistical",
            "n_seeds": len(SEEDS),
            "instability_rate": instability_rate,
            "distinct_selected_configurations": distinct_selections,
            "acceptance_condition": "all accuracies finite and in [0,1]; every selected configuration exactly categorical {0,pi}",
            "finding": "Selected configurations and their fresh-trajectory accuracy vary across training seeds; this documents instability, not a robust classifier-performance claim (per docs/physical/physical-training.md).",
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "N/A", "status": status},
    )

    fig, ax = plt.subplots(figsize=(8, 4.2))
    x = range(len(rows))
    ax.bar(
        [i - 0.15 for i in x],
        [r["training_trajectory_accuracy"] for r in rows],
        width=0.3,
        label="training trajectories",
        color=common.COLORS["photographiqml"],
    )
    ax.bar(
        [i + 0.15 for i in x],
        [r["fresh_trajectory_accuracy"] for r in rows],
        width=0.3,
        label="fresh trajectories",
        color=common.COLORS["piquasso"],
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels([str(s) for s in SEEDS])
    # A zero-height bar (seed 7's fresh-trajectory accuracy) must not read as
    # "no data"; label every bar's value explicitly.
    for i, r in enumerate(rows):
        ax.text(
            i - 0.15,
            r["training_trajectory_accuracy"] + 0.02,
            f"{r['training_trajectory_accuracy']:.2f}",
            ha="center",
            va="bottom",
            fontsize=6,
        )
        ax.text(
            i + 0.15,
            r["fresh_trajectory_accuracy"] + 0.02,
            f"{r['fresh_trajectory_accuracy']:.2f}",
            ha="center",
            va="bottom",
            fontsize=6,
        )
    ax.set_ylim(0, 1.15)
    ax.set(
        title=f"R48: discrete physical training instability (instability_rate={instability_rate:.2f}, {distinct_selections} distinct configs)",
        xlabel="training seed",
        ylabel="accuracy",
    )
    ax.legend(fontsize=7)
    common.save_figure(fig, "R48_physical_training_stability")
    plt.close(fig)

    common.print_summary(
        "R48 physical training stability",
        instability_rate=instability_rate,
        distinct_selections=distinct_selections,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R48 failed: all_finite={all_finite} all_categorical={all_categorical}"
        )


if __name__ == "__main__":
    main()


## 47_resource_axis_convergence



In [ ]:
"""R47: Separate one-axis-at-a-time convergence studies for cutoff, grid
points, peak count, peak width, and envelope.

Scientific question: For a one-wire zero-angle physical-conditional
baseline, how do decoded probabilities change as each of the five resource/
numerical axes (cutoff, grid_points, peaks, peak_width, envelope) is
independently varied while holding the others fixed, and does the package
correctly refuse to silently certify convergence from this alone?

Theory/equations: docs/physical/convergence.md distinguishes numerical
refinement axes (cutoff, grid_points, peaks -- should stabilize decoded
probabilities as resolution increases, for a *fixed* physical resource) from
physical resource-change axes (peak_width, envelope -- these change the
finite-energy resource itself, so their "convergence" reflects a genuine
physical modification, not just numerical error shrinking).

Functionality tested: PhysicalMuTA.physical_convergence /
photographiqml.physical.validate_physical_model, for all five declared
axes.

Oracle and independence class: E (self-consistency -- successive-point
probability deltas are checked against a declared threshold for the
numerical axes; this is not compared to an external convergence oracle,
since no independent finite-resource GKP simulator is available at this
resolution).

Exact/approximate/statistical status: statistical/descriptive (finite-
resource numerical study; no claimed convergence rate).

Primary metric: max_probability_delta between successive points on each
axis; certified flag (must always be False).

Declared acceptance condition: certified is False for every axis (the
package must never silently claim certification); for the two numerical
axes (cutoff, grid_points), the max_probability_delta at the finest pair of
points is smaller than at the coarsest pair (a genuine refinement trend,
not merely "some decrease somewhere").

Expected cost: moderate-to-heavy (5 axes x up to 4 points each, all
physical-conditional Fock simulations).

Manuscript destination: Main text (Fig. 12, convergence panel).

Scientific limitations: One-wire, zero-angle case only (matching the
tractable resource baseline in docs/physical/evidence.json); numerical
grid/cutoff stability does not by itself remove finite-resource physical
error (peak_width/envelope axes), per docs/physical/convergence.md.
"""

import sys
from pathlib import Path

import common

from photographiqml import GKPPhysicalConfig, PhysicalMuTA

EXPERIMENT_ID = "R47"


def main():
    plt = common.setup_style()
    baseline = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
    model = PhysicalMuTA(1, physical_config=baseline)
    outcomes = dict.fromkeys(model.measurement_order, 0.0)

    axes_values = {
        "cutoff": [20, 24, 28],
        "grid_points": [513, 1025, 2049],
        "peaks": [3, 4, 5],
        "peak_width": [0.85, 0.9, 0.95],
        "envelope": [0.85, 0.9, 0.95],
    }

    studies = {}
    rows = []
    for axis, values in axes_values.items():
        study = model.physical_convergence(
            [1, 0], values, axis=axis, mode="physical-conditional", analog_outcomes=outcomes
        )
        studies[axis] = study
        for row in study["rows"]:
            rows.append(
                {
                    "axis": axis,
                    "axis_type": study["axis_type"],
                    "value": row["value"],
                    "max_probability_delta": row["max_probability_delta"],
                    "prediction": row["prediction"],
                }
            )

    all_uncertified = all(not s["certified"] for s in studies.values())
    numerical_axes_ok = True
    for axis in ("cutoff", "grid_points"):
        deltas = [
            r["max_probability_delta"]
            for r in rows
            if r["axis"] == axis and r["max_probability_delta"] is not None
        ]
        if len(deltas) < 2 or not (deltas[-1] < deltas[0]):
            numerical_axes_ok = False

    status = "pass" if (all_uncertified and numerical_axes_ok) else "fail"

    common.save_result(
        rows,
        "R47_resource_axis_convergence",
        extra={
            "protocol": "One-axis-at-a-time physical_convergence study, 5 axes, one-wire zero-angle baseline",
            "oracle_class": "E",
            "status_category": "statistical",
            "baseline_config": baseline.to_dict(),
            "acceptance_condition": "certified False for every axis; numerical axes (cutoff, grid_points) show decreasing successive-point delta",
            "all_uncertified": all_uncertified,
            "numerical_axes_ok": numerical_axes_ok,
            "status": status,
        },
        meta_extra={"experiment_id": EXPERIMENT_ID, "oracle_class": "E", "status": status},
    )

    fig, axes = plt.subplots(1, 5, figsize=(18, 3.4))
    for ax, (axis, values) in zip(axes, axes_values.items()):
        axis_rows = [r for r in rows if r["axis"] == axis]
        deltas = [
            r["max_probability_delta"] if r["max_probability_delta"] is not None else 0
            for r in axis_rows
        ]
        color = (
            common.COLORS["photographiqml"]
            if axis in ("cutoff", "grid_points", "peaks")
            else common.COLORS["piquasso"]
        )
        ax.plot([r["value"] for r in axis_rows], deltas, "o-", color=color)
        ax.set(
            title=f"{axis}\n({'numerical' if axis in ('cutoff', 'grid_points', 'peaks') else 'physical resource'})",
            xlabel=axis,
            ylabel="max prob delta",
        )
    fig.suptitle(f"R47: one-axis-at-a-time convergence (status={status}, never certified)")
    common.save_figure(fig, "R47_resource_axis_convergence")
    plt.close(fig)

    common.print_summary(
        "R47 resource axis convergence",
        all_uncertified=all_uncertified,
        numerical_axes_ok=numerical_axes_ok,
        status=status,
    )
    if status != "pass":
        raise AssertionError(
            f"R47 failed: all_uncertified={all_uncertified} numerical_axes_ok={numerical_axes_ok}"
        )


if __name__ == "__main__":
    main()


## Aggregate performance figure: resource and runtime scaling across the suite.

Scientific question: none directly -- this script does not run a new
experiment. It assembles one multi-panel summary figure from already-saved
JSON produced by R13 (matched-workload runtime/resource scaling), R36
(lowering preservation + Fock dimension), R42 (abstraction overhead), and
R47 (resource-axis convergence), as required by the task's performance
panels (section 6: "Include performance panels within R13, R36, R42, and
R47, and generate a final aggregate performance figure").

Functionality tested: none (pure reporting/aggregation of prior results).

Oracle and independence class: N/A (reporting only).

Exact/approximate/statistical status: n/a.

Primary metric: n/a -- this script reproduces the aggregate figure from
saved CSV/JSON without rerunning any simulation, satisfying the
reproduce-without-rerunning-expensive-simulations requirement.

Declared acceptance condition: all four source JSON files must exist and be
readable; if any is missing, that panel is drawn with an explicit "missing:
run R<N> first" placeholder rather than silently omitted.

Expected cost: negligible (JSON loading + plotting only).

Manuscript destination: Main text (Fig. 13, aggregate performance figure).

Scientific limitations: Aggregates local, single-machine timings already
reported by the source experiments; adds no new measurement of its own.

In [ ]:
"""Aggregate performance figure: resource and runtime scaling across the suite.

Scientific question: none directly -- this script does not run a new
experiment. It assembles one multi-panel summary figure from already-saved
JSON produced by R13 (matched-workload runtime/resource scaling), R36
(lowering preservation + Fock dimension), R42 (abstraction overhead), and
R47 (resource-axis convergence), as required by the task's performance
panels (section 6: "Include performance panels within R13, R36, R42, and
R47, and generate a final aggregate performance figure").

Functionality tested: none (pure reporting/aggregation of prior results).

Oracle and independence class: N/A (reporting only).

Exact/approximate/statistical status: n/a.

Primary metric: n/a -- this script reproduces the aggregate figure from
saved CSV/JSON without rerunning any simulation, satisfying the
reproduce-without-rerunning-expensive-simulations requirement.

Declared acceptance condition: all four source JSON files must exist and be
readable; if any is missing, that panel is drawn with an explicit "missing:
run R<N> first" placeholder rather than silently omitted.

Expected cost: negligible (JSON loading + plotting only).

Manuscript destination: Main text (Fig. 13, aggregate performance figure).

Scientific limitations: Aggregates local, single-machine timings already
reported by the source experiments; adds no new measurement of its own.
"""

import sys
from pathlib import Path

import json

import common

EXPERIMENT_ID = "R_PERF"


def load(name):
    path = common.JSON_DIR / f"{name}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def placeholder(ax, source_id):
    ax.text(
        0.5,
        0.5,
        f"missing: run {source_id} first",
        ha="center",
        va="center",
        transform=ax.transAxes,
        color=common.COLORS["unsupported"],
    )
    ax.set_xticks([])
    ax.set_yticks([])


def main():
    plt = common.setup_style()
    r13 = load("R13_runtime_scaling")
    r36 = load("R36_lowering_preservation")
    r42 = load("R42_abstraction_overhead")
    r47 = load("R47_resource_axis_convergence")

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))

    ax = axes[0, 0]
    if r13:
        rows = r13["rows"]
        wires = [r["n_wires"] for r in rows]
        ax.errorbar(
            wires,
            [r["photographiqml_median_seconds"] for r in rows],
            yerr=[r["photographiqml_iqr_seconds"] / 2 for r in rows],
            marker="o",
            color=common.COLORS["photographiqml"],
            label="PhotoGraphiQML",
        )
        ax.errorbar(
            wires,
            [r["mentpy_median_seconds"] for r in rows],
            yerr=[r["mentpy_iqr_seconds"] / 2 for r in rows],
            marker="s",
            color=common.COLORS["mentpy"],
            label="MentPy",
        )
        ax.set_yscale("log")
        ax.set(
            title="(a) Matched-workload logical runtime vs. wires (R13)",
            xlabel="n_wires",
            ylabel="seconds",
        )
        ax.legend(fontsize=7)
        common.panel_label(ax, "a")
    else:
        placeholder(ax, "R13")

    ax = axes[0, 1]
    if r36:
        rows = r36["rows"]
        labels = [f"n{r['n_wires']}L{r['n_layers']}" for r in rows]
        ax.bar(
            range(len(rows)),
            [r["reported_fock_dimension"] for r in rows],
            color=common.COLORS["piquasso"],
        )
        ax.set_xticks(range(len(rows)))
        ax.set_xticklabels(labels)
        ax.set_yscale("log")
        ax.set(title="(b) Total-photon Fock dimension D=C(K+m-1,m) (R36)", ylabel="dimension")
        common.panel_label(ax, "b")
    else:
        placeholder(ax, "R36")

    ax = axes[1, 0]
    if r42:
        rows = r42["rows"]
        ax.bar(
            range(len(rows)),
            [r["median_seconds"] for r in rows],
            yerr=[r["iqr_seconds"] / 2 for r in rows],
            color=common.COLORS["photographiqml"],
            capsize=4,
        )
        ax.set_xticks(range(len(rows)))
        ax.set_xticklabels([r["pipeline"] for r in rows], rotation=25, ha="right", fontsize=6.5)
        ax.set(
            title=f"(c) Abstraction overhead, matched 1-wire workload (R42)\nwrapper overhead={r42['wrapper_overhead_seconds']:.3f}s",
            ylabel="median seconds",
        )
        common.panel_label(ax, "c")
    else:
        placeholder(ax, "R42")

    ax = axes[1, 1]
    if r47:
        rows = r47["rows"]
        # cutoff (~20-28) and grid_points (~513-2049) live on incompatible
        # x-scales; sharing one linear x-axis collapses the cutoff series
        # into an invisible sliver, so each gets its own x-axis (twiny),
        # sharing only the log-scale y-axis. The first point of each study
        # has max_probability_delta=None (no previous point yet) and is
        # skipped rather than plotted as a fake zero on a log axis.
        ax_cutoff, ax_grid = ax, ax.twiny()
        for axis, plot_ax, color in (
            ("cutoff", ax_cutoff, common.COLORS["photographiqml"]),
            ("grid_points", ax_grid, common.COLORS["mentpy"]),
        ):
            axis_rows = [
                r for r in rows if r["axis"] == axis and r["max_probability_delta"] is not None
            ]
            plot_ax.plot(
                [r["value"] for r in axis_rows],
                [r["max_probability_delta"] for r in axis_rows],
                "o-",
                color=color,
                label=axis,
            )
            plot_ax.set_xlabel(axis, color=color, fontsize=8)
            plot_ax.tick_params(axis="x", colors=color, labelsize=7)
        ax_cutoff.set_yscale("log")
        ax_cutoff.set(
            title="(d) Numerical-refinement convergence (R47)", ylabel="max probability delta"
        )
        handles = ax_cutoff.get_lines() + ax_grid.get_lines()
        ax_cutoff.legend(handles, [h.get_label() for h in handles], fontsize=7)
        common.panel_label(ax, "d")
    else:
        placeholder(ax, "R47")

    fig.suptitle(
        "Aggregate performance and resource scaling (reproduced from saved R13/R36/R42/R47 results)"
    )
    common.save_figure(fig, "R_PERF_aggregate_performance")
    plt.close(fig)

    sources_present = sum(x is not None for x in (r13, r36, r42, r47))
    common.save_json(
        {
            "sources_present": sources_present,
            "sources_total": 4,
            "oracle_class": "N/A",
            "status": "pass" if sources_present == 4 else "partial",
        },
        "R_PERF_aggregate_performance",
    )
    common.write_metadata(
        "R_PERF_aggregate_performance", experiment_id=EXPERIMENT_ID, oracle_class="N/A"
    )
    common.print_summary(
        "R_PERF aggregate performance figure", sources_present=sources_present, sources_total=4
    )


if __name__ == "__main__":
    main()


## Aggregate manuscript tables

Rebuilds `tables/` from the saved results above.

In [ ]:
import generate_tables
generate_tables.main()
